<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/SUBA_V8_PURE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
# @title ⚙️ Ячейка 0/10: Окружение SUBA RUN V8 (Java 17 + Flutter + Android SDK/NDK 28)
# ============================================================================
# SUBA RUN V8 — Subaru EJ20X · A2TB100B · SSM2-over-CAN Tuning Logger
# Clean-room сборка: только Subaru, без Nissan-наследия.
# Запускай один раз за сессию Colab. Идемпотентна (можно перезапускать).
# ============================================================================
import os

print('=' * 64)
print('  SUBA RUN V8 // A2TB100B  — подготовка окружения (~10 мин)')
print('=' * 64)

print('\n[1/4] Системные пакеты + Java 17 ...')
!apt-get update -qq
!apt-get install -y -qq curl git unzip xz-utils zip libglu1-mesa openjdk-17-jdk-headless ninja-build cmake > /dev/null
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java 2>&1 | tail -1

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')
!java -version 2>&1 | head -1

print('\n[2/4] Flutter SDK (stable) ...')
if not os.path.exists('/content/flutter'):
    !git clone https://github.com/flutter/flutter.git -b stable --depth 1 /content/flutter 2>/dev/null
else:
    print('  Flutter уже скачан — пропуск.')

os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'
!/content/flutter/bin/flutter config --no-analytics --no-cli-animations 2>/dev/null
!/content/flutter/bin/flutter --disable-telemetry 2>/dev/null

print('\n[3/4] Android SDK 36 + Build-tools 36 + NDK 28 ...')
if not os.path.exists('/content/android-sdk/cmdline-tools/latest'):
    !wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
    !mkdir -p /content/android-sdk/cmdline-tools
    !unzip -q /tmp/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
    !mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest 2>/dev/null || true

os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']

!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!/content/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-36" "build-tools;36.0.0" "ndk;28.2.13676358" > /dev/null 2>&1

!/content/flutter/bin/flutter config --android-sdk /content/android-sdk 2>/dev/null
!/content/flutter/bin/flutter precache --android 2>/dev/null

print('\n[4/4] Проверка ...')
!/content/flutter/bin/flutter --version | head -2

print()
print('=' * 64)
print('  ГОТОВО: Java 17 / Flutter / SDK 36 / NDK 28.2.13676358')
print('  Далее -> ячейка 1/10 (проект + конфиги)')
print('=' * 64)


  SUBA RUN V8 // A2TB100B  — подготовка окружения (~10 мин)

[1/4] Системные пакеты + Java 17 ...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.20" 2026-07-21

[2/4] Flutter SDK (stable) ...
  Flutter уже скачан — пропуск.
Analytics reporting disabled.
Setting "cli-animations" value to "false".

You may need to restart any open editors for them to read new settings.

[3/4] Android SDK 36 + Build-tools 36 + NDK 28 ...
Setting "android-sdk" value to "/content/android-sdk".

You may need to restart any open editors for them to read new settings.

[4/4] Проверка ...
Flutter 3.47.2 • channel stable • https://github.com/flutter/flutter.git
Framework • revision d3b14c8769 (12 days ago) • 2026-08-26 16:07:51 -0700

  ГОТОВО: Java 17 / Flutter / SDK 36 / NDK 28.2.13676358
  Далее -> ячейка 1/10 (проект + конфиги)


In [68]:
# @title 🏗️ Ячейка 1/10: Проект SUBA RUN V8 + конфиги Android
# ============================================================================
# Создаёт flutter-проект suba_run_v8 и всю Android-обвязку.
# Проверенный набор из V6/V7: Gradle 8.10 · AGP 8.6 · SDK 36 · NDK 28.
# ============================================================================
import os

os.chdir('/content')
!rm -rf /content/suba_run_v8
!/content/flutter/bin/flutter create --org com.subarun --project-name suba_run_v8 suba_run_v8

os.chdir('/content/suba_run_v8')

for folder in ['models', 'services', 'screens', 'widgets', 'ssm', 'generated']:
    os.makedirs(f'lib/{folder}', exist_ok=True)

# ============ pubspec.yaml ============
with open('pubspec.yaml', 'w') as f:
    f.write('''name: suba_run_v8
description: Subaru EJ20X A2TB100B SSM2-over-CAN Tuning Logger V8
version: 8.0.0+1
publish_to: none

environment:
  sdk: ">=3.0.0 <4.0.0"

dependencies:
  flutter:
    sdk: flutter
  cupertino_icons: ^1.0.6
  fl_chart: 0.68.0
  flutter_bluetooth_serial: 0.4.0
  permission_handler: 11.3.1
  path_provider: 2.1.4
  path: ^1.9.0
  csv: 6.0.0
  shared_preferences: 2.3.2
  file_picker: 8.1.2
  share_plus: 10.0.2
  intl: ^0.19.0
  xml: ^6.5.0

dev_dependencies:
  flutter_test:
    sdk: flutter
  flutter_lints: ^4.0.0

flutter:
  uses-material-design: true
''')
print('OK  pubspec.yaml')

# ============ AndroidManifest.xml ============
with open('android/app/src/main/AndroidManifest.xml', 'w') as f:
    f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.INTERNET"/>
    <uses-permission android:name="android.permission.WRITE_EXTERNAL_STORAGE" android:maxSdkVersion="28"/>
    <application
        android:label="SUBA RUN V8"
        android:name="${applicationName}"
        android:icon="@mipmap/ic_launcher">
        <activity
            android:name=".MainActivity"
            android:exported="true"
            android:launchMode="singleTop"
            android:theme="@style/LaunchTheme"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:hardwareAccelerated="true"
            android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme" android:resource="@style/NormalTheme"/>
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2"/>
    </application>
</manifest>
''')
print('OK  AndroidManifest.xml')

# ============ MainActivity.kt ============
main_dir = 'android/app/src/main/kotlin/com/subarun/suba_run_v8'
os.makedirs(main_dir, exist_ok=True)
!rm -rf android/app/src/main/java

with open(f'{main_dir}/MainActivity.kt', 'w') as f:
    f.write('''package com.subarun.suba_run_v8
import android.os.Bundle
import android.view.WindowManager
import io.flutter.embedding.android.FlutterActivity
class MainActivity : FlutterActivity() {
    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        window.addFlags(WindowManager.LayoutParams.FLAG_KEEP_SCREEN_ON)
    }
}
''')
print('OK  MainActivity.kt')

# ============ Gradle ============
with open('android/gradle/wrapper/gradle-wrapper.properties', 'w') as f:
    f.write('''distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\\://services.gradle.org/distributions/gradle-8.14-all.zip
''')

with open('android/settings.gradle.kts', 'w') as f:
    f.write('''pluginManagement {
    val flutterSdkPath = run {
        val properties = java.util.Properties()
        file("local.properties").inputStream().use { properties.load(it) }
        val flutterSdkPath = properties.getProperty("flutter.sdk")
        require(flutterSdkPath != null) { "flutter.sdk not set in local.properties" }
        flutterSdkPath
    }
    includeBuild("$flutterSdkPath/packages/flutter_tools/gradle")
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}
plugins {
    id("dev.flutter.flutter-plugin-loader") version "1.0.0"
    id("com.android.application") version "8.13.0" apply false
    id("org.jetbrains.kotlin.android") version "2.1.20" apply false
}
include(":app")
''')

with open('android/build.gradle.kts', 'w') as f:
    f.write('''allprojects {
    repositories { google(); mavenCentral() }
    configurations.all {
        resolutionStrategy {
            force("androidx.core:core:1.13.1")
            force("androidx.core:core-ktx:1.13.1")
            force("androidx.appcompat:appcompat:1.7.0")
            force("androidx.annotation:annotation:1.8.2")
        }
    }
}
val newBuildDir: Directory = rootProject.layout.buildDirectory.dir("../../build").get()
rootProject.layout.buildDirectory.value(newBuildDir)
subprojects {
    val newSubprojectBuildDir: Directory = newBuildDir.dir(project.name)
    project.layout.buildDirectory.value(newSubprojectBuildDir)
}
subprojects { project.evaluationDependsOn(":app") }
tasks.register("clean") { delete(rootProject.layout.buildDirectory) }
''')

with open('android/app/build.gradle.kts', 'w') as f:
    f.write('''plugins {
    id("com.android.application")
    id("kotlin-android")
    id("dev.flutter.flutter-gradle-plugin")
}
android {
    namespace = "com.subarun.suba_run_v8"
    compileSdk = 36
    ndkVersion = "28.2.13676358"
    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
    kotlinOptions { jvmTarget = JavaVersion.VERSION_17.toString() }
    defaultConfig {
        applicationId = "com.subarun.suba_run_v8"
        minSdk = 21
        targetSdk = 34
        versionCode = 8
        versionName = "8.0.0"
        multiDexEnabled = true
    }
    buildTypes {
        release {
            signingConfig = signingConfigs.getByName("debug")
            isMinifyEnabled = false
            isShrinkResources = false
        }
    }
}
dependencies {
    implementation("androidx.core:core:1.13.1")
    implementation("androidx.core:core-ktx:1.13.1")
    implementation("androidx.appcompat:appcompat:1.7.0")
    implementation("androidx.multidex:multidex:2.0.1")
}
flutter { source = "../.." }
''')

with open('android/gradle.properties', 'w') as f:
    f.write('''org.gradle.jvmargs=-Xmx4G -XX:+UseParallelGC -XX:MaxMetaspaceSize=2G
android.useAndroidX=true
android.enableJetifier=true
android.nonTransitiveRClass=false
kotlin.code.style=official
org.gradle.parallel=true
org.gradle.caching=false
org.gradle.configuration-cache=false
kotlin.jvm.target.validation.mode=warning
android.suppressUnsupportedCompileSdk=36
''')
print('OK  Gradle-конфиги')
print()
print('=' * 64)
print('  Проект suba_run_v8 создан. Далее -> ячейка 2/10 (модели данных)')
print('=' * 64)


Creating project suba_run_v8...
Resolving dependencies in `suba_run_v8`...
Got dependencies in `suba_run_v8`.
Wrote 131 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd suba_run_v8
  $ flutter run

Your application code is in suba_run_v8/lib/main.dart.

OK  pubspec.yaml
OK  AndroidManifest.xml
OK  MainActivity.kt
OK  Gradle-конфиги

  Проект suba_run_v8 создан. Далее -> ячейка 2/10 (модели данных)


In [69]:
# @title 📦 Ячейка 2/10: Модели данных (constants / snapshot / ROM-таблицы)
# ============================================================================
# constants.dart        — профиль A2TB100B + пороги турбо-мониторинга (EJ20X)
# models/live_snapshot  — единый live-снимок по каноническим ключам (rpm, boost,
#                         iam, fbkc, fkl, kca...) — опрос больше НЕ зависит от имён PID
# models/rom_table      — определения карт + чтение значений из .bin ROM
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/constants.dart ============
with open('lib/constants.dart', 'w') as f:
    f.write('''class AppConstants {
  static const String appVersion = '8.0.0';
  static const String appName = 'SUBA RUN V8';
  static const String calId = 'A2TB100B';
  static const String ecuIdExpected = '5204584007';
  static const String engine = 'EJ20X (2.0 Turbo, AVCS)';
  static const double gasolineDensity = 745.0; // г/л
  static const double stoich = 14.7;

  // ── Пороги турбо-мониторинга (Subaru-специфика, не Nissan!) ──
  // FBKC/FKL — отрицательные градусы коррекции. Чем "глубже" минус, тем хуже.
  static const double fbkcWarn = -1.0;
  static const double fbkcDanger = -2.8;
  static const double fklWarn = -1.0;
  static const double fklDanger = -2.8;
  // IAM — множитель опережения 0..1 (float). Падение ниже 1.0 = детонация была.
  static const double iamWarn = 0.999;
  static const double iamDanger = 0.5;
  // Рассогласование буста (факт - цель), бар
  static const double boostErrWarn = 0.15;
  static const double boostErrDanger = 0.25;
  static const double overboostDanger = 1.35; // бар абсолютного избытка
  static const int ectWarn = 100;
  static const int ectDanger = 108;
  // Под бустом смесь беднее этого порога — опасно для поршней/колец
  static const double afrBoostLean = 12.0;
  static const double boostActive = 0.05; // бар — считаем что "под бустом"

  // Автолог
  static const int autoLogRpm = 1500;
  static const int autoLogIdleSec = 25;

  // Опрос (базовые значения, меняются в настройках)
  static const int elmInitTimeoutMs = 2500;
  static const int elmCmdTimeoutMs = 700;
  static const int maxBlockBytes = 0x50; // дефолтный потолок одного A8-запроса
}
''')
print('OK  lib/constants.dart')

# ============ lib/models/live_snapshot.dart ============
with open('lib/models/live_snapshot.dart', 'w') as f:
    f.write('''import '../constants.dart';

/// Единый live-снимок ECU.
/// Значения хранятся по КАНОНИЧЕСКИМ ключам (rpm, boost, iam, fbkc...),
/// которые генератор PID-библиотеки вычисляет из адресов/имён logger.xml.
/// Благодаря этому UI не зависит от того, какой именно вариант PID
/// (1-byte / 2-byte / 4-byte float) дал значение.
class LiveSnapshot {
  final DateTime ts;
  final Map<String, double> c;    // canon -> value
  final Map<String, double> byId; // pid id -> value (для CSV/отладки)

  LiveSnapshot(this.ts, this.c, [this.byId = const {}]);

  double g(String k, [double def = 0]) => c[k] ?? def;
  bool has(String k) => c[k] != null;

  double get rpm    => g('rpm');
  double get speed  => g('speed');
  double get ect    => g('ect');
  double get iat    => g('iat');
  double get boost  => g('boost');           // bar относительное
  double get tps    => g('tps');             // %
  double get kca    => g('kca');             // Knock Correction Advance = итоговый УОЗ
  double get fbkc   => g('fbkc');            // Feedback Knock Correction (обычно <= 0)
  double get fkl    => g('fkl');             // Fine Learning Knock Correction
  double get iam    => g('iam', 1.0);        // IAM 0..1
  double get afr    => g('afr', AppConstants.stoich);
  double get maf    => g('maf');             // g/s
  double get mafV   => g('mafV');
  double get tboost => g('tboost');          // bar целевой (relative)
  double get wgd    => g('wgd');             // % wastegate duty
  double get injMs  => g('injms');
  double get stft   => g('stft');            // A/F Correction #1
  double get ltft   => g('ltft');            // A/F Learning #1
  double get avcs   => g('avcs');            // град
  int    get knockSum => g('knocksum').round();
  double get batt   => g('batt');

  double get boostErr => has('tboost') ? (boost - tboost) : 0.0;
  bool   get underBoost => boost >= AppConstants.boostActive;

  /// Расход л/ч на основе MAF и AFR (как в V6, но MAF Subaru — г/с напрямую)
  double get fuelLph {
    if (maf <= 0 || afr <= 0) return 0;
    final gps = maf / afr;                    // г/с топлива
    return gps * 3600.0 / AppConstants.gasolineDensity;
  }

  String get mode {
    if (rpm < 1100 && speed < 3) return 'IDLE';
    if (tps > 85) return 'WOT';
    if (underBoost) return 'BOOST';
    if (rpm > 1500 && tps < 12 && speed > 40) return 'COAST';
    return 'CRUISE';
  }

  LiveSnapshot merge(LiveSnapshot newer) {
    final m = Map<String, double>.from(c)..addAll(newer.c);
    final i = Map<String, double>.from(byId)..addAll(newer.byId);
    return LiveSnapshot(newer.ts, m, i);
  }
}
''')
print('OK  lib/models/live_snapshot.dart')

# ============ lib/models/rom_table.dart ============
with open('lib/models/rom_table.dart', 'w') as f:
    f.write('''import 'dart:math' as math;
import 'dart:typed_data';

/// Колонка данных ROM (ось или Z-данные): где лежит, как трактовать байты.
/// [to] — raw -> физ. величина (toexpr из ECUFlash-дефинишна)
/// [fr] — физ. величина -> raw (frexpr), может быть null => read-only
class RomCol {
  final int address;
  final int count;
  final String storage; // float | uint8 | int8 | uint16 | int16
  final String endian;  // big | little
  final double Function(double num)? to;
  final double Function(double num)? fr;

  const RomCol({
    required this.address,
    required this.count,
    this.storage = 'float',
    this.endian = 'big',
    this.to,
    this.fr,
  });

  int get sizeOf {
    switch (storage) {
      case 'float':
      case 'uint32':
      case 'int32':
        return 4;
      case 'uint16':
      case 'int16':
        return 2;
      default:
        return 1;
    }
  }

  int get byteLen => sizeOf * count;

  int readRaw(List<int> rom, ByteData bd, int index) {
    final off = address + index * sizeOf;
    switch (storage) {
      case 'uint16':
        return endian == 'little' ? bd.getUint16(off, Endian.little) : bd.getUint16(off, Endian.big);
      case 'int16':
        return endian == 'little' ? bd.getInt16(off, Endian.little) : bd.getInt16(off, Endian.big);
      case 'int8':
        return bd.getInt8(off);
      case 'uint8':
      default:
        return bd.getUint8(off);
    }
  }

  double readFloat(List<int> rom, ByteData bd, int index) {
    final off = address + index * sizeOf;
    if (storage == 'float') {
      return endian == 'little' ? bd.getFloat32(off, Endian.little) : bd.getFloat32(off, Endian.big);
    }
    return readRaw(rom, bd, index).toDouble();
  }

  double value(List<int> rom, ByteData bd, int index) {
    final raw = readFloat(rom, bd, index);
    return to == null ? raw : to!(raw);
  }
}

/// Описание карты из ECUFlash-дефинишна A2TB100B (генерируется ячейкой 4/8)
class RomTableDef {
  final String name;
  final String category;
  final int address;
  final int rows;
  final int cols;
  final bool swapxy;
  final String units;
  final RomCol data;
  final RomCol? xAxis;
  final RomCol? yAxis;
  final double minHint;
  final double maxHint;

  const RomTableDef({
    required this.name,
    required this.category,
    required this.address,
    required this.rows,
    required this.cols,
    required this.data,
    this.swapxy = false,
    this.units = '',
    this.xAxis,
    this.yAxis,
    this.minHint = double.nan,
    this.maxHint = double.nan,
  });

  bool get is3d => rows > 1 || (yAxis != null && cols > 1);
  bool get editable => data.fr != null;
  String get addrHex => '0x${address.toRadixString(16).toUpperCase()}';
}

/// Загруженная из .bin карта со значениями
class RomTable {
  final RomTableDef def;
  final List<double> xValues; // физические, длина = cols
  final List<double> yValues; // физические, длина = rows
  final List<List<double>> z; // [rows][cols]

  RomTable({required this.def, required this.xValues, required this.yValues, required this.z});

  double get minV => z.expand((r) => r).fold(double.infinity, math.min);
  double get maxV => z.expand((r) => r).fold(-double.infinity, math.max);

  void setCell(int r, int c, double v) => z[r][c] = v;

  int _nearest(List<double> axis, double v) {
    var idx = 0;
    var best = double.infinity;
    for (var i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  double at(double x, double y) => z[_nearest(yValues, y)][_nearest(xValues, x)];

  List<List<String>> toCsvRows() {
    final rows = <List<String>>[];
    rows.add(['${def.name} [${def.units}]', ...xValues.map((e) => e.toStringAsFixed(1))]);
    for (var r = 0; r < def.rows; r++) {
      rows.add([
        yValues.isNotEmpty ? yValues[r].toStringAsFixed(1) : '$r',
        ...z[r].map((e) => e.toStringAsFixed(3)),
      ]);
    }
    return rows;
  }
}
''')
print('OK  lib/models/rom_table.dart')
print()
print('=' * 64)
print('  Модели готовы. Далее -> ячейка 3/10 (SSM2-over-CAN протокол)')
print('=' * 64)


OK  lib/constants.dart
OK  lib/models/live_snapshot.dart
OK  lib/models/rom_table.dart

  Модели готовы. Далее -> ячейка 3/10 (SSM2-over-CAN протокол)


In [70]:
# @title 🔌 Ячейка 3/10: SSM2-over-CAN стек (ELM327) + блочный опрос
# ============================================================================
# ГЛАВНОЕ УСКОРЕНИЕ ОТНОСИТЕЛЬНО V7:
#   V7: 1 CAN-запрос = 1 PID  => ~150 запросов на цикл (6-9 сек, 0.1 Гц)
#   V8: SSM2 читает ДИАПАЗОН адресов одним A8-запросом. Соседние PID
#       склеиваются в блоки (до maxBlock байт) => весь набор = 8-14 запросов.
#       + приоритетные ярусы: fast каждый цикл, mid 1/4, slow 1/25.
#       + ответ-ориентированный пайплайн (НИ ОДНОГО sleep между кадрами).
#       + count-1 семантика байта длины (по спецификации SSM2) с фолбэком.
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

with open('lib/ssm/ssm_elm.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'dart:convert';
import 'dart:typed_data';

import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../models/live_snapshot.dart';

// ─────────────────────────────────────────────────────────────
// Статистика протокола (чтобы тормоза было ВИДНО на экране)
// ─────────────────────────────────────────────────────────────
class SsmStats {
  int framesOk = 0;
  int framesErr = 0;
  int noData = 0;
  double lastMs = 0;
  double avgMs = 0;
  double hz = 0;
  int _snapCounter = 0;
  DateTime _hzStart = DateTime.now();

  void ok(double ms) {
    framesOk++;
    lastMs = ms;
    avgMs = avgMs == 0 ? ms : avgMs * 0.85 + ms * 0.15;
  }

  void err() => framesErr++;
  void nodata() => noData++;

  void snap() {
    _snapCounter++;
    final d = DateTime.now().difference(_hzStart).inMilliseconds;
    if (d >= 1000) {
      hz = _snapCounter * 1000.0 / d;
      _snapCounter = 0;
      _hzStart = DateTime.now();
    }
  }

  void reset() {
    framesOk = 0;
    framesErr = 0;
    noData = 0;
    avgMs = 0;
    hz = 0;
  }
}

// ─────────────────────────────────────────────────────────────
// Блок чтения: непрерывный диапазон адресов + PID внутри
// ─────────────────────────────────────────────────────────────
class SsmBlock {
  final int start;
  final int len;
  final List<SubaruPid> pids;
  final int prio; // 1 fast, 2 mid, 3 slow
  int consecErr = 0;
  int skipUntilCycle = 0;

  SsmBlock(this.start, this.len, this.pids, this.prio);

  String get rangeHex => '0x${start.toRadixString(16).toUpperCase().padLeft(6, '0')}'
      '+${len.toRadixString(16).toUpperCase()}';
}

enum SsmState { disconnected, connecting, elmReady, ecuReady, polling, error }

// ─────────────────────────────────────────────────────────────
// Низкоуровневый ELM327 -> SSM2 over CAN (0x7E0 / 0x7E8, 500k, 11bit)
// ─────────────────────────────────────────────────────────────
class SsmElm {
  BluetoothConnection? _conn;
  StreamSubscription<Uint8List>? _sub;
  final StringBuffer _rx = StringBuffer();
  Completer<void>? _promptWaiter;

  final stats = SsmStats();
  SsmState state = SsmState.disconnected;

  int stTimeoutCode = 0x08; // AT ST (x4 мс): 0x08 = 32 мс ожидание байтов ответа
  String elmVersion = '';
  String ecuId = '';

  final StreamController<SsmState> stateCtl = StreamController<SsmState>.broadcast();
  Stream<SsmState> get onState => stateCtl.stream;

  void _setState(SsmState s) {
    state = s;
    if (!stateCtl.isClosed) stateCtl.add(s);
  }

  /// Публичный маркер для поллера: опрос идёт / остановлен
  void markPolling(bool v) => _setState(v ? SsmState.polling : SsmState.ecuReady);

  Future<bool> connect(String address) async {
    _setState(SsmState.connecting);
    try {
      _conn = await BluetoothConnection.toAddress(address)
          .timeout(const Duration(seconds: 12));
    } catch (_) {
      _setState(SsmState.error);
      return false;
    }
    _sub = _conn!.input!.listen(_onData, onDone: () => disconnect(), onError: (_) => disconnect());
    final ok = await setupElm();
    return ok;
  }

  void _onData(Uint8List chunk) {
    for (final b in chunk) {
      if (b == 0x3E) { // '>'
        _promptWaiter?.complete();
        _promptWaiter = null;
      } else {
        _rx.writeCharCode(b);
      }
    }
  }

  Future<String> transact(String cmd, {int timeoutMs = AppConstants.elmCmdTimeoutMs}) async {
    if (_conn == null) throw StateError('not connected');
    _rx.clear();
    _promptWaiter = Completer<void>();
    _conn!.output.add(Uint8List.fromList(ascii.encode('$cmd\\r')));
    await _conn!.output.allSent;
    try {
      await _promptWaiter!.future.timeout(Duration(milliseconds: timeoutMs));
    } catch (_) {
      // таймаут: вернём что успели накопить (парсер сам решит)
    }
    return _rx.toString();
  }

  Future<String> _expectOk(String cmd, {int timeoutMs = 900}) async {
    final r = await transact(cmd, timeoutMs: timeoutMs);
    return r.toUpperCase();
  }

  Future<bool> setupElm() async {
    // Сброс: ждём баннер ELM327
    String resp = '';
    for (var i = 0; i < 3; i++) {
      resp = await transact('ATZ', timeoutMs: AppConstants.elmInitTimeoutMs);
      if (resp.toUpperCase().contains('ELM327')) break;
      await Future.delayed(const Duration(milliseconds: 300));
    }
    if (!resp.toUpperCase().contains('ELM327')) {
      _setState(SsmState.error);
      return false;
    }
    elmVersion = RegExp(r'ELM327[^\\r\\n]*').firstMatch(resp)?.group(0) ?? 'ELM327';

    final st = stTimeoutCode.toRadixString(16).padLeft(2, '0').toUpperCase();
    final steps = <String>[
      'ATE0',   // без эха
      'ATL0',   // без linefeed
      'ATS0',   // без пробелов — чистый hex-поток
      'ATH0',   // без CAN-заголовков — данные сразу
      'ATAL',   // allow long (>7 байт) — ISO-TP мультифрейм SSM2
      'ATST$st',// короткий таймаут ожидания (дефолт 200мс — тормоз!)
      'ATAT0',  // фиксированный тайминг, без адаптации
      'ATSP6',  // ISO 15765-4 CAN, 11 bit, 500 kbit
      'ATSH7E0' // наш моторный ECU
    ];
    for (final cmd in steps) {
      final r = await _expectOk(cmd);
      if (!r.contains('OK') && !r.contains('ELM')) {
        // ATSH на тонких клонах иногда ругается — пробуем продолжить
        if (!cmd.startsWith('ATSH')) { /* мягко игнорируем */ }
      }
    }
    // Фильтр приёма 7E8 — на части клонов CRA капризен, делаем best-effort
    await _expectOk('ATCRA7E8');
    _setState(SsmState.elmReady);
    return true;
  }

  /// SSM2 init. По CAN шлём тот же кадр, что и по K-line:
  /// 80 10 F0 01 BF 40  ->  ответ содержит 0xFF + ECU ID (ASCII)
  Future<String?> ecuInit() async {
    for (var attempt = 0; attempt < 3; attempt++) {
      final r = await transact('8010F001BF40', timeoutMs: 1400);
      final hex = r.replaceAll(RegExp(r'[^0-9A-Fa-f]'), '').toUpperCase();
      final idx = hex.indexOf('FF');
      if (idx >= 0 && hex.length >= idx + 2 + 22) {
        final idBytes = <int>[];
        for (var i = idx + 2; i + 2 <= hex.length && idBytes.length < 11; i += 2) {
          final code = int.tryParse(hex.substring(i, i + 2), radix: 16) ?? 0;
          if (code >= 0x30 && code <= 0x39) idBytes.add(code); // только цифры
        }
        ecuId = ascii.decode(idBytes);
        _setState(SsmState.ecuReady);
        return ecuId;
      }
      // некоторые ECU отвечают FF без ID в первом кадре — всё равно сессия жива
      if (idx >= 0) {
        _setState(SsmState.ecuReady);
        return ecuId.isEmpty ? 'OK' : ecuId;
      }
      await Future.delayed(const Duration(milliseconds: 250));
    }
    return null;
  }

  /// Чтение len байт начиная с адреса addr одним SSM2 A8-запросом.
  /// Байт счётчика в SSM2 = (количество байт - 1). Фолбэк: если ECU вернул меньше,
  /// повторяем с "прямым" счётчиком (некоторые прошивки трактуют его как abs).
  Future<Uint8List?> readBytes(int addr, int len) async {
    if (len < 1 || len > 255) return null;
    final a = addr.toRadixString(16).padLeft(6, '0').toUpperCase();
    final cnt1 = ((len - 1) & 0xFF).toRadixString(16).padLeft(2, '0').toUpperCase();
    final sw = Stopwatch()..start();

    var data = _parseRead(await transact('A8' '00' '$a$cnt1'), addr, len);
    if (data == null) {
      final cntAbs = (len & 0xFF).toRadixString(16).padLeft(2, '0').toUpperCase();
      data = _parseRead(await transact('A8' '00' '$a$cntAbs'), addr, len);
    }
    sw.stop();

    if (data != null) {
      stats.ok(sw.elapsedMicroseconds / 1000.0);
    } else {
      stats.err();
    }
    return data;
  }

  Uint8List? _parseRead(String resp, int addr, int len) {
    final hex = resp.replaceAll(RegExp(r'[^0-9A-Fa-f]'), '').toUpperCase();
    if (hex.contains('NODATA') || hex.isEmpty) { stats.nodata(); return null; }
    if (hex.contains('7F')) return null; // negative response
    var from = 0;
    final wantHi = (addr >> 16) & 0xFF, wantMid = (addr >> 8) & 0xFF, wantLo = addr & 0xFF;
    while (true) {
      final idx = hex.indexOf('E8', from);
      if (idx < 0) return null;
      // E8 00 00 00 aHi aMid aLo [data...]
      if (hex.length >= idx + 14) {
        final h = int.tryParse(hex.substring(idx + 2, idx + 4), radix: 16) ?? -1;
        final m = int.tryParse(hex.substring(idx + 4, idx + 6), radix: 16) ?? -1;
        final a1 = int.tryParse(hex.substring(idx + 8, idx + 10), radix: 16) ?? -1;
        final a2 = int.tryParse(hex.substring(idx + 10, idx + 12), radix: 16) ?? -1;
        final a3 = int.tryParse(hex.substring(idx + 12, idx + 14), radix: 16) ?? -1;
        final addrOk = (h == 0 || h == 0xFF) && m == 0 && a1 == wantHi && a2 == wantMid && a3 == wantLo;
        if (addrOk) {
          final avail = (hex.length - (idx + 14)) ~/ 2;
          if (avail >= len) {
            final out = Uint8List(len);
            for (var i = 0; i < len; i++) {
              out[i] = int.parse(hex.substring(idx + 14 + i * 2, idx + 16 + i * 2), radix: 16);
            }
            return out;
          }
        }
      }
      from = idx + 2;
      if (from >= hex.length) return null;
    }
  }

  Future<void> disconnect() async {
    try { await _sub?.cancel(); } catch (_) {}
    _sub = null;
    try { await _conn?.close(); } catch (_) {}
    _conn = null;
    _rx.clear();
    _setState(SsmState.disconnected);
  }
}

// ─────────────────────────────────────────────────────────────
// Поллер: блочный план + приоритетные ярусы + реконнект
// ─────────────────────────────────────────────────────────────
class SsmPoller {
  final SsmElm elm;
  int maxBlock = AppConstants.maxBlockBytes;
  int gapTol = 2;       // склеивать адреса, если разрыв <= gapTol байт
  int midEveryN = 4;    // средний ярус: каждый 4-й цикл
  int slowEveryN = 25;  // медленный ярус: каждый 25-й цикл

  List<SsmBlock> _blocks = [];
  final Map<String, double> _canon = {};
  final Map<String, double> _byId = {};

  Timer? _timer;
  bool _running = false;
  int _cycle = 0;
  int _globalErrStreak = 0;

  final StreamController<LiveSnapshot> _snapCtl = StreamController<LiveSnapshot>.broadcast();
  Stream<LiveSnapshot> get snapshots => _snapCtl.stream;
  LiveSnapshot? last;
  bool get isRunning => _running;
  int get blockCount => _blocks.length;
  int get pidCount => _blocks.fold(0, (a, b) => a + b.pids.length);

  SsmPoller(this.elm);

  /// Строим блоки: сортировка по адресу + склейка соседей.
  /// ЭТО главный ускоритель: вместо N запросов — ceil(N/плотность).
  List<SsmBlock> buildBlocks(List<SubaruPid> selected, {bool log = false}) {
    final list = selected.where((p) => p.len > 0).toList()
      ..sort((a, b) => a.address.compareTo(b.address));
    final blocks = <SsmBlock>[];
    var cur = <SubaruPid>[];
    var start = 0, end = 0, prio = 3;
    void flush() {
      if (cur.isEmpty) return;
      blocks.add(SsmBlock(start, end - start, List.of(cur), prio));
      cur = [];
    }
    for (final p in list) {
      final pStart = p.address, pEnd = p.address + p.len;
      if (cur.isEmpty) {
        start = pStart; end = pEnd; prio = p.priority; cur.add(p); continue;
      }
      if (pStart <= end + gapTol && (pEnd - start) <= maxBlock) {
        end = pEnd > end ? pEnd : end;
        if (p.priority < prio) prio = p.priority;
        cur.add(p);
      } else {
        flush();
        start = pStart; end = pEnd; prio = p.priority; cur.add(p);
      }
    }
    flush();
    _blocks = blocks;
    return blocks;
  }

  void start() {
    if (_running || _blocks.isEmpty) return;
    _running = true;
    _globalErrStreak = 0;
    elm.markPolling(true);
    _tick();
  }

  void stop() {
    _running = false;
    _timer?.cancel();
    if (elm.state == SsmState.polling) elm.markPolling(false);
  }

  Iterable<SsmBlock> _cycleBlocks(int cycle) sync* {
    for (final b in _blocks) {
      final due = b.prio == 1 || (b.prio == 2 && cycle % midEveryN == 0) || (b.prio == 3 && cycle % slowEveryN == 0);
      if (due && cycle >= b.skipUntilCycle) yield b;
    }
  }

  Future<void> _tick() async {
    if (!_running) return;
    _cycle++;
    for (final b in _cycleBlocks(_cycle)) {
      if (!_running) return;
      Uint8List? data;
      try {
        data = await elm.readBytes(b.start, b.len);
      } catch (_) {
        data = null; // отвал BT/адаптера — считаем ошибочным кадром
      }
      if (data != null) {
        b.consecErr = 0;
        _globalErrStreak = 0;
        for (final p in b.pids) {
          final off = p.address - b.start;
          if (off < 0 || off + p.len > data.length) continue;
          final sub = Uint8List.fromList(data.sublist(off, off + p.len));
          try {
            final v = p.formula(sub);
            if (v.isNaN || v.isInfinite) continue;
            _byId[p.id] = v;
            if (p.canon.isNotEmpty) _canon[p.canon] = v;
          } catch (_) {}
        }
      } else {
        b.consecErr++;
        _globalErrStreak++;
        if (b.consecErr >= 4) {
          b.skipUntilCycle = _cycle + 60; // блок «молчит» — отложим, не будем стоять в очереди
          b.consecErr = 0;
        }
      }
    }
    // снапшот после цикла быстрых блоков
    last = LiveSnapshot(DateTime.now(), Map.of(_canon), Map.of(_byId));
    elm.stats.snap();
    if (!_snapCtl.isClosed) _snapCtl.add(last!);

    // самолечение: много ошибок подряд -> re-init ECU
    if (_globalErrStreak > 40) {
      _globalErrStreak = 0;
      try {
        final id = await elm.ecuInit();
        if (id == null) { stop(); return; }
      } catch (_) {
        stop();
        return;
      }
    }
    // сразу следующий цикл — без искусственных задержек.
    // Темп ограничен реальным временем ответа ECU, а не sleep().
    _timer = Timer(Duration.zero, _tick);
  }

  Future<void> dispose() async {
    stop();
    await _snapCtl.close();
  }
}
''')
print('OK  lib/ssm/ssm_elm.dart  (ELM + A8-блоки + ярусы + статистика)')
print()
print('=' * 64)
print('  Протокол готов. Далее -> ячейка 4/10 (генератор PID + ROM-дефиниций)')
print('=' * 64)


OK  lib/ssm/ssm_elm.dart  (ELM + A8-блоки + ярусы + статистика)

  Протокол готов. Далее -> ячейка 4/10 (генератор PID + ROM-дефиниций)


In [71]:
# @title 🧬 Ячейка 4/10: Генератор PID-библиотеки и ROM-дефиниций A2TB100B
# ============================================================================
# v8.1.2 — поддержка ДВУХ форматов ECUFlash-дефинишен:
#   А) стандартный: type/category/scaling инлайн в <table>
#   Б) SubMerp (этот репозиторий): в CAL-файле только имя+адрес+подоси,
#      а type/category/scaling/elements — в Bases/32BITBASE.xml.
#      Мерж по нормализованному имени ("Target Boost_" == "Target Boost").
# Цепочка: A2TB100B -> A2TB100K -> 32BITBASE (качаем ВСЕ три файла).
# ============================================================================
import os, re, json, urllib.request
import xml.etree.ElementTree as ET
from collections import Counter

os.chdir('/content/suba_run_v8')

ECU_IDS = ['5204584007', '5204784007']
TARGET_CAL = 'A2TB100B'
LOGGER_URL = 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/RomRaider/logger/metric/logger.xml'
CAL_URLS = {
    'A2TB100B': 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/ECUFlash/subaru%20metric/Legacy%20GT/A2TB100B.xml',
    'A2TB100K': 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/ECUFlash/subaru%20metric/Legacy%20GT%20spec.B/A2TB100K.xml',
    '32BITBASE': 'https://raw.githubusercontent.com/fredcooke/SubMerp/master/ECUFlash/subaru%20metric/Bases/32BITBASE.xml',
}

def fetch(url, path, min_warn=800):
    if not os.path.exists(path) or os.path.getsize(path) < 400:
        print('  загрузка %s ...' % url.split('/')[-1])
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=90) as r, open(path, 'wb') as f:
            f.write(r.read())
    sz = os.path.getsize(path)
    print('  %s: %.0f KB%s' % (path, sz / 1024, '  <- МАЛО' if sz < min_warn else ''))
    return sz

print('=' * 64)
print('  [1/4] Источники определений')
print('=' * 64)
fetch(LOGGER_URL, '/content/logger.xml', 500000)
for cal, url in CAL_URLS.items():
    fetch(url, '/content/%s.xml' % cal)

# ─────────────────────────────────────────────────────────────
# expr вида x*0.01953125 -> текст внутри Dart-замыкания
# ─────────────────────────────────────────────────────────────
def dart_body(expr):
    if not expr:
        return None
    e = str(expr).strip()
    # только числа/операторы/скобки/x; идентификаторы кроме одинокого x запрещены
    # (числа 1e-3 не считаются идентификатором — защита отгораживается lookbehind)
    if re.search(r'[^0-9\.\+\-\*/\(\)\sa-zA-Z]', e):
        return None
    idents = re.findall(r'(?<![0-9.])[a-zA-Z_]\w*', e)
    if any(i != 'x' for i in idents):
        return None
    # Нормализация числовых литералов ECUFlash -> канонические Dart-double:
    # .84 -> 0.84 · 5. -> 5.0 · 50 -> 50.0 · 1e-3 -> 0.001 · 14.7 -> 14.7
    num_pat = re.compile(r'(?<![0-9A-Za-z_.])(\d*\.\d+|\d+\.?\d*(?:[eE][+-]?\d+)?)')
    def _num(m):
        try:
            return repr(float(m.group(1)))
        except ValueError:
            return m.group(1)
    return num_pat.sub(_num, e)

def esc(s):
    return (s or '').replace('\\', '\\\\').replace("'", "\\'")

def fnum(v):
    try:
        return repr(float(v))
    except (TypeError, ValueError):
        return 'double.nan'

def norm_name(s):
    return re.sub(r'[^a-z0-9]+', '', (s or '').lower())

# ════════════════════════════════════════════════════════════
# ЧАСТЬ А: logger.xml -> PID
# ════════════════════════════════════════════════════════════
print()
print('=' * 64)
print('  [2/4] RomRaider logger.xml -> subaru_pids.g.dart')
print('=' * 64)

lroot = ET.parse('/content/logger.xml').getroot()

def store_default(length, addr_int):
    if length == 4 and addr_int >= 0xFF0000:
        return 'float'
    return 'uint8' if length == 1 else ('uint16' if length == 2 else 'bytes')

def pick_conv(el, length, addr_int):
    convs = el.findall('.//conversion')
    if not convs:
        return None
    if store_default(length, addr_int) == 'float':
        for c in convs:
            if (c.get('storagetype') or '').lower() == 'float':
                return c
    return convs[0]

CANON_RULES = [
    (r'^engine speed$', 'rpm'),
    (r'^vehicle speed$', 'speed'),
    (r'^coolant temperature$', 'ect'),
    (r'^intake air temperature|^air temp', 'iat'),
    (r'^throttle opening angle$|^throttle plate opening angle', 'tps'),
    (r'^manifold relative pressure', 'boost'),
    (r'^target boost relative|^target boost', 'tboost'),
    (r'^knock correction advance', 'kca'),
    (r'^feedback knock correction', 'fbkc'),
    (r'^fine learning knock correction', 'fkl'),
    (r'^iam(\s|\(|$)', 'iam'),
    (r'^a/f sensor #1$', 'afr'),
    (r'^a/f correction #1$', 'stft'),
    (r'^a/f learning #1$', 'ltft'),
    (r'^mass air flow', 'maf'),
    (r'^mass airflow sensor voltage', 'mafV'),
    (r'injector.*pulse|pulse width', 'injms'),
    (r'^primary wastegate duty cycle$', 'wgd'),
    (r'^avcs (intake|inlet) left', 'avcs'),
    (r'^knock sum', 'knocksum'),
    (r'^battery voltage|battery.*voltage', 'batt'),
]
FAST = {'rpm', 'speed', 'boost', 'tps', 'kca', 'fbkc', 'iam', 'afr', 'maf'}
MID = {'ect', 'iat', 'tboost', 'fkl', 'wgd', 'stft', 'ltft', 'injms', 'avcs', 'mafV', 'batt'}

def canon_for(name):
    n = name.lower().strip()
    for rx, key in CANON_RULES:
        if re.search(rx, n):
            return key
    return ''

def cat_of(unit, name):
    u = (unit + ' ' + name).lower()
    if re.search(r'boost|wastegate|turbo|tgv', u): return 'turbo'
    if re.search(r'timing|knock|ignition|iam|spark|advance|misfire', u): return 'ignition'
    if re.search(r'fuel|injector|a/f|lambda|o2|oxygen|ethanol', u): return 'fuel'
    if re.search(r'temp|coolant', u): return 'temp'
    if re.search(r'maf|airflow|manifold|pressure|baro', u): return 'air'
    if re.search(r'throttle|pedal|accelerator', u): return 'throttle'
    if re.search(r'battery|voltage|alternator', u): return 'electric'
    if re.search(r'gear|speed|rpm|engine|avcs|cam', u): return 'engine'
    return 'other'

pids = []
seen = set()

def collect(el, xmlid, addr_int, length, name, desc, conv):
    body = dart_body(conv.get('expr') or 'x')
    if body is None:
        return
    base = re.sub(r'[^0-9A-Za-z]+', '_', name.upper()).strip('_')[:28] or xmlid
    pid_id, k = base, 1
    while pid_id in seen:
        k += 1
        pid_id = '%s_%d' % (base, k)
    seen.add(pid_id)
    storage = (conv.get('storagetype') or store_default(length, addr_int)).lower()
    pids.append(dict(src=xmlid[0], xmlid=xmlid, id=pid_id, name=name,
                     desc=(desc or name)[:90], unit=conv.get('units') or '',
                     addr=addr_int, len=length, storage=storage, body=body))

for param in lroot.iter('parameter'):
    xmlid = param.get('id') or ''
    if not xmlid.startswith('P'):
        continue
    a_el = param.find('address')
    if a_el is None or not (a_el.text or '').strip():
        continue
    addr_int = int(a_el.text.strip(), 16)
    length = int(a_el.get('length') or '1')
    conv = pick_conv(param, length, addr_int)
    if conv is None:
        continue
    collect(param, xmlid, addr_int, length,
            (param.get('name') or xmlid).strip(),
            param.findtext('description'), conv)

for ecuparam in lroot.iter('ecuparam'):
    xmlid = ecuparam.get('id') or ''
    if not xmlid.startswith('E'):
        continue
    matched, mlen = None, 1
    for ecu in ecuparam.findall('ecu'):
        ids = [x.strip() for x in (ecu.get('id') or '').split(',') if x.strip()]
        if any(x in ids for x in ECU_IDS):
            a = ecu.find('address')
            if a is not None and (a.text or '').strip():
                matched, mlen = a.text.strip(), int(a.get('length') or '1')
                break
    if not matched:
        continue
    conv = pick_conv(ecuparam, mlen, int(matched, 16))
    if conv is None:
        continue
    collect(ecuparam, xmlid, int(matched, 16), mlen,
            (ecuparam.get('name') or xmlid).strip(),
            ecuparam.get('name'), conv)

by_canon = {}
for p in sorted(pids, key=lambda q: (0 if q['addr'] < 0xFF0000 else 1, q['addr'], -q['len'])):
    key = canon_for(p['name'])
    if key and key not in by_canon:
        by_canon[key] = p['id']
        p['canon'] = key
for p in pids:
    p.setdefault('canon', '')

def prio_of(p):
    if p['canon'] in FAST: return 1
    if p['canon'] in MID: return 2
    if p['canon']: return 2
    return 3

pids.sort(key=lambda q: (q['addr'], q['len']))

if len(pids) < 100:
    raise SystemExit('ОШИБКА: распознано только %d PID.' % len(pids))

dart = []
dart.append('''// АВТОСГЕНЕРИРОВАНО ячейкой 4/10 SUBA RUN V8
// RomRaider logger.xml (metric), ECU 5204584007 / 5204784007 (A2TB100B)
// ignore_for_file: constant_identifier_names
import 'dart:typed_data';

typedef PidFormula = double Function(List<int> b);

double _f32(List<int> b) {
  final l = b.length >= 4 ? b.sublist(0, 4) : <int>[...b, ...List.filled(4 - b.length, 0)];
  return ByteData.sublistView(Uint8List.fromList(l)).getFloat32(0, Endian.big);
}

double _raw(List<int> b, String storage) {
  switch (storage) {
    case 'float':
      return _f32(b);
    case 'int8':
      final v = b[0];
      return (v < 128 ? v : v - 256).toDouble();
    case 'uint16':
      return ((b[0] << 8) | b[1]).toDouble();
    case 'int16':
      var v = (b[0] << 8) | b[1];
      if (v >= 32768) v -= 65536;
      return v.toDouble();
    case 'bytes':
      var v = 0;
      for (final x in b) { v = (v << 8) | x; }
      return v.toDouble();
    case 'uint8':
    default:
      return b[0].toDouble();
  }
}

class SubaruPid {
  final String id;
  final String xmlId;
  final String name;
  final String desc;
  final String unit;
  final String category;
  final int address;
  final int len;
  final String storage;
  final int priority; // 1 fast / 2 mid / 3 slow
  final String canon;
  final PidFormula formula;
  const SubaruPid({
    required this.id, required this.xmlId, required this.name, required this.desc,
    required this.unit, required this.category, required this.address, required this.len,
    required this.storage, required this.priority, required this.canon, required this.formula,
  });
  String get addrHex => '0x' + address.toRadixString(16).toUpperCase().padLeft(6, '0');
}

class SubaruPids {
  static final List<SubaruPid> all = [
''')
for p in pids:
    dart.append("    SubaruPid(id: '%s', xmlId: '%s', name: '%s', desc: '%s',\n"
                "        unit: '%s', category: '%s', address: 0x%06X, len: %d,\n"
                "        storage: '%s', priority: %d, canon: '%s',\n"
                "        formula: (b) { final x = _raw(b, '%s'); return %s; }),\n"
                % (esc(p['id']), esc(p['xmlid']), esc(p['name']), esc(p['desc']),
                   esc(p['unit']), cat_of(p['unit'], p['name']), p['addr'], p['len'],
                   p['storage'], prio_of(p), p['canon'], p['storage'], p['body']))
dart.append('''  ];

  static SubaruPid? byId(String id) {
    for (final p in all) { if (p.id == id) return p; }
    return null;
  }

  static List<SubaruPid> get defaults => all.where((p) => p.canon.isNotEmpty).toList();

  static Map<String, List<SubaruPid>> byCategory() {
    final m = <String, List<SubaruPid>>{};
    for (final p in all) { m.putIfAbsent(p.category, () => []).add(p); }
    return m;
  }
}
''')
os.makedirs('lib/generated', exist_ok=True)
with open('lib/generated/subaru_pids.g.dart', 'w') as f:
    f.write(''.join(dart))

print('  PID всего: %d (P: %d, E: %d) · канонических: %d' % (
    len(pids), sum(1 for p in pids if p['src'] == 'P'),
    sum(1 for p in pids if p['src'] == 'E'), len(by_canon)))

# ════════════════════════════════════════════════════════════
# ЧАСТЬ Б: CAL + 32BITBASE -> карты (двухформатный парсер)
# ════════════════════════════════════════════════════════════
print()
print('=' * 64)
print('  [3/4] ECUFlash: мерж CAL-файлов с базой 32BITBASE')
print('=' * 64)

roots = {cal: ET.parse('/content/%s.xml' % cal).getroot() for cal in CAL_URLS}

def rom_list(root):
    return [root] if root.tag == 'rom' else list(root.iter('rom'))

roms_by_id = {}
for cal, root in roots.items():
    for rom in rom_list(root):
        romid = rom.find('romid')
        if romid is None:
            continue
        xmlid = (romid.findtext('xmlid') or '').strip()
        if xmlid:
            roms_by_id[xmlid] = rom

# Реестр именованных скейлингов + реестр "шаблонных" таблиц (с type) — из ВСЕХ файлов
scalings = {}     # name -> ET scaling element
base_tables = {}  # norm_name -> ET table element (с type attr)
for root in roots.values():
    for rom_el in rom_list(root):
        for sc in rom_el.findall('scaling'):
            if sc.get('name'):
                scalings[sc.get('name')] = sc
        for t in rom_el.iter('table'):
            if t.get('type'):
                base_tables.setdefault(norm_name(t.get('name')), t)
print('  скейлингов в реестре: %d · шаблонных таблиц: %d' % (len(scalings), len(base_tables)))

def resolve_chain(xmlid, seen, depth=0):
    """CAL-таблицы (с адресами): include сначала, свои перекрывают по имени"""
    if xmlid in seen or depth > 8 or xmlid not in roms_by_id:
        return {}, []
    seen.add(xmlid)
    rom = roms_by_id[xmlid]
    tables, chain = {}, [xmlid]
    for inc in rom.findall('include'):
        inc_id = (inc.text or '').strip()
        if not inc_id:
            continue
        sub_t, sub_c = resolve_chain(inc_id, seen, depth + 1)
        for k, v in sub_t.items():
            tables.setdefault(k, v)
        chain += sub_c
    for t in rom.findall('table'):
        tables[t.get('name') or 'unnamed'] = t
    return tables, chain

if TARGET_CAL not in roms_by_id:
    raise SystemExit('ОШИБКА: CAL %s не найден (%s)' % (TARGET_CAL, ', '.join(sorted(roms_by_id))))

tmap, chain = resolve_chain(TARGET_CAL, set())
romid_t = roms_by_id[TARGET_CAL].find('romid')
cpu = ((romid_t.findtext('cpu') or romid_t.findtext('memmodel') or '') if romid_t is not None else '')
fsize = ((romid_t.findtext('filesize') or '') if romid_t is not None else '')
mm = re.match(r'(\d+)\s*kb', (fsize or '').lower())
rom_bytes = int(mm.group(1)) * 1024 if mm else 1310720
print('  цепочка: %s · кандидатов: %d · CPU %s' % (' -> '.join(chain), len(tmap), cpu))

AX_X = ('X', 'x', 'X Axis', 'Static X Axis')
AX_Y = ('Y', 'y', 'Y Axis', 'Static Y Axis')

def sub_kind(sub):
    lbl = sub.get('type') or sub.get('name') or ''
    if lbl in AX_X: return 'x'
    if lbl in AX_Y: return 'y'
    return None

static_variants = ('Static X Axis', 'Static Y Axis')

def axis_resolve(cal_sub, base_sub):
    """Объединение информации об оси из CAL (адрес/elements) и базы (scaling/elements)."""
    info = dict(count=0, addr=None, sc=None, stat=None)
    for src in (cal_sub, base_sub):
        if src is None:
            continue
        if info['count'] == 0 and src.get('elements'):
            info['count'] = int(src.get('elements'))
        data = src.findall('data')
        if data and info['stat'] is None:
            vals = []
            for d in data:
                try:
                    vals.append(float((d.text or '0').strip().split()[0]))
                except (ValueError, IndexError):
                    vals.append(0.0)
            if info['count'] == 0:
                info['count'] = len(vals)
            info['stat'] = vals
        if info['addr'] is None and src.get('address'):
            try:
                info['addr'] = int(src.get('address'), 16)
            except ValueError:
                pass
        if info['sc'] is None:
            sc = src.find('scaling')
            if sc is None and src.get('scaling') and src.get('scaling') in scalings:
                sc = scalings[src.get('scaling')]
            if sc is not None:
                info['sc'] = sc
    return info

def sc_fields(sc):
    if sc is None:
        return None
    storage = (sc.get('storagetype') or 'float').lower()
    if storage not in ('float', 'uint8', 'int8', 'uint16', 'int16'):
        storage = 'float'
    return dict(
        storage=storage,
        endian=(sc.get('endian') or 'big').lower(),
        units=sc.get('units') or '',
        to=dart_body(sc.get('toexpr') or 'x'),
        fr=dart_body(sc.get('frexpr') or ''),
        vmin=fnum(sc.get('min')),
        vmax=fnum(sc.get('max')),
    )

def table_scaling(t, base):
    sc = t.find('scaling')
    if sc is None:
        ref = t.get('scaling') or (base.get('scaling') if base is not None else None)
        sc = scalings.get(ref) if ref else None
    return sc

all_tables = []
skipped = []
for name_raw, t in tmap.items():
    # карты без адреса (шаблоны базы) не тащим
    try:
        addr = int(t.get('address') or '', 16)
    except ValueError:
        skipped.append((name_raw, 'no address'))
        continue
    base = base_tables.get(norm_name(name_raw))
    cal_subs = {}
    for s in t.findall('table'):
        k = sub_kind(s)
        if k and k not in cal_subs:
            cal_subs[k] = s
    base_subs = {}
    if base is not None:
        for s in base.findall('table'):
            k = sub_kind(s)
            if k and k not in base_subs:
                base_subs[k] = s

    ttype = (t.get('type') or (base.get('type') if base is not None else '') or '').strip()
    if ttype not in ('1D', '2D', '3D'):
        # выводим по подосям (компактный SubMerp-формат)
        has_x, has_y = ('x' in cal_subs), ('y' in cal_subs)
        if has_x and has_y:
            ttype = '3D'
        elif has_x or has_y:
            ttype = '2D'
        else:
            ttype = '1D'
    cat = t.get('category') or (base.get('category') if base is not None else '') or 'other'
    swap = ((t.get('swapxy') or (base.get('swapxy') if base is not None else '') or 'false').lower() == 'true')

    sc = table_scaling(t, base)
    scf = sc_fields(sc)
    if scf is None or scf['to'] is None:
        skipped.append((name_raw, 'no scaling'))
        continue

    def build_axis(kind_label):
        cal_s = cal_subs.get(kind_label)
        base_s = base_subs.get(kind_label)
        if base_s is None and kind_label == 'x' and 'y' in base_subs:
            base_s = base_subs['y']  # у 2D базы может быть "одна" ось
        return axis_resolve(cal_s, base_s)

    x_el = y_el = None
    if ttype == '3D' or ttype == '2D':
        x_el = cal_subs.get('x') or cal_subs.get('y')  # у 2D в дампе ось бывает подписана "Y"
        y_el = cal_subs.get('y') if ttype == '3D' else None
        if x_el is None and 'x' in base_subs:
            x_el = None  # возьмём из base_subs позже
    # решаем оси
    if ttype == '1D':
        rows, cols = 1, 1
        xd, yd = 'null', 'null'
        xs, ys = 'null', 'null'
    else:
        ax_info = axis_resolve(cal_subs.get('x') or cal_subs.get('y'),
                               base_subs.get('x') or base_subs.get('y'))
        ay_info = axis_resolve(cal_subs.get('y'), base_subs.get('y')) if ttype == '3D' else None
        cols = ax_info['count']
        rows = (ay_info['count'] if ay_info else 1)

        def axis_dart(a):
            if a is None or a['count'] == 0:
                return 'null'
            if a['stat'] is not None and a['addr'] is None:
                return "RomCol(address: -1, count: %d, storage: 'static')" % a['count']
            if a['addr'] is None:
                return 'null'
            f = sc_fields(a['sc'])
            stx = f['storage'] if f else 'float'
            enx = f['endian'] if f else 'big'
            body = (f['to'] if f and f['to'] else 'x')
            return ("RomCol(address: 0x%X, count: %d, storage: '%s', endian: '%s', "
                    "to: (x) => (%s))" % (a['addr'], a['count'], stx, enx, body))

        def axis_vals(a):
            if a is not None and a['stat'] is not None:
                return '[' + ','.join(('%g' % v) for v in a['stat']) + ']'
            return 'null'

        xd, yd = axis_dart(ax_info), axis_dart(ay_info)
        xs, ys = axis_vals(ax_info), axis_vals(ay_info)
        if cols == 0 or rows == 0:
            skipped.append((name_raw, 'axis count %dx%d' % (rows, cols)))
            continue
    if rows * cols > 2048:
        skipped.append((name_raw, 'too big %dx%d' % (rows, cols)))
        continue

    all_tables.append(dict(
        name=name_raw, cat=cat, addr=addr, rows=rows, cols=cols, swapxy=swap,
        units=scf['units'], storage=scf['storage'], endian=scf['endian'],
        to=scf['to'], fr=('(x) => (%s)' % scf['fr']) if scf['fr'] else 'null',
        vmin=scf['vmin'], vmax=scf['vmax'], x=xd, y=yd, xs=xs, ys=ys,
        ttype=ttype))

n1 = sum(1 for t in all_tables if t['ttype'] == '1D')
n2 = sum(1 for t in all_tables if t['ttype'] == '2D')
n3 = sum(1 for t in all_tables if t['ttype'] == '3D')
print('  Карт распознано: %d (1D: %d · 2D: %d · 3D: %d) | пропущено: %d'
      % (len(all_tables), n1, n2, n3, len(skipped)))
for c, n in sorted(Counter(t['cat'] for t in all_tables).items(), key=lambda x: -x[1])[:16]:
    print('    %-30s %d' % (c, n))
if skipped:
    print('  примеры пропусков: ' + '; '.join('%s(%s)' % s for s in skipped[:10]))
# контрольные карты
for probe in ['targetboost', 'basetimingprimarycruise', 'primaryopenloopfueling',
              'knockcorrectionadvancemaxcruise', 'maxwastegateduty']:
    hit = next((t for t in all_tables if norm_name(t['name']) == probe), None)
    if hit:
        print('  ✓ %s: %dx%d [%s] @ 0x%X, %s' % (hit['name'], hit['rows'], hit['cols'],
                                                   hit['storage'], hit['addr'], hit['units']))

if len(all_tables) < 20:
    raise SystemExit('ОШИБКА: распознано только %d карт (цепочка %s).'
                     % (len(all_tables), ' -> '.join(chain)))

print()
print('=' * 64)
print('  [4/4] Генерация subaru_rom.g.dart')
print('=' * 64)

rd = []
rd.append('''// АВТОСГЕНЕРИРОВАНО ячейкой 4/10 SUBA RUN V8
// ECUFlash %s (цепочка: %s) — SH7058, subarucan
// ignore_for_file: constant_identifier_names
import '../models/rom_table.dart';

class SubaruRom {
  static const String calId = '%s';
  static const int romSize = %d;

  static final List<RomTableDef> tables = [
''' % (TARGET_CAL, ' / '.join(chain), TARGET_CAL, rom_bytes))
for i, t in enumerate(all_tables):
    rd.append("    RomTableDef(\n"
              "      name: '%s', category: '%s', address: 0x%X,\n"
              "      rows: %d, cols: %d, swapxy: %s,\n"
              "      units: '%s', minHint: %s, maxHint: %s,\n"
              "      data: RomCol(address: 0x%X, count: %d, storage: '%s', endian: '%s',\n"
              "          to: (x) => (%s), fr: %s),\n"
              "      xAxis: %s, yAxis: %s),\n"
              % (esc(t['name']), esc(t['cat']), t['addr'],
                 t['rows'], t['cols'], str(t['swapxy']).lower(),
                 esc(t['units']), t['vmin'], t['vmax'],
                 t['addr'], t['rows'] * t['cols'], t['storage'], t['endian'],
                 t['to'], t['fr'], t['x'], t['y']))
rd.append('''  ];

  /// Статические оси (Static Axis): ключ = x/y + индекс в tables
  static final Map<String, List<double>> staticAxes = {
''')
for i, t in enumerate(all_tables):
    if t['xs'] != 'null':
        rd.append("    'x%d': %s,\n" % (i, t['xs']))
    if t['ys'] != 'null':
        rd.append("    'y%d': %s,\n" % (i, t['ys']))
rd.append('''  };
}
''')
with open('lib/generated/subaru_rom.g.dart', 'w') as f:
    f.write(''.join(rd))

report = dict(pids=len(pids), canon=by_canon, tables=len(all_tables), chain=chain,
              types={'1D': n1, '2D': n2, '3D': n3},
              categories=dict(Counter(t['cat'] for t in all_tables)),
              skipped=skipped[:40], romBytes=rom_bytes)
with open('/content/v8_definitions_report.json', 'w') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print()
print('=' * 64)
print('  СГЕНЕРИРОВАНО: %d PID + %d карт (%s)' % (len(pids), len(all_tables), ' -> '.join(chain)))
print('  Далее -> ячейка 5/10 (сервисы ядра)')
print('=' * 64)


  [1/4] Источники определений
  /content/logger.xml: 2026 KB
  /content/A2TB100B.xml: 2 KB
  /content/A2TB100K.xml: 28 KB
  /content/32BITBASE.xml: 489 KB

  [2/4] RomRaider logger.xml -> subaru_pids.g.dart
  PID всего: 150 (P: 91, E: 59) · канонических: 19

  [3/4] ECUFlash: мерж CAL-файлов с базой 32BITBASE
  скейлингов в реестре: 399 · шаблонных таблиц: 813
  цепочка: A2TB100B -> A2TB100K -> 32BITBASE · кандидатов: 807 · CPU SH7058
  Карт распознано: 289 (1D: 104 · 2D: 143 · 3D: 42) | пропущено: 518
    Diagnostic Trouble Codes       103
    Ignition Timing - Knock Control 23
    Fueling - Warm-Up Enrichment   16
    Ignition Timing - Compensation 14
    Fueling - CL/OL Transition     13
    Boost Control - Turbo Dynamics 12
    Ignition Timing - Advance      12
    Fueling - Cranking             9
    Fueling - Tip-in Enrichment    9
    Drive-by-Wire Throttle (DBW)   9
    Fueling - Primary Open Loop    8
    Miscellaneous - Limits         7
    Boost Control - Target         6
  

/tmp/ipykernel_1746/1753225597.py:497: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  x_el = cal_subs.get('x') or cal_subs.get('y')  # у 2D в дампе ось бывает подписана "Y"
/tmp/ipykernel_1746/1753225597.py:507: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  ax_info = axis_resolve(cal_subs.get('x') or cal_subs.get('y'),
/tmp/ipykernel_1746/1753225597.py:508: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  base_subs.get('x') or base_subs.get('y'))


In [72]:
# @title 🧰 Ячейка 5/10: Сервисы ядра (настройки / логгер / алерты / ROM / анализатор)
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/services/settings_service.dart ============
with open('lib/services/settings_service.dart', 'w') as f:
    f.write('''import 'package:flutter/foundation.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../generated/subaru_pids.g.dart';

class SettingsService extends ChangeNotifier {
  static final SettingsService I = SettingsService._();
  SettingsService._();

  String btAddress = '';
  String btName = '';
  Set<String> enabledIds = {};
  int maxBlock = 0x50;
  int gapTol = 2;
  int midEveryN = 4;
  int slowEveryN = 25;
  int stCode = 8;
  bool autoLog = true;
  bool _loaded = false;

  Future<void> load() async {
    if (_loaded) return;
    _loaded = true;
    final p = await SharedPreferences.getInstance();
    btAddress = p.getString('btAddress') ?? '';
    btName = p.getString('btName') ?? '';
    enabledIds = (p.getStringList('enabledIds') ?? SubaruPids.defaults.map((e) => e.id).toList()).toSet();
    maxBlock = p.getInt('maxBlock') ?? 0x50;
    gapTol = p.getInt('gapTol') ?? 2;
    midEveryN = p.getInt('midEveryN') ?? 4;
    slowEveryN = p.getInt('slowEveryN') ?? 25;
    stCode = p.getInt('stCode') ?? 8;
    autoLog = p.getBool('autoLog') ?? true;
    notifyListeners();
  }

  Future<void> save() async {
    final p = await SharedPreferences.getInstance();
    await p.setString('btAddress', btAddress);
    await p.setString('btName', btName);
    await p.setStringList('enabledIds', enabledIds.toList());
    await p.setInt('maxBlock', maxBlock);
    await p.setInt('gapTol', gapTol);
    await p.setInt('midEveryN', midEveryN);
    await p.setInt('slowEveryN', slowEveryN);
    await p.setInt('stCode', stCode);
    await p.setBool('autoLog', autoLog);
    notifyListeners();
  }

  List<SubaruPid> get selectedPids {
    final list = SubaruPids.all.where((p) => enabledIds.contains(p.id)).toList();
    if (list.isEmpty) return SubaruPids.defaults;
    return list;
  }
}
''')
print('OK  settings_service.dart')

# ============ lib/services/logger_service.dart ============
with open('lib/services/logger_service.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'dart:io';

import 'package:csv/csv.dart';
import 'package:intl/intl.dart';
import 'package:path_provider/path_provider.dart';

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../models/live_snapshot.dart';

/// CSV-логгер. Исправлено относительно V6: IOSink всегда в try/catch,
/// flush дозированный, подписка корректно закрывается (нет утечек).
class LoggerService {
  StreamSubscription<LiveSnapshot>? _sub;
  IOSink? _sink;
  File? _file;
  int rows = 0;
  DateTime? startedAt;
  List<SubaruPid> _cols = [];
  bool get logging => _sink != null;
  String? get filePath => _file?.path;
  Timer? _flushTimer;

  Future<void> start(Stream<LiveSnapshot> stream, List<SubaruPid> cols) async {
    if (logging) return;
    _cols = cols;
    final dir = await getApplicationDocumentsDirectory();
    final logsDir = Directory('${dir.path}/logs');
    if (!logsDir.existsSync()) logsDir.createSync(recursive: true);
    final name = 'V8_${DateFormat('yyyyMMdd_HHmmss').format(DateTime.now())}.csv';
    _file = File('${logsDir.path}/$name');
    _sink = _file!.openWrite();
    rows = 0;
    startedAt = DateTime.now();
    _sink!.writeln(const ListToCsvConverter().convert([
      <String>['ts_ms', ...cols.map((c) => c.id), 'boostErr', 'fuelLph', 'mode']
    ]));
    _sub = stream.listen(_write, onError: (_) {});
    _flushTimer = Timer.periodic(const Duration(seconds: 2), (_) {
      try { _sink?.flush(); } catch (_) {}
    });
  }

  void _write(LiveSnapshot s) {
    final sk = _sink;
    if (sk == null) return;
    try {
      sk.writeln(const ListToCsvConverter().convert([
        <Object>[
          s.ts.millisecondsSinceEpoch,
          ..._cols.map((c) {
            final v = s.byId[c.id];
            return v == null ? '' : v.toStringAsFixed(4);
          }),
          s.has('tboost') ? s.boostErr.toStringAsFixed(3) : '',
          s.fuelLph.toStringAsFixed(3),
          s.mode,
        ]
      ]));
      rows++;
    } catch (_) {}
  }

  Future<String?> stop() async {
    _flushTimer?.cancel();
    await _sub?.cancel();
    _sub = null;
    try {
      await _sink?.flush();
      await _sink?.close();
    } catch (_) {}
    _sink = null;
    return _file?.path;
  }

  static Future<List<FileSystemEntity>> listLogs() async {
    final dir = await getApplicationDocumentsDirectory();
    final logsDir = Directory('${dir.path}/logs');
    if (!logsDir.existsSync()) return [];
    final list = logsDir.listSync().where((e) => e.path.endsWith('.csv')).toList()
      ..sort((a, b) => b.statSync().modified.compareTo(a.statSync().modified));
    return list;
  }

  /// Парсинг CSV обратно в строки canon->value (для анализатора «Из лога»)
  static Future<List<Map<String, double>>> parseLog(String path) async {
    final out = <Map<String, double>>[];
    try {
      final content = await File(path).readAsString();
      final rows = const CsvToListConverter(shouldParseNumbers: false).convert(content);
      if (rows.length < 2) return out;
      final head = rows.first.map((e) => e.toString()).toList();
      final canonIdx = <int, String>{};
      for (var i = 0; i < head.length; i++) {
        final h = head[i];
        if (h == 'boostErr' || h == 'fuelLph') {
          canonIdx[i] = h;
        } else {
          final pid = SubaruPids.byId(h);
          if (pid != null && pid.canon.isNotEmpty) canonIdx[i] = pid.canon;
        }
      }
      for (var r = 1; r < rows.length; r++) {
        final row = rows[r];
        final m = <String, double>{};
        canonIdx.forEach((i, key) {
          if (i < row.length) {
            final v = double.tryParse(row[i].toString());
            if (v != null) m[key] = v;
          }
        });
        if (m.isNotEmpty) out.add(m);
      }
    } catch (_) {}
    return out;
  }
}
''')
print('OK  logger_service.dart')

# ============ lib/services/alert_service.dart ============
with open('lib/services/alert_service.dart', 'w') as f:
    f.write('''import 'dart:async';

import '../constants.dart';
import '../models/live_snapshot.dart';

class AlertEvent {
  final DateTime ts;
  final int level; // 1 warn, 2 danger
  final String code;
  final String text;
  final Map<String, double> snapshot;
  AlertEvent(this.ts, this.level, this.code, this.text, this.snapshot);
}

/// Турбо-алерты Subaru: FBKC/FKL/IAM/буст — то, что реально важно на EJ20X,
/// а не Nissan-ориентированные knockRetard/VTC.
class AlertService {
  final List<AlertEvent> events = [];
  final _ctl = StreamController<AlertEvent>.broadcast();
  Stream<AlertEvent> get stream => _ctl.stream;
  final Map<String, DateTime> _cooldown = {};

  bool _cool(String code, [int sec = 5]) {
    final now = DateTime.now();
    final lastT = _cooldown[code];
    if (lastT != null && now.difference(lastT).inSeconds < sec) return false;
    _cooldown[code] = now;
    return true;
  }

  void _fire(int level, String code, String text, LiveSnapshot s) {
    if (!_cool(code)) return;
    final e = AlertEvent(s.ts, level, code, text, Map.of(s.c));
    events.insert(0, e);
    if (events.length > 200) events.removeLast();
    if (!_ctl.isClosed) _ctl.add(e);
  }

  void check(LiveSnapshot s) {
    if (s.rpm > 2500 && s.underBoost) {
      if (s.has('fbkc') && s.fbkc <= AppConstants.fbkcDanger) {
        _fire(2, 'FBKC', 'FBKC ${s.fbkc.toStringAsFixed(1)}° при ${s.boost.toStringAsFixed(2)} бар — детонация!', s);
      } else if (s.has('fbkc') && s.fbkc <= AppConstants.fbkcWarn) {
        _fire(1, 'FBKC', 'FBKC ${s.fbkc.toStringAsFixed(1)}° — лёгкая коррекция', s);
      }
      if (s.has('fkl') && s.fkl <= AppConstants.fklDanger) {
        _fire(2, 'FKL', 'FKL ${s.fkl.toStringAsFixed(1)}° — обученная коррекция, проверь топливо/настройку', s);
      }
      if (s.has('afr') && s.afr >= AppConstants.afrBoostLean) {
        _fire(2, 'AFR', 'Смесь ${s.afr.toStringAsFixed(1)} под бустом! Опасно бедно', s);
      }
    }
    if (s.has('iam') && s.iam < AppConstants.iamDanger) {
      _fire(2, 'IAM', 'IAM упал до ${s.iam.toStringAsFixed(2)} — ECU режет углы', s);
    } else if (s.has('iam') && s.iam < AppConstants.iamWarn) {
      _fire(1, 'IAM', 'IAM ${s.iam.toStringAsFixed(2)} < 1.0', s);
    }
    if (s.has('tboost') && s.tps > 50) {
      final err = s.boostErr;
      if (err >= AppConstants.boostErrDanger) {
        _fire(2, 'BSTERR', 'Перебуст: +${err.toStringAsFixed(2)} бар к цели', s);
      } else if (err <= -AppConstants.boostErrDanger) {
        _fire(1, 'BSTLOW', 'Недобуст: ${err.toStringAsFixed(2)} бар к цели', s);
      }
    }
    if (s.has('boost') && s.boost >= AppConstants.overboostDanger) {
      _fire(2, 'OVBST', 'Овербуст ${s.boost.toStringAsFixed(2)} бар!', s);
    }
    if (s.ect >= AppConstants.ectDanger) {
      _fire(2, 'ECT', 'Температура ОЖ ${s.ect.toStringAsFixed(0)}°C!', s);
    } else if (s.ect >= AppConstants.ectWarn) {
      _fire(1, 'ECT', 'Температура ОЖ ${s.ect.toStringAsFixed(0)}°C', s);
    }
  }

  void clear() => events.clear();
}
''')
print('OK  alert_service.dart')

# ============ lib/services/rom_service.dart ============
with open('lib/services/rom_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:typed_data';

import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';

class RomService {
  List<int>? rom;
  String fileName = '';
  ByteData? _bd;

  bool get loaded => rom != null;
  List<RomTableDef> get defs => SubaruRom.tables;

  Future<String?> pickAndLoad() async {
    final res = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['bin', 'hex', 'rom']);
    if (res == null || res.files.single.path == null) return null;
    final bytes = await File(res.files.single.path!).readAsBytes();
    return loadBytes(bytes, res.files.single.name);
  }

  String? loadBytes(List<int> bytes, String name) {
    if (bytes.length < 0x40000) return 'Файл слишком мал для SH7058 ROM';
    rom = bytes;
    fileName = name;
    _bd = ByteData.sublistView(Uint8List.fromList(bytes));
    return null; // null = ок
  }

  List<double> _readAxis(RomCol? col, int tableIndex, bool isX, int fallbackCount) {
    if (col == null) return List<double>.generate(fallbackCount, (i) => i.toDouble());
    if (col.storage == 'static') {
      return SubaruRom.staticAxes['${isX ? 'x' : 'y'}$tableIndex'] ??
          List<double>.generate(fallbackCount, (i) => i.toDouble());
    }
    final values = <double>[];
    for (var i = 0; i < col.count; i++) {
      try {
        values.add(col.value(rom!, _bd!, i));
      } catch (_) {
        values.add(double.nan);
      }
    }
    return values;
  }

  RomTable readTable(RomTableDef def) {
    final idx = SubaruRom.tables.indexOf(def);
    final xValues = _readAxis(def.xAxis, idx, true, def.cols);
    final yValues = _readAxis(def.yAxis, idx, false, def.rows);
    final z = List.generate(def.rows, (_) => List<double>.filled(def.cols, 0));
    for (var r = 0; r < def.rows; r++) {
      for (var c = 0; c < def.cols; c++) {
        final i = def.swapxy ? (c * def.rows + r) : (r * def.cols + c);
        try {
          z[r][c] = def.data.value(rom!, _bd!, i);
        } catch (_) {
          z[r][c] = double.nan;
        }
      }
    }
    return RomTable(def: def, xValues: xValues, yValues: yValues, z: z);
  }

  /// Запись правок в НОВЫЙ файл (оригинал не трогаем) — как NLP_MOD1 в V6
  Future<String?> saveMod(RomTable table) async {
    if (rom == null || !table.def.editable) return null;
    final mod = List<int>.of(rom!);
    final fr = table.def.data.fr!;
    final sizeOf = table.def.data.sizeOf;
    for (var r = 0; r < table.def.rows; r++) {
      for (var c = 0; c < table.def.cols; c++) {
        final i = table.def.swapxy ? (c * table.def.rows + r) : (r * table.def.cols + c);
        final off = table.def.address + i * sizeOf;
        final raw = fr(table.z[r][c]).round();
        switch (table.def.data.storage) {
          case 'uint16':
            mod[off] = (raw >> 8) & 0xFF; mod[off + 1] = raw & 0xFF; break;
          case 'int16':
            final v = raw < 0 ? raw + 65536 : raw;
            mod[off] = (v >> 8) & 0xFF; mod[off + 1] = v & 0xFF; break;
          case 'int8':
            mod[off] = raw < 0 ? raw + 256 : raw; break;
          default:
            mod[off] = raw.clamp(0, 255);
        }
      }
    }
    final dir = await getApplicationDocumentsDirectory();
    final base = fileName.replaceAll(RegExp(r'\.(bin|hex|rom)$', caseSensitive: false), '');
    // ignore: avoid_escaping_inner_quotes
    final f = File('${dir.path}/V8MOD_$base.bin');
    await f.writeAsBytes(mod);
    return f.path;
  }
}
''')
print('OK  rom_service.dart')

# ============ lib/services/analyzer_service.dart ============
with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'dart:math' as math;

import '../models/live_snapshot.dart';

/// Ось анализатора: источник значения + диапазон + число корзин
class AxisOpt {
  final String key;
  final String label;
  final double min;
  final double max;
  final int bins;
  const AxisOpt(this.key, this.label, this.min, this.max, this.bins);
}

/// Тепловая сетка "как в V6": биннинг X×Y, усреднение, счётчик попаданий,
/// слияние сеток (сплиттер логов) и живой режим из потока опроса.
class HeatGrid {
  final int cols;
  final int rows;
  final double xmin, xmax, ymin, ymax;
  late List<List<double>> sum;
  late List<List<int>> n;

  HeatGrid(this.cols, this.rows, this.xmin, this.xmax, this.ymin, this.ymax) {
    reset();
  }

  void reset() {
    sum = List.generate(rows, (_) => List.filled(cols, 0.0));
    n = List.generate(rows, (_) => List.filled(cols, 0));
  }

  void add(double x, double y, double v) {
    if (v.isNaN || x.isNaN || y.isNaN) return;
    if (x < xmin || x > xmax || y < ymin || y > ymax) return;
    final c = (((x - xmin) / (xmax - xmin)) * cols).floor().clamp(0, cols - 1);
    final r = (((y - ymin) / (ymax - ymin)) * rows).floor().clamp(0, rows - 1);
    sum[r][c] += v;
    n[r][c]++;
  }

  void merge(HeatGrid other) {
    if (other.cols != cols || other.rows != rows) return;
    for (var r = 0; r < rows; r++) {
      for (var c = 0; c < cols; c++) {
        sum[r][c] += other.sum[r][c];
        n[r][c] += other.n[r][c];
      }
    }
  }

  double? avg(int r, int c) => n[r][c] > 0 ? sum[r][c] / n[r][c] : null;

  int get totalHits => n.expand((e) => e).fold(0, (a, b) => a + b);

  ({double lo, double hi}) range({double? hintLo, double? hintHi}) {
    var lo = double.infinity, hi = -double.infinity;
    for (var r = 0; r < rows; r++) {
      for (var c = 0; c < cols; c++) {
        final a = avg(r, c);
        if (a != null) { lo = math.min(lo, a); hi = math.max(hi, a); }
      }
    }
    if (lo == double.infinity) return (lo: hintLo ?? 0, hi: hintHi ?? 1);
    return (lo: lo, hi: hi == lo ? lo + 1 : hi);
  }

  String xLabel(int c) => (xmin + (c + 0.5) * (xmax - xmin) / cols).toStringAsFixed(0);
  String yLabel(int r) => (ymin + (rows - r - 0.5) * (ymax - ymin) / rows).toStringAsFixed(2);
}

class AnalyzerService {
  static const List<AxisOpt> axisX = [
    AxisOpt('rpm', 'Обороты, об/мин', 400, 8000, 16),
    AxisOpt('maf', 'MAF, г/с', 0, 200, 16),
    AxisOpt('tps', 'Дроссель, %', 0, 100, 10),
    AxisOpt('speed', 'Скорость, км/ч', 0, 200, 16),
  ];
  static const List<AxisOpt> axisY = [
    AxisOpt('boost', 'Буст, бар', -0.65, 1.5, 14),
    AxisOpt('maf', 'MAF, г/с', 0, 200, 14),
    AxisOpt('tps', 'Дроссель, %', 0, 100, 10),
    AxisOpt('injms', 'Время впрыска, мс', 0, 20, 14),
  ];

  static const Map<String, String> metricNames = {
    'kca': 'УОЗ итоговый (KCA)',
    'fbkc': 'FBKC, °',
    'fkl': 'FKL, °',
    'iam': 'IAM',
    'boost': 'Буст, бар',
    'boostErr': 'Ошибка буста, бар',
    'afr': 'AFR',
    'maf': 'MAF, г/с',
    'wgd': 'Wastegate duty, %',
    'avcs': 'AVCS впуск, °',
    'stft': 'STFT (AF Corr), %',
    'ltft': 'LTFT (AF Learn), %',
    'fuelLph': 'Расход, л/ч',
    'iat': 'Темп. впуска, °C',
    'ect': 'Темп. ОЖ, °C',
    'injms': 'Время впрыска, мс',
    'tps': 'Дроссель, %',
    'speed': 'Скорость, км/ч',
    'ect2': '—',
  };

  AxisOpt xOpt = axisX.first;
  AxisOpt yOpt = axisY.first;
  String metric = 'kca';
  HeatGrid grid = HeatGrid(axisX.first.bins, axisY.first.bins,
      axisX.first.min, axisX.first.max, axisY.first.min, axisY.first.max);

  StreamSubscription<LiveSnapshot>? _sub;
  bool get online => _sub != null;

  void setAxes(AxisOpt x, AxisOpt y, String m) {
    xOpt = x; yOpt = y; metric = m;
    grid = HeatGrid(x.bins, y.bins, x.min, x.max, y.min, y.max);
  }

  double _val(Map<String, double> c, String k) {
    if (k == 'boostErr') {
      final b = c['boost'], t = c['tboost'];
      return (b != null && t != null) ? b - t : double.nan;
    }
    if (k == 'fuelLph') {
      final maf = c['maf'], afr = c['afr'];
      if (maf == null || afr == null || afr <= 0) return double.nan;
      return maf / afr * 3600.0 / 745.0;
    }
    return c[k] ?? double.nan;
  }

  void addRow(Map<String, double> c) {
    final v = _val(c, metric);
    if (v.isNaN) return;
    grid.add(_val(c, xOpt.key), _val(c, yOpt.key), v);
  }

  void startOnline(Stream<LiveSnapshot> stream) {
    stopOnline();
    _sub = stream.listen((s) => addRow(s.c));
  }

  void stopOnline() {
    _sub?.cancel();
    _sub = null;
  }

  /// «Из лога»: экран вызывает LoggerService.parseLog(path) и скармливает
  /// строки через addRow(). Здесь — только построение из готовых строк.
  void buildFromRows(List<Map<String, double>> rows) {
    for (final r in rows) {
      addRow(r);
    }
  }
}
''')
print('OK  analyzer_service.dart')
print()
print('=' * 64)
print('  Сервисы ядра готовы. Далее -> ячейка 6/10 (DTC, PID CRUD, профили, экспорт, связь)')
print('=' * 64)


OK  settings_service.dart
OK  logger_service.dart
OK  alert_service.dart
OK  rom_service.dart
OK  analyzer_service.dart

  Сервисы ядра готовы. Далее -> ячейка 6/10 (DTC, PID CRUD, профили, экспорт, связь)


In [73]:
# @title 🧩 Ячейка 6/10: DTC на русском · PID CRUD · Профили · Экспорт · Связь
# ============================================================================
# Сервисы, которые были в V6 (17 экранов), переписанные под параметры V7/Subaru.
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/services/expr_eval.dart ============
with open('lib/services/expr_eval.dart', 'w') as f:
    f.write(r'''import 'dart:typed_data';

/// Мини-вычислитель выражений (замена math_expressions, как в V6-CRUD):
/// поддержка: числа, x, + - * /, скобки, унарный минус. RPN через shunting-yard.
class ExprEval {
  final List<String> _rpn;
  ExprEval(String expr) : _rpn = _toRpn(expr);

  static List<String> _tok(String e) {
    final out = <String>[];
    var i = 0, prevOp = true;
    while (i < e.length) {
      final ch = e[i];
      if (ch.trim().isEmpty) { i++; continue; }
      if (ch == 'x' || ch == 'X') { out.add('x'); prevOp = false; i++; continue; }
      if (RegExp(r'[0-9.]').hasMatch(ch)) {
        final m = RegExp(r'[0-9]*\.?[0-9]+([eE][+-]?[0-9]+)?').matchAsPrefix(e.substring(i));
        out.add(m!.group(0)!); prevOp = false; i += m.group(0)!.length; continue;
      }
      if ('+-*/'.contains(ch)) {
        if (ch == '-' && prevOp) { out.add('u-'); } else { out.add(ch); }
        prevOp = true; i++; continue;
      }
      if (ch == '(') { out.add('('); prevOp = true; i++; continue; }
      if (ch == ')') { out.add(')'); prevOp = false; i++; continue; }
      throw FormatException('bad char $ch');
    }
    return out;
  }

  static int _prec(String op) => op == 'u-' ? 3 : (op == '*' || op == '/' ? 2 : (op == '+' || op == '-' ? 1 : 0));

  static List<String> _toRpn(String e) {
    final out = <String>[], ops = <String>[];
    for (final t in _tok(e)) {
      if (t == 'x' || RegExp(r'^[0-9]').hasMatch(t)) {
        out.add(t);
      } else if (t == '(') {
        ops.add(t);
      } else if (t == ')') {
        while (ops.isNotEmpty && ops.last != '(') { out.add(ops.removeLast()); }
        if (ops.isEmpty) { throw const FormatException('parens'); }
        ops.removeLast();
      } else {
        while (ops.isNotEmpty && ops.last != '(' && _prec(ops.last) >= _prec(t)) {
          out.add(ops.removeLast());
        }
        ops.add(t);
      }
    }
    while (ops.isNotEmpty) {
      final o = ops.removeLast();
      if (o == '(') throw const FormatException('parens');
      out.add(o);
    }
    return out;
  }

  double call(double x) {
    final st = <double>[];
    for (final t in _rpn) {
      if (t == 'x') { st.add(x); continue; }
      final num = double.tryParse(t);
      if (num != null) { st.add(num); continue; }
      if (t == 'u-') { st.add(-st.removeLast()); continue; }
      final b = st.removeLast();
      final a = st.removeLast();
      switch (t) {
        case '+': st.add(a + b); break;
        case '-': st.add(a - b); break;
        case '*': st.add(a * b); break;
        case '/': st.add(b == 0 ? double.nan : a / b); break;
      }
    }
    return st.isEmpty ? double.nan : st.last;
  }

  static double rawOf(List<int> b, String storage) {
    switch (storage) {
      case 'float':
        return ByteData.sublistView(Uint8List.fromList(
                b.length >= 4 ? b.sublist(0, 4) : <int>[...b, ...List.filled(4 - b.length, 0)]))
            .getFloat32(0, Endian.big);
      case 'int8':
        return (b[0] < 128 ? b[0] : b[0] - 256).toDouble();
      case 'uint16':
        return ((b[0] << 8) | b[1]).toDouble();
      case 'int16':
        var v = (b[0] << 8) | b[1];
        if (v >= 32768) v -= 65536;
        return v.toDouble();
      case 'uint8':
      default:
        return b[0].toDouble();
    }
  }

  static double Function(List<int>) closure(String expr, String storage) {
    final ev = ExprEval(expr);
    return (b) {
      try {
        final v = ev(rawOf(b, storage));
        return v;
      } catch (_) {
        return double.nan;
      }
    };
  }
}
''')
print('OK  expr_eval.dart')

# ============ lib/services/dtc_service.dart ============
with open('lib/services/dtc_service.dart', 'w') as f:
    f.write(r'''import '../ssm/ssm_elm.dart';

class DtcItem {
  final String code;
  final String text;
  DtcItem(this.code, this.text);
}

/// DTC через стандартный OBD-II (режим 03/04) — на CAN-Субару работает
/// на той же шине 0x7E0 параллельно с SSM2. Описания — Subaru-специфичные.
class DtcService {
  static const Map<String, String> db = {
    'P0011': 'AVCS впуск (банк 1): синхронизация опережения',
    'P0021': 'AVCS впуск (банк 2): синхронизация опережения',
    'P0030': 'Подогрев датчика O2 (б1 д1): цепь',
    'P0031': 'Подогрев A/F датчика (б1 д1): низкий уровень',
    'P0032': 'Подогрев A/F датчика (б1 д1): высокий уровень',
    'P0037': 'Подогрев O2 (б1 д2): низкий уровень',
    'P0038': 'Подогрев O2 (б1 д2): высокий уровень',
    'P0101': 'MAF: диапазон/производительность',
    'P0102': 'MAF: низкий сигнал',
    'P0103': 'MAF: высокий сигнал',
    'P0112': 'IAT: низкий сигнал',
    'P0113': 'IAT: высокий сигнал',
    'P0117': 'ECT: низкий сигнал',
    'P0118': 'ECT: высокий сигнал',
    'P0121': 'TPS: диапазон/производительность',
    'P0122': 'TPS: низкий сигнал',
    'P0123': 'TPS: высокий сигнал',
    'P0130': 'O2 датчик (б1 д1): цепь',
    'P0131': 'A/F датчик (б1 д1): низкое напряжение',
    'P0132': 'A/F датчик (б1 д1): высокое напряжение',
    'P0171': 'Система слишком бедная (банк 1)',
    'P0172': 'Система слишком богатая (банк 1)',
    'P0244': 'Соленоид wastegate «A»: диапазон/производительность',
    'P0245': 'Соленоид wastegate «A»: низкий уровень',
    'P0246': 'Соленоид wastegate «A»: высокий уровень',
    'P0301': 'Пропуски зажигания: цилиндр 1',
    'P0302': 'Пропуски зажигания: цилиндр 2',
    'P0303': 'Пропуски зажигания: цилиндр 3',
    'P0304': 'Пропуски зажигания: цилиндр 4',
    'P0327': 'Датчик детонации (б1): низкий сигнал',
    'P0328': 'Датчик детонации (б1): высокий сигнал',
    'P0335': 'Датчик коленвала (CKP): цепь',
    'P0340': 'Датчик распредвала (CMP): цепь',
    'P0420': 'Катализатор: эффективность ниже порога (б1)',
    'P0500': 'Датчик скорости авто (VSS): цепь',
    'P0604': 'ECU: ошибка внутренней памяти (RAM)',
    'P0607': 'ECU: производительность модуля управления',
    'P0851': 'Датчик нейтрали: низкий сигнал',
    'P0852': 'Датчик нейтрали: высокий сигнал',
    'P1443': 'EVAP: клапан вентиляции — цепь',
    'P2004': 'TGV заслонки (банк 1): заклинило открытыми',
    'P2006': 'TGV заслонки (банк 1): заклинило закрытыми',
    'P2008': 'TGV: электроцепь (банк 1)',
    'P2011': 'TGV: цепь управления (банк 2)',
    'P2016': 'TGV датчик положения (б1): низкий',
    'P2021': 'TGV датчик положения (б2): низкий',
    'P2088': 'OCV AVCS (банк 1): низкий уровень',
    'P2090': 'OCV AVCS (банк 1): низкий уровень цепи',
    'P2091': 'OCV AVCS (банк 1): высокий уровень цепи',
    'P2101': 'ETC привод дросселя: диапазон/производительность',
    'P2111': 'ETC дроссель: заклинил открытым',
    'P2119': 'ETC дроссель: диапазон/производительность',
    'P2122': 'APP датчик педали «D»: низкий сигнал',
    'P2123': 'APP датчик педали «D»: высокий сигнал',
    'P2127': 'APP датчик педали «E»: низкий сигнал',
    'P2128': 'APP датчик педали «E»: высокий сигнал',
    'P2135': 'TPS сенсоры «A»/«B»: рассогласование',
    'P2138': 'APP сенсоры «D»/«E»: рассогласование',
    'P2226': 'BARO датчик: цепь',
    'P2227': 'BARO датчик: диапазон/производительность',
    'U0073': 'CAN: модуль отключён от шины (bus off)',
    'U0101': 'Нет связи с TCM',
    'U0122': 'Нет связи с VDC/ABS',
  };

  /// Разбор кадров режима 03: ищем все вхождения '43', читаем пары байт
  static List<DtcItem> parseMode03(String resp) {
    final hex = resp.replaceAll(RegExp(r'[^0-9A-Fa-f]'), '').toUpperCase();
    final out = <DtcItem>[];
    var from = 0;
    while (true) {
      final idx = hex.indexOf('43', from);
      if (idx < 0 || idx + 2 >= hex.length) break;
      var p = idx + 2;
      while (p + 4 <= hex.length) {
        final b1 = int.tryParse(hex.substring(p, p + 2), radix: 16) ?? 0;
        final b2 = int.tryParse(hex.substring(p + 2, p + 4), radix: 16) ?? 0;
        p += 4;
        if (b1 == 0 && b2 == 0) break;
        const sys = ['P', 'C', 'B', 'U'];
        final code = '${sys[(b1 >> 6) & 3]}${(b1 >> 4) & 3}${(b1 & 0xF).toRadixString(16).toUpperCase()}'
            '${b2.toRadixString(16).padLeft(2, '0').toUpperCase()}';
        out.add(DtcItem(code, db[code] ?? 'нет описания в базе V8'));
        if (out.length > 40) return out;
      }
      from = idx + 2;
    }
    return out;
  }

  static Future<List<DtcItem>> read(SsmElm elm) async {
    final r = await elm.transact('03', timeoutMs: 1800);
    return parseMode03(r);
  }

  static Future<bool> clear(SsmElm elm) async {
    final r = await elm.transact('04', timeoutMs: 1800);
    return r.toUpperCase().contains('44');
  }
}
''')
print('OK  dtc_service.dart (40+ Subaru-кодов на русском)')

# ============ lib/services/custom_pid_service.dart ============
with open('lib/services/custom_pid_service.dart', 'w') as f:
    f.write(r'''import 'dart:convert';

import 'package:flutter/foundation.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../generated/subaru_pids.g.dart';
import 'expr_eval.dart';

class CustomPid {
  String id;
  String name;
  String unit;
  String category;
  int address;
  int len;
  String storage; // uint8 | int8 | uint16 | int16 | float
  String expr;    // выражение от x
  int priority;   // 1 fast / 2 mid / 3 slow

  CustomPid({
    required this.id, required this.name, this.unit = '', this.category = 'custom',
    required this.address, this.len = 1, this.storage = 'uint8',
    this.expr = 'x', this.priority = 2,
  });

  Map<String, dynamic> toJson() => {
        'id': id, 'name': name, 'unit': unit, 'category': category,
        'address': address, 'len': len, 'storage': storage, 'expr': expr, 'priority': priority,
      };

  factory CustomPid.fromJson(Map<String, dynamic> j) => CustomPid(
        id: j['id'] as String, name: j['name'] as String,
        unit: (j['unit'] ?? '') as String, category: (j['category'] ?? 'custom') as String,
        address: (j['address'] as num).toInt(), len: (j['len'] ?? 1).toInt(),
        storage: (j['storage'] ?? 'uint8') as String,
        expr: (j['expr'] ?? 'x') as String, priority: (j['priority'] ?? 2).toInt(),
      );

  SubaruPid toSubaruPid() => SubaruPid(
        id: 'C_$id', xmlId: 'custom', name: name, desc: 'custom PID (user)',
        unit: unit, category: category.isEmpty ? 'custom' : category,
        address: address, len: len, storage: storage, priority: priority, canon: '',
        formula: ExprEval.closure(expr, storage),
      );
}

/// CRUD кастомных PID + живой тест значения (как PID-редактор в V6)
class CustomPidService extends ChangeNotifier {
  static final CustomPidService I = CustomPidService._();
  CustomPidService._();

  final List<CustomPid> items = [];
  bool _loaded = false;

  Future<void> load() async {
    if (_loaded) return;
    _loaded = true;
    final p = await SharedPreferences.getInstance();
    final raw = p.getString('customPids');
    if (raw != null && raw.isNotEmpty) {
      try {
        final list = (jsonDecode(raw) as List).cast<Map<String, dynamic>>();
        items.addAll(list.map(CustomPid.fromJson));
      } catch (_) {}
    }
  }

  Future<void> _save() async {
    final p = await SharedPreferences.getInstance();
    await p.setString('customPids', jsonEncode(items.map((e) => e.toJson()).toList()));
    notifyListeners();
  }

  Future<void> upsert(CustomPid pid) async {
    final i = items.indexWhere((e) => e.id == pid.id);
    if (i >= 0) { items[i] = pid; } else { items.add(pid); }
    await _save();
  }

  Future<void> remove(String id) async {
    items.removeWhere((e) => e.id == id);
    await _save();
  }

  List<SubaruPid> asPids() => items.map((e) => e.toSubaruPid()).toList();
}
''')
print('OK  custom_pid_service.dart')

# ============ lib/services/profile_service.dart ============
with open('lib/services/profile_service.dart', 'w') as f:
    f.write(r'''import 'dart:convert';

import 'package:flutter/foundation.dart';
import 'package:shared_preferences/shared_preferences.dart';

import '../generated/subaru_pids.g.dart';
import 'settings_service.dart';

/// Профили = ВСЕ настройки опроса и алертов (как в V6): набор PID,
/// параметры блоков/ярусов, калибровки. Сиды: сток A2TB100B.
class ProfileService extends ChangeNotifier {
  static final ProfileService I = ProfileService._();
  ProfileService._();

  final Map<String, Map<String, dynamic>> profiles = {};
  bool _loaded = false;

  Future<void> load() async {
    if (_loaded) return;
    _loaded = true;
    final p = await SharedPreferences.getInstance();
    final raw = p.getString('profiles');
    if (raw != null && raw.isNotEmpty) {
      try {
        final m = (jsonDecode(raw) as Map).cast<String, dynamic>();
        m.forEach((k, v) => profiles[k] = (v as Map).cast<String, dynamic>());
      } catch (_) {}
    }
    profiles.putIfAbsent('A2TB100B · сток (канон)', () => _captureDefaults());
  }

  Map<String, dynamic> _captureDefaults() => {
        'enabledIds': SubaruPids.defaults.map((e) => e.id).toList(),
        'maxBlock': 0x50, 'gapTol': 2, 'midEveryN': 4, 'slowEveryN': 25, 'stCode': 8,
        'autoLog': true,
      };

  Map<String, dynamic> _capture(SettingsService st) => {
        'enabledIds': st.enabledIds.toList(),
        'maxBlock': st.maxBlock, 'gapTol': st.gapTol,
        'midEveryN': st.midEveryN, 'slowEveryN': st.slowEveryN,
        'stCode': st.stCode, 'autoLog': st.autoLog,
      };

  Future<void> saveCurrent(String name) async {
    profiles[name] = _capture(SettingsService.I);
    await _persist();
  }

  Future<void> apply(String name) async {
    final d = profiles[name];
    if (d == null) return;
    final st = SettingsService.I;
    st.enabledIds = ((d['enabledIds'] as List?) ?? []).map((e) => e.toString()).toSet();
    st.maxBlock = (d['maxBlock'] ?? st.maxBlock) as int;
    st.gapTol = (d['gapTol'] ?? st.gapTol) as int;
    st.midEveryN = (d['midEveryN'] ?? st.midEveryN) as int;
    st.slowEveryN = (d['slowEveryN'] ?? st.slowEveryN) as int;
    st.stCode = (d['stCode'] ?? st.stCode) as int;
    st.autoLog = (d['autoLog'] ?? st.autoLog) as bool;
    await st.save();
    notifyListeners();
  }

  Future<void> remove(String name) async {
    profiles.remove(name);
    await _persist();
  }

  Future<void> _persist() async {
    final p = await SharedPreferences.getInstance();
    await p.setString('profiles', jsonEncode(profiles));
    notifyListeners();
  }
}
''')
print('OK  profile_service.dart')

# ============ lib/services/export_service.dart ============
with open('lib/services/export_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';

import 'package:path_provider/path_provider.dart';

import 'rom_service.dart';

/// Экспорт карт и ROM (WinOLS/ecuEdit/HEX) — как экран Экспорт в V6.
class ExportService {
  /// Все карты одним CSV в стиле ecuEdit (разделитель ; для RU-Excel)
  static Future<File> exportAllTablesCsv(RomService rom) async {
    final buf = StringBuffer();
    buf.writeln('\uFEFF'); // BOM для Excel
    for (final def in rom.defs) {
      try {
        final t = rom.readTable(def);
        buf.writeln('"${def.name}";"${def.category}";${def.addrHex};${def.rows}x${def.cols};"${def.units}"');
        buf.writeln('"X\\Y";${t.xValues.map((e) => e.toStringAsFixed(1)).join(';')}');
        for (var r = 0; r < def.rows; r++) {
          buf.writeln('"${t.yValues.isNotEmpty ? t.yValues[r].toStringAsFixed(1) : r}";'
              '${t.z[r].map((e) => e.toStringAsFixed(3)).join(';')}');
        }
        buf.writeln();
      } catch (_) {
        buf.writeln('"${def.name}";"READ ERROR"');
        buf.writeln();
      }
    }
    final dir = await getApplicationDocumentsDirectory();
    final f = File('${dir.path}/V8_maps_ecuEdit.csv');
    await f.writeAsString(buf.toString());
    return f;
  }

  /// WinOLS-подобный дамп отдельной карты: адрес + сырые значения сетки
  static Future<File> exportTableWinols(RomService rom, String tableName) async {
    final def = rom.defs.firstWhere((d) => d.name == tableName,
        orElse: () => throw ArgumentError('no table'));
    final t = rom.readTable(def);
    final buf = StringBuffer();
    buf.writeln(';MAP ${def.name}');
    buf.writeln(';ADDR ${def.addrHex} SIZE ${def.rows * def.cols * def.data.sizeOf}');
    buf.writeln(';AXES X=[${t.xValues.join(', ')}] Y=[${t.yValues.join(', ')}]');
    for (final row in t.toCsvRows()) {
      buf.writeln(row.join(';'));
    }
    final dir = await getApplicationDocumentsDirectory();
    final safe = tableName.replaceAll(RegExp(r'[^0-9A-Za-zА-Яа-я]+'), '_');
    final f = File('${dir.path}/V8_winols_$safe.csv');
    await f.writeAsString(buf.toString());
    return f;
  }

  /// HEX-дамп ROM (первые N байт постранично, 16 байт/строка)
  static Future<File> hexDump(List<int> rom, {int maxBytes = 0x10000}) async {
    final n = rom.length < maxBytes ? rom.length : maxBytes;
    final buf = StringBuffer();
    for (var off = 0; off < n; off += 16) {
      final chunk = rom.sublist(off, off + 16 > n ? n : off + 16);
      final hexs = chunk.map((b) => b.toRadixString(16).padLeft(2, '0')).join(' ');
      final ascii = chunk.map((b) => (b >= 32 && b < 127) ? String.fromCharCode(b) : '.').join();
      buf.writeln('${off.toRadixString(16).padLeft(8, '0')}  ${hexs.padRight(47)}  $ascii');
    }
    final dir = await getApplicationDocumentsDirectory();
    final f = File('${dir.path}/V8_rom_hexdump.txt');
    await f.writeAsString(buf.toString());
    return f;
  }
}
''')
print('OK  export_service.dart')

# ============ lib/services/connection_service.dart (обновлён: эксклюзив + кастомные PID) ============
with open('lib/services/connection_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/foundation.dart';

import '../ssm/ssm_elm.dart';
import 'alert_service.dart';
import 'analyzer_service.dart';
import 'custom_pid_service.dart';
import 'logger_service.dart';
import 'rom_service.dart';
import 'settings_service.dart';

/// Центральный узел: ELM + поллер + алерты + логгер + ROM.
/// exclusive() — монопольный доступ к шине для DTC/терминала/чтения карт:
/// опрос приостанавливается, операция выполняется, опрос продолжается.
class ConnectionService extends ChangeNotifier {
  static final ConnectionService I = ConnectionService._();
  ConnectionService._();

  final SsmElm elm = SsmElm();
  final AlertService alerts = AlertService();
  final LoggerService logger = LoggerService();
  final AnalyzerService analyzer = AnalyzerService();
  final RomService rom = RomService();

  SsmPoller? poller;
  StreamSubscription? _alertSub;
  bool connecting = false;

  Future<String> connectAndInit() async {
    if (connecting) return 'уже подключаюсь...';
    final st = SettingsService.I;
    if (st.btAddress.isEmpty) return 'Сначала выбери ELM327 на экране настроек';
    connecting = true;
    notifyListeners();
    elm.stTimeoutCode = st.stCode;
    final okBt = await elm.connect(st.btAddress);
    if (!okBt) {
      connecting = false;
      notifyListeners();
      return 'Bluetooth: не удалось подключиться к ${st.btName}';
    }
    final ecuId = await elm.ecuInit();
    connecting = false;
    if (ecuId == null) {
      await elm.disconnect();
      notifyListeners();
      return 'ECU не ответил на SSM2 init (8010F001BF40)';
    }
    buildPoller();
    startPolling();
    notifyListeners();
    return 'OK: ECU ${elm.ecuId.isEmpty ? ecuId : elm.ecuId}';
  }

  /// Перестроение блочного плана: библиотека + кастомные PID
  void buildPoller() {
    final st = SettingsService.I;
    poller?.stop();
    final p = SsmPoller(elm)
      ..maxBlock = st.maxBlock
      ..gapTol = st.gapTol
      ..midEveryN = st.midEveryN
      ..slowEveryN = st.slowEveryN;
    p.buildBlocks([...st.selectedPids, ...CustomPidService.I.asPids()]);
    poller = p;
    _alertSub?.cancel();
    _alertSub = p.snapshots.listen(alerts.check);
  }

  bool get isPolling => poller?.isRunning ?? false;

  void startPolling() {
    if (poller != null && (elm.state == SsmState.ecuReady || elm.state == SsmState.polling)) {
      poller!.start();
    }
    notifyListeners();
  }

  void stopPolling() {
    poller?.stop();
    notifyListeners();
  }

  /// Монопольная операция на шине (DTC, терминал, чтение карты из ECU).
  Future<T> exclusive<T>(Future<T> Function(SsmElm elm) job) async {
    final was = isPolling;
    if (was) stopPolling();
    await Future<void>.delayed(const Duration(milliseconds: 180)); // дожать in-flight кадр
    try {
      return await job(elm);
    } finally {
      if (was) startPolling();
    }
  }

  Future<void> disconnect() async {
    stopPolling();
    if (logger.logging) await logger.stop();
    await elm.disconnect();
    notifyListeners();
  }
}
''')
print('OK  connection_service.dart (exclusive + custom PID merge)')
print()
print('=' * 64)
print('  Доп. сервисы готовы. Далее -> ячейка 7/10 (главный + приборы + графики + лог)')
print('=' * 64)


OK  expr_eval.dart
OK  dtc_service.dart (40+ Subaru-кодов на русском)
OK  custom_pid_service.dart
OK  profile_service.dart
OK  export_service.dart
OK  connection_service.dart (exclusive + custom PID merge)

  Доп. сервисы готовы. Далее -> ячейка 7/10 (главный + приборы + графики + лог)


In [78]:
# @title 🖥️ Ячейка 7/10: main (17 вкладок) + Приборы + Графики + ЛогГраф + Лог + События
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/widgets/heat_colors.dart ============
with open('lib/widgets/heat_colors.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

/// Цветная шкала "как в V6": синий -> циан -> зелёный -> жёлтый -> красный
const _stops = <Color>[
  Color(0xFF0D47A1),
  Color(0xFF0288D1),
  Color(0xFF00BCD4),
  Color(0xFF4CAF50),
  Color(0xFFFFEB3B),
  Color(0xFFF44336),
];

Color heatColor(double t) {
  final tt = t.isNaN ? 0.0 : t.clamp(0.0, 1.0).toDouble();
  final pos = tt * (_stops.length - 1);
  final i = pos.floor();
  final frac = pos - i;
  if (i >= _stops.length - 1) return _stops.last;
  return Color.lerp(_stops[i], _stops[i + 1], frac)!;
}

class HeatColors {
  static const bg = Color(0xFF070D1F);
  static const panel = Color(0xFF0D1630);
  static const grid = Color(0xFF1B2A52);
  static const accent = Color(0xFF3D7BFF);
  static const gold = Color(0xFFE7B93C);
  static const text = Color(0xFFDCE6FF);
  static const dim = Color(0xFF7C8BB5);
}
''')
print('OK  widgets/heat_colors.dart')

# ============ lib/widgets/heat_map.dart ============
with open('lib/widgets/heat_map.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/analyzer_service.dart';
import 'heat_colors.dart';

/// Цветная карта (тепловая таблица) с осями, легендой и тапом по ячейке.
/// Единый виджет для Анализатора, ROM/ЭБУ-карт и ROM Diff — как анализатор V6.
class HeatMapView extends StatefulWidget {
  final int cols;
  final int rows;
  final double? Function(int r, int c) value; // null = нет данных
  final int Function(int r, int c)? hits;
  final String Function(int c)? xLabel;
  final String Function(int r)? yLabel;
  final String title;
  final String unit;
  final double? hintLo;
  final double? hintHi;

  const HeatMapView({
    super.key,
    required this.cols,
    required this.rows,
    required this.value,
    this.hits,
    this.xLabel,
    this.yLabel,
    this.title = '',
    this.unit = '',
    this.hintLo,
    this.hintHi,
  });

  factory HeatMapView.fromGrid(HeatGrid g,
      {String title = '', String unit = '', double? hintLo, double? hintHi}) {
    final range = g.range(hintLo: hintLo, hintHi: hintHi);
    final lo = range.lo, hi = range.hi;
    return HeatMapView(
      cols: g.cols,
      rows: g.rows,
      title: title,
      unit: unit,
      hintLo: lo,
      hintHi: hi,
      value: (r, c) => g.avg(g.rows - 1 - r, c),
      hits: (r, c) => g.n[g.rows - 1 - r][c],
      xLabel: (c) => g.xLabel(c),
      yLabel: (r) => g.yLabel(g.rows - 1 - r),
    );
  }

  @override
  State<HeatMapView> createState() => _HeatMapViewState();
}

class _HeatMapViewState extends State<HeatMapView> {
  ({int r, int c})? _sel;

  double get _lo {
    var lo = double.infinity;
    for (var r = 0; r < widget.rows; r++) {
      for (var c = 0; c < widget.cols; c++) {
        final v = widget.value(r, c);
        if (v != null && v < lo) lo = v;
      }
    }
    if (lo == double.infinity) return widget.hintLo ?? 0;
    return lo;
  }

  double get _hi {
    var hi = -double.infinity;
    for (var r = 0; r < widget.rows; r++) {
      for (var c = 0; c < widget.cols; c++) {
        final v = widget.value(r, c);
        if (v != null && v > hi) hi = v;
      }
    }
    if (hi == -double.infinity) return widget.hintHi ?? 1;
    return hi;
  }

  String _fmt(double v) {
    final a = v.abs();
    if (a >= 1000) return v.toStringAsFixed(0);
    if (a >= 100) return v.toStringAsFixed(1);
    return v.toStringAsFixed(2);
  }

  @override
  Widget build(BuildContext context) {
    final lo = widget.hintLo ?? _lo;
    var hi = widget.hintHi ?? _hi;
    if (hi <= lo) hi = lo + 1;
    return LayoutBuilder(builder: (ctx, cons) {
      const axW = 46.0, axH = 26.0;
      final cw = (cons.maxWidth - axW) / widget.cols;
      final ch = ((cons.maxHeight - axH) / widget.rows).clamp(14.0, 60.0);
      return Column(
        crossAxisAlignment: CrossAxisAlignment.stretch,
        children: [
          Expanded(
            child: Row(
              children: [
                SizedBox(
                  width: axW,
                  child: Column(
                    children: List.generate(widget.rows, (r) {
                      return SizedBox(
                        height: ch,
                        child: Align(
                          alignment: Alignment.centerRight,
                          child: Padding(
                            padding: const EdgeInsets.only(right: 4),
                            child: Text(widget.yLabel?.call(r) ?? '$r',
                                style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
                          ),
                        ),
                      );
                    }),
                  ),
                ),
                Expanded(
                  child: GestureDetector(
                    onTapDown: (d) {
                      final c = (d.localPosition.dx / cw).floor().clamp(0, widget.cols - 1);
                      final r = (d.localPosition.dy / ch).floor().clamp(0, widget.rows - 1);
                      setState(() => _sel = (r: r, c: c));
                    },
                    child: Column(
                      children: List.generate(widget.rows, (r) {
                        return SizedBox(
                          height: ch,
                          child: Row(
                            children: List.generate(widget.cols, (c) {
                              final v = widget.value(r, c);
                              final t = v == null ? null : (v - lo) / (hi - lo);
                              final sel = _sel != null && _sel!.r == r && _sel!.c == c;
                              return Container(
                                width: cw,
                                height: ch,
                                decoration: BoxDecoration(
                                  color: v == null ? HeatColors.bg : heatColor(t!),
                                  border: Border.all(
                                    color: sel ? Colors.white : HeatColors.bg,
                                    width: sel ? 2 : 1,
                                  ),
                                ),
                                alignment: Alignment.center,
                                child: cw > 26 && v != null
                                    ? Text(
                                        _fmt(v),
                                        style: TextStyle(
                                          fontSize: 8,
                                          color: t! > 0.68 || t < 0.25
                                              ? Colors.white
                                              : Colors.black87,
                                          fontWeight: FontWeight.w600,
                                        ),
                                      )
                                    : null,
                              );
                            }),
                          ),
                        );
                      }),
                    ),
                  ),
                ),
              ],
            ),
          ),
          SizedBox(
            height: axH,
            child: Row(
              children: [
                const SizedBox(width: 46),
                ...List.generate(widget.cols, (c) {
                  return Expanded(
                    child: Text(widget.xLabel?.call(c) ?? '',
                        textAlign: TextAlign.center,
                        style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
                  );
                }),
              ],
            ),
          ),
          if (_sel != null && widget.value(_sel!.r, _sel!.c) != null)
            Padding(
              padding: const EdgeInsets.only(top: 4),
              child: Text(
                'ячейка [${widget.yLabel?.call(_sel!.r) ?? ''} × ${widget.xLabel?.call(_sel!.c) ?? ''}] = '
                '${_fmt(widget.value(_sel!.r, _sel!.c)!)} ${widget.unit}'
                '${widget.hits != null ? ' · попаданий: ${widget.hits!(_sel!.r, _sel!.c)}' : ''}',
                style: const TextStyle(fontSize: 11, color: HeatColors.gold),
              ),
            ),
        ],
      );
    });
  }
}
''')
print('OK  widgets/heat_map.dart (цветные карты как в V6)')

# ============ lib/main.dart ============
with open('lib/main.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:permission_handler/permission_handler.dart';

import 'screens/analyzer_screen.dart';
import 'screens/custom_pid_screen.dart';
import 'screens/dashboard_screen.dart';
import 'screens/dtc_screen.dart';
import 'screens/ecu_maps_screen.dart';
import 'screens/events_screen.dart';
import 'screens/export_screen.dart';
import 'screens/graphs_screen.dart';
import 'screens/log_graph_screen.dart';
import 'screens/logging_screen.dart';
import 'screens/perf_screen.dart';
import 'screens/profile_screen.dart';
import 'screens/rom_diff_screen.dart';
import 'screens/rom_screen.dart';
import 'screens/service_screen.dart';
import 'screens/settings_screen.dart';
import 'screens/terminal_screen.dart';
import 'services/custom_pid_service.dart';
import 'services/profile_service.dart';
import 'services/settings_service.dart';
import 'widgets/heat_colors.dart';

void main() {
  WidgetsFlutterBinding.ensureInitialized();
  runApp(const SubaApp());
}

class SubaApp extends StatelessWidget {
  const SubaApp({super.key});

  @override
  Widget build(BuildContext context) {
    final base = ThemeData.dark(useMaterial3: true);
    return MaterialApp(
      title: 'SUBA RUN V8',
      debugShowCheckedModeBanner: false,
      theme: base.copyWith(
        scaffoldBackgroundColor: HeatColors.bg,
        colorScheme: const ColorScheme.dark(
          primary: HeatColors.accent,
          secondary: HeatColors.gold,
          surface: HeatColors.panel,
        ),
        appBarTheme: const AppBarTheme(
            backgroundColor: HeatColors.bg, foregroundColor: HeatColors.text, elevation: 0),
        tabBarTheme: const TabBarThemeData(
          labelColor: HeatColors.gold,
          unselectedLabelColor: HeatColors.dim,
          indicatorColor: HeatColors.gold,
          labelStyle: TextStyle(fontSize: 10, fontWeight: FontWeight.w700),
          unselectedLabelStyle: TextStyle(fontSize: 10),
        ),
      ),
      home: const HomeShell(),
    );
  }
}

class HomeShell extends StatefulWidget {
  const HomeShell({super.key});

  @override
  State<HomeShell> createState() => _HomeShellState();
}

class _HomeShellState extends State<HomeShell> {
  @override
  void initState() {
    super.initState();
    _boot();
  }

  Future<void> _boot() async {
    await SettingsService.I.load();
    await CustomPidService.I.load();
    await ProfileService.I.load();
    await Permission.bluetoothConnect.request();
    await Permission.bluetoothScan.request();
    await Permission.locationWhenInUse.request();
  }

  static const _tabs = [
    Tab(text: 'ПРИБОРЫ'),
    Tab(text: 'ГРАФИКИ'),
    Tab(text: 'ЛОГГРАФ'),
    Tab(text: 'ЛОГ'),
    Tab(text: 'СОБЫТИЯ'),
    Tab(text: 'DTC'),
    Tab(text: 'АНАЛИЗАТОР'),
    Tab(text: 'ЭБУ КАРТЫ'),
    Tab(text: 'ROM'),
    Tab(text: 'ROM DIFF'),
    Tab(text: 'СЕРВИС'),
    Tab(text: 'ЗАМЕР'),
    Tab(text: 'ЭКСПОРТ'),
    Tab(text: 'PID'),
    Tab(text: 'ПРОФИЛИ'),
    Tab(text: 'ТЕРМИНАЛ'),
    Tab(text: 'НАСТРОЙКИ'),
  ];

  static const _views = [
    DashboardScreen(),
    GraphsScreen(),
    LogGraphScreen(),
    LoggingScreen(),
    EventsScreen(),
    DtcScreen(),
    AnalyzerScreen(),
    EcuMapsScreen(),
    RomScreen(),
    RomDiffScreen(),
    ServiceScreen(),
    PerfScreen(),
    ExportScreen(),
    CustomPidScreen(),
    ProfileScreen(),
    TerminalScreen(),
    SettingsScreen(),
  ];

  @override
  Widget build(BuildContext context) {
    return DefaultTabController(
      length: _tabs.length,
      child: Scaffold(
        appBar: AppBar(
          titleSpacing: 12,
          title: const Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              Text('SUBA RUN V8',
                  style: TextStyle(fontSize: 15, fontWeight: FontWeight.w800, letterSpacing: 1.2)),
              Text('EJ20X · A2TB100B · SSM2/CAN · 17 экранов',
                  style: TextStyle(fontSize: 9.5, color: HeatColors.dim)),
            ],
          ),
          bottom: const TabBar(isScrollable: true, tabAlignment: TabAlignment.start, tabs: _tabs),
        ),
        body: const TabBarView(children: _views),
      ),
    );
  }
}
''')
print('OK  main.dart (17 вкладок)')

# ============ lib/screens/dashboard_screen.dart ============
with open('lib/screens/dashboard_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/material.dart';

import '../constants.dart';
import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

class DashboardScreen extends StatefulWidget {
  const DashboardScreen({super.key});

  @override
  State<DashboardScreen> createState() => _DashboardScreenState();
}

class _DashboardScreenState extends State<DashboardScreen> {
  Timer? _ticker;

  @override
  void initState() {
    super.initState();
    _ticker = Timer.periodic(const Duration(milliseconds: 500), (_) {
      if (mounted) setState(() {});
    });
  }

  @override
  void dispose() {
    _ticker?.cancel();
    super.dispose();
  }

  Color _stateColor(SsmState s) {
    switch (s) {
      case SsmState.polling:
        return Colors.greenAccent;
      case SsmState.ecuReady:
      case SsmState.elmReady:
        return HeatColors.gold;
      case SsmState.connecting:
        return HeatColors.accent;
      case SsmState.error:
        return Colors.redAccent;
      case SsmState.disconnected:
        return HeatColors.dim;
    }
  }

  static String _mode(String m) {
    switch (m) {
      case 'IDLE':
        return 'ХОЛОСТОЙ';
      case 'BOOST':
        return 'БУСТ';
      case 'WOT':
        return 'ПОЛНЫЙ ГАЗ';
      case 'COAST':
        return 'НАКАТ';
      default:
        return 'КРУИЗ';
    }
  }

  @override
  Widget build(BuildContext context) {
    final svc = ConnectionService.I;
    final tiles = SettingsService.I.selectedPids;
    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 8),
          child: Row(
            children: [
              Icon(Icons.circle, size: 10, color: _stateColor(svc.elm.state)),
              const SizedBox(width: 8),
              Expanded(
                child: Text(
                  svc.elm.state == SsmState.polling
                      ? 'ОПРОС · ${svc.elm.stats.hz.toStringAsFixed(1)} Гц · '
                          '${svc.elm.stats.avgMs.toStringAsFixed(0)} мс/кадр · '
                          '${svc.poller?.blockCount ?? 0} блоков'
                      : svc.elm.state.name.toUpperCase(),
                  style: const TextStyle(fontSize: 12, color: HeatColors.text),
                  overflow: TextOverflow.ellipsis,
                ),
              ),
              if (svc.poller?.last != null)
                Text(_mode((svc.poller!.last!).mode),
                    style: const TextStyle(fontSize: 11, color: HeatColors.gold)),
            ],
          ),
        ),
        StreamBuilder<LiveSnapshot>(
          stream: svc.poller?.snapshots,
          builder: (ctx, snap) {
            final s = snap.data ?? svc.poller?.last;
            final boost = s?.boost ?? 0;
            final target = s?.has('tboost') == true ? s!.tboost : null;
            final t = ((boost + 0.65) / 2.15).clamp(0.0, 1.0);
            return Container(
              margin: const EdgeInsets.all(10),
              padding: const EdgeInsets.all(12),
              decoration: BoxDecoration(
                  color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
              child: Column(
                crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(
                    children: [
                      const Text('BOOST', style: TextStyle(fontSize: 11, color: HeatColors.dim)),
                      const Spacer(),
                      Text('${boost.toStringAsFixed(2)} бар',
                          style: TextStyle(
                              fontSize: 22,
                              fontWeight: FontWeight.w800,
                              color: boost > AppConstants.overboostDanger
                                  ? Colors.redAccent
                                  : HeatColors.text)),
                      if (target != null)
                        Text('  / цель ${target.toStringAsFixed(2)}',
                            style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
                    ],
                  ),
                  const SizedBox(height: 6),
                  ClipRRect(
                    borderRadius: BorderRadius.circular(6),
                    child: LinearProgressIndicator(
                      value: t,
                      minHeight: 10,
                      backgroundColor: HeatColors.bg,
                      valueColor: AlwaysStoppedAnimation(
                          boost < 0 ? HeatColors.accent : heatColor(t)),
                    ),
                  ),
                ],
              ),
            );
          },
        ),
        Expanded(
          child: StreamBuilder<LiveSnapshot>(
            stream: svc.poller?.snapshots,
            builder: (ctx, snap) {
              final s = snap.data ?? svc.poller?.last;
              return GridView.builder(
                padding: const EdgeInsets.fromLTRB(10, 0, 10, 10),
                gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
                    crossAxisCount: 3,
                    childAspectRatio: 1.25,
                    mainAxisSpacing: 8,
                    crossAxisSpacing: 8),
                itemCount: tiles.length,
                itemBuilder: (ctx, i) => _Tile(pid: tiles[i], s: s),
              );
            },
          ),
        ),
      ],
    );
  }
}

class _Tile extends StatelessWidget {
  final dynamic pid;
  final LiveSnapshot? s;
  const _Tile({required this.pid, required this.s});

  Color? _alertColor() {
    if (s == null) return null;
    switch (pid.canon) {
      case 'fbkc':
        if (s!.fbkc <= AppConstants.fbkcDanger) return Colors.redAccent;
        if (s!.fbkc <= AppConstants.fbkcWarn) return Colors.orangeAccent;
        break;
      case 'fkl':
        if (s!.fkl <= AppConstants.fklDanger) return Colors.redAccent;
        if (s!.fkl <= AppConstants.fklWarn) return Colors.orangeAccent;
        break;
      case 'iam':
        if (s!.iam < AppConstants.iamDanger) return Colors.redAccent;
        if (s!.iam < AppConstants.iamWarn) return Colors.orangeAccent;
        break;
      case 'ect':
        if (s!.ect >= AppConstants.ectDanger) return Colors.redAccent;
        if (s!.ect >= AppConstants.ectWarn) return Colors.orangeAccent;
        break;
      case 'boost':
        if (s!.boost >= AppConstants.overboostDanger) return Colors.redAccent;
        break;
    }
    return null;
  }

  @override
  Widget build(BuildContext context) {
    final v = s?.byId[pid.id];
    final alert = _alertColor();
    final frac = v == null
        ? ''
        : (v.abs() >= 100
            ? v.toStringAsFixed(0)
            : v.abs() >= 10
                ? v.toStringAsFixed(1)
                : v.toStringAsFixed(2));
    return Container(
      padding: const EdgeInsets.all(8),
      decoration: BoxDecoration(
        color: HeatColors.panel,
        borderRadius: BorderRadius.circular(10),
        border: Border.all(color: alert ?? Colors.transparent, width: 1.5),
      ),
      child: Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Text(pid.name,
              maxLines: 1,
              overflow: TextOverflow.ellipsis,
              style: TextStyle(fontSize: 10, color: alert ?? HeatColors.dim)),
          const Spacer(),
          FittedBox(
            fit: BoxFit.scaleDown,
            alignment: Alignment.centerLeft,
            child: Text(v == null ? '--' : frac,
                style: TextStyle(
                    fontSize: 26,
                    fontWeight: FontWeight.w800,
                    color: alert ?? HeatColors.text)),
          ),
          Text(pid.unit, style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
        ],
      ),
    );
  }
}
''')
print('OK  dashboard_screen.dart')

# ============ lib/screens/graphs_screen.dart ============
with open('lib/screens/graphs_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:math' as math;

import 'package:fl_chart/fl_chart.dart';
import 'package:flutter/material.dart';

import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';

/// Живые графики реального времени (как вкладка Графики в V6):
/// 1-4 параметра одновременно, скользящее окно ~240 снапшотов.
class GraphsScreen extends StatefulWidget {
  const GraphsScreen({super.key});

  @override
  State<GraphsScreen> createState() => _GraphsScreenState();
}

class _GraphsScreenState extends State<GraphsScreen> {
  static const _colors = [Color(0xFF4ADE80), Color(0xFF60A5FA), Color(0xFFE7B93C), Color(0xFFF472B6)];
  static const _params = <String, String>{
    'rpm': 'RPM',
    'boost': 'Буст',
    'kca': 'KCA °',
    'fbkc': 'FBKC °',
    'iam': 'IAM',
    'afr': 'AFR',
    'tps': 'TPS %',
    'maf': 'MAF г/с',
    'iat': 'IAT',
    'ect': 'ECT',
    'wgd': 'WG duty',
    'speed': 'Скорость',
  };

  final List<LiveSnapshot> _buf = [];
  final Set<String> _sel = {'rpm', 'boost', 'kca', 'fbkc'};
  StreamSubscription? _sub;
  Timer? _ticker;

  @override
  void initState() {
    super.initState();
    final p = ConnectionService.I.poller;
    if (p != null) {
      _sub = p.snapshots.listen(_add);
    }
    _ticker = Timer.periodic(const Duration(milliseconds: 700), (_) {
      if (mounted) setState(() {});
    });
  }

  void _add(LiveSnapshot s) {
    _buf.add(s);
    if (_buf.length > 240) _buf.removeRange(0, _buf.length - 240);
  }

  @override
  void dispose() {
    _sub?.cancel();
    _ticker?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final svc = ConnectionService.I;
    if (svc.poller == null) {
      return const Center(
          child: Text('Подключись в настройках — графики живые из опроса',
              style: TextStyle(color: HeatColors.dim)));
    }
    final sel = _sel.where((k) => _params.containsKey(k)).toList();
    return Column(
      children: [
        SizedBox(
          height: 44,
          child: ListView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 6),
            children: _params.entries.map((e) {
              final on = _sel.contains(e.key);
              return Padding(
                padding: const EdgeInsets.only(right: 6),
                child: FilterChip(
                  label: Text(e.value, style: const TextStyle(fontSize: 11)),
                  selected: on,
                  onSelected: (v) {
                    setState(() {
                      if (v) {
                        if (_sel.length < 4) _sel.add(e.key);
                      } else {
                        _sel.remove(e.key);
                      }
                    });
                  },
                ),
              );
            }).toList(),
          ),
        ),
        Expanded(
          child: Padding(
            padding: const EdgeInsets.fromLTRB(4, 4, 12, 12),
            child: _buf.length < 4
                ? const Center(child: Text('жду данные опроса...', style: TextStyle(color: HeatColors.dim)))
                : LineChart(
                    LineChartData(
                      clipData: const FlClipData.all(),
                      gridData: FlGridData(
                        show: true,
                        drawVerticalLine: false,
                        getDrawingHorizontalLine: (_) =>
                            const FlLine(color: HeatColors.grid, strokeWidth: 0.5),
                      ),
                      borderData: FlBorderData(show: false),
                      titlesData: const FlTitlesData(
                        topTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
                        bottomTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
                        rightTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
                        leftTitles: AxisTitles(
                            sideTitles: SideTitles(showTitles: true, reservedSize: 40)),
                      ),
                      lineBarsData: List.generate(sel.length, (i) {
                        final key = sel[i];
                        final spots = <FlSpot>[];
                        for (var j = 0; j < _buf.length; j++) {
                          final v = _buf[j].c[key];
                          if (v != null) spots.add(FlSpot(j.toDouble(), v));
                        }
                        return LineChartBarData(
                          spots: spots,
                          color: _colors[i % _colors.length],
                          barWidth: 1.6,
                          isCurved: false,
                          dotData: const FlDotData(show: false),
                        );
                      }),
                    ),
                    duration: Duration.zero,
                  ),
          ),
        ),
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 8),
          child: Row(
            children: [
              for (var i = 0; i < sel.length; i++)
                Expanded(
                  child: Column(
                    children: [
                      Text('${_params[sel[i]]}',
                          style: TextStyle(fontSize: 9, color: _colors[i % _colors.length])),
                      Text(
                        _buf.isEmpty || _buf.last.c[sel[i]] == null
                            ? '--'
                            : _fmt(_buf.last.c[sel[i]]!),
                        style: TextStyle(
                            fontSize: 17, fontWeight: FontWeight.w800, color: _colors[i % _colors.length]),
                      ),
                      if (_buf.isNotEmpty) _minMax(sel[i], _colors[i % _colors.length]),
                    ],
                  ),
                ),
            ],
          ),
        ),
      ],
    );
  }

  Widget _minMax(String key, Color color) {
    var lo = double.infinity, hi = -double.infinity;
    for (final s in _buf) {
      final v = s.c[key];
      if (v != null) { lo = math.min(lo, v); hi = math.max(hi, v); }
    }
    if (lo == double.infinity) return const SizedBox.shrink();
    return Text('${_fmt(lo)} … ${_fmt(hi)}',
        style: const TextStyle(fontSize: 8.5, color: HeatColors.dim));
  }

  String _fmt(double v) =>
      v.abs() >= 1000 ? v.toStringAsFixed(0) : v.abs() >= 100 ? v.toStringAsFixed(0) : v.abs() >= 10 ? v.toStringAsFixed(1) : v.toStringAsFixed(2);
}
''')
print('OK  graphs_screen.dart')

# ============ lib/screens/log_graph_screen.dart ============
with open('lib/screens/log_graph_screen.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:math' as math;

import 'package:fl_chart/fl_chart.dart';
import 'package:flutter/material.dart';

import '../services/logger_service.dart';
import '../widgets/heat_colors.dart';

/// ЛогГраф (как в V6): загрузка CSV, до 4 параметров, статистика min/max/avg,
/// децимация до 1500 точек. Zoom — через InteractiveViewer.
class LogGraphScreen extends StatefulWidget {
  const LogGraphScreen({super.key});

  @override
  State<LogGraphScreen> createState() => _LogGraphScreenState();
}

class _LogGraphScreenState extends State<LogGraphScreen> {
  static const _colors = [Color(0xFF4ADE80), Color(0xFF60A5FA), Color(0xFFE7B93C), Color(0xFFF472B6)];

  List<Map<String, double>> _rows = [];
  String _fileName = '';
  final Set<String> _sel = {'rpm', 'boost', 'kca'};
  String _status = 'Открой CSV лог из вкладки ниже';

  List<String> get _keys {
    final s = <String>{};
    for (final r in _rows.take(200)) { s.addAll(r.keys); }
    s.removeWhere((k) => k == 'ts');
    return s.toList()..sort();
  }

  Future<void> _pick() async {
    final files = await LoggerService.listLogs();
    if (!mounted) return;
    if (files.isEmpty) {
      setState(() => _status = 'Логов нет — запиши на вкладке ЛОГ');
      return;
    }
    final chosen = await showModalBottomSheet<File>(
      context: context,
      backgroundColor: HeatColors.panel,
      builder: (ctx) => ListView(
        children: files
            .map((f) => ListTile(
                  leading: const Icon(Icons.description_outlined, color: HeatColors.accent),
                  title: Text(f.path.split('/').last, style: const TextStyle(fontSize: 13)),
                  onTap: () => Navigator.pop(ctx, File(f.path)),
                ))
            .toList(),
      ),
    );
    if (chosen == null) return;
    final rows = await LoggerService.parseLog(chosen.path);
    setState(() {
      _rows = rows;
      _fileName = chosen.path.split('/').last;
      _status = '$_fileName · ${rows.length} строк';
    });
  }

  @override
  Widget build(BuildContext context) {
    return Column(
      children: [
        Padding(
          padding: const EdgeInsets.all(10),
          child: Row(children: [
            FilledButton.icon(
                onPressed: _pick,
                icon: const Icon(Icons.folder_open, size: 18),
                label: const Text('ОТКРЫТЬ ЛОГ')),
            const SizedBox(width: 10),
            Expanded(
                child: Text(_status,
                    style: const TextStyle(fontSize: 11, color: HeatColors.dim),
                    overflow: TextOverflow.ellipsis)),
          ]),
        ),
        if (_rows.isNotEmpty)
          SizedBox(
            height: 40,
            child: ListView(
              scrollDirection: Axis.horizontal,
              padding: const EdgeInsets.symmetric(horizontal: 8),
              children: _keys.map((k) {
                final on = _sel.contains(k);
                return Padding(
                  padding: const EdgeInsets.only(right: 6),
                  child: FilterChip(
                    label: Text(k, style: const TextStyle(fontSize: 11)),
                    selected: on,
                    onSelected: (v) => setState(() {
                      if (v) {
                        if (_sel.length < 4) _sel.add(k);
                      } else {
                        _sel.remove(k);
                      }
                    }),
                  ),
                );
              }).toList(),
            ),
          ),
        Expanded(
          child: _rows.isEmpty
              ? const Center(
                  child: Icon(Icons.show_chart, size: 48, color: HeatColors.grid))
              : Padding(
                  padding: const EdgeInsets.fromLTRB(4, 6, 12, 8),
                  child: InteractiveViewer(
                    constrained: false,
                    scaleEnabled: true,
                    panEnabled: true,
                    minScale: 0.5,
                    maxScale: 8,
                    child: SizedBox(
                      width: MediaQuery.of(context).size.width - 24,
                      height: double.infinity,
                      child: _buildChart(),
                    ),
                  ),
                ),
        ),
        if (_rows.isNotEmpty)
          Container(
            color: HeatColors.panel,
            padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 8),
            child: Row(
              children: [
                for (var i = 0; i < _sel.length; i++)
                  Expanded(child: _stat(_sel.elementAt(i), _colors[i % 4])),
              ],
            ),
          ),
      ],
    );
  }

  Widget _buildChart() {
    final sel = _sel.where((k) => _rows.any((r) => r.containsKey(k))).toList();
    final step = math.max(1, (_rows.length / 1500).floor());
    return LineChart(
      LineChartData(
        clipData: const FlClipData.all(),
        gridData: FlGridData(
          show: true,
          drawVerticalLine: false,
          getDrawingHorizontalLine: (_) => const FlLine(color: HeatColors.grid, strokeWidth: 0.5),
        ),
        borderData: FlBorderData(show: false),
        titlesData: const FlTitlesData(
          topTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
          rightTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
          bottomTitles: AxisTitles(sideTitles: SideTitles(showTitles: false)),
          leftTitles: AxisTitles(sideTitles: SideTitles(showTitles: true, reservedSize: 40)),
        ),
        lineBarsData: List.generate(sel.length, (i) {
          final key = sel[i];
          final spots = <FlSpot>[];
          for (var j = 0; j < _rows.length; j += step) {
            final v = _rows[j][key];
            if (v != null) spots.add(FlSpot(j.toDouble(), v));
          }
          return LineChartBarData(
            spots: spots,
            color: _colors[i % _colors.length],
            barWidth: 1.4,
            isCurved: false,
            dotData: const FlDotData(show: false),
          );
        }),
      ),
      duration: Duration.zero,
    );
  }

  Widget _stat(String key, Color color) {
    var lo = double.infinity, hi = -double.infinity, sum = 0.0;
    var n = 0;
    for (final r in _rows) {
      final v = r[key];
      if (v != null) { lo = math.min(lo, v); hi = math.max(hi, v); sum += v; n++; }
    }
    if (n == 0) return const SizedBox.shrink();
    return Column(children: [
      Text(key, style: TextStyle(fontSize: 9, color: color)),
      Text('${hi == lo ? '' : ''}${_f(lo)} / ${_f(sum / n)} / ${_f(hi)}',
          style: TextStyle(fontSize: 10.5, fontWeight: FontWeight.w700, color: color)),
      const Text('min / avg / max', style: TextStyle(fontSize: 7.5, color: HeatColors.dim)),
    ]);
  }

  String _f(double v) => v.abs() >= 1000
      ? v.toStringAsFixed(0)
      : v.abs() >= 100
          ? v.toStringAsFixed(0)
          : v.abs() >= 10
              ? v.toStringAsFixed(1)
              : v.toStringAsFixed(2);
}
''')
print('OK  log_graph_screen.dart')

# ============ lib/screens/logging_screen.dart ============
with open('lib/screens/logging_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:share_plus/share_plus.dart';

import '../constants.dart';
import '../services/connection_service.dart';
import '../services/logger_service.dart';
import '../services/settings_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

class LoggingScreen extends StatefulWidget {
  const LoggingScreen({super.key});

  @override
  State<LoggingScreen> createState() => _LoggingScreenState();
}

class _LoggingScreenState extends State<LoggingScreen> {
  Timer? _t;
  int _filesVersion = 0;
  DateTime? _lastActive;

  @override
  void initState() {
    super.initState();
    _t = Timer.periodic(const Duration(seconds: 1), (_) {
      if (mounted) setState(() {});
      _autoTick();
    });
  }

  void _autoTick() {
    if (!SettingsService.I.autoLog) return;
    final svc = ConnectionService.I;
    final snap = svc.poller?.last;
    if (snap == null || svc.elm.state != SsmState.polling) return;
    if (snap.rpm > AppConstants.autoLogRpm || snap.underBoost) {
      _lastActive = DateTime.now();
      if (!svc.logger.logging) _startLog();
    } else if (svc.logger.logging && _lastActive != null) {
      if (DateTime.now().difference(_lastActive!).inSeconds > AppConstants.autoLogIdleSec) {
        _stopLog();
      }
    }
  }

  Future<void> _startLog() async {
    final svc = ConnectionService.I;
    final p = svc.poller;
    if (p == null) return;
    await svc.logger.start(p.snapshots, SettingsService.I.selectedPids);
    if (mounted) setState(() {});
  }

  Future<void> _stopLog() async {
    await ConnectionService.I.logger.stop();
    if (mounted) setState(() => _filesVersion++);
  }

  @override
  void dispose() {
    _t?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final svc = ConnectionService.I;
    final logging = svc.logger.logging;
    return Column(
      children: [
        Container(
          margin: const EdgeInsets.all(12),
          padding: const EdgeInsets.all(14),
          decoration:
              BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          child: Row(
            children: [
              Icon(logging ? Icons.fiber_manual_record : Icons.stop_circle_outlined,
                  color: logging ? Colors.redAccent : HeatColors.dim),
              const SizedBox(width: 10),
              Expanded(
                child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start,
                  children: [
                    Text(logging ? 'ЗАПИСЬ · ${svc.logger.rows} строк' : 'Лог остановлен',
                        style: const TextStyle(fontWeight: FontWeight.w700)),
                    Text(
                      logging
                          ? svc.logger.filePath?.split('/').last ?? ''
                          : 'CSV: выбранные PID + ошибка буста + расход',
                      style: const TextStyle(fontSize: 11, color: HeatColors.dim),
                    ),
                  ],
                ),
              ),
              Switch(
                value: SettingsService.I.autoLog,
                onChanged: (v) {
                  SettingsService.I.autoLog = v;
                  SettingsService.I.save();
                  setState(() {});
                },
              ),
              const Text('авто', style: TextStyle(fontSize: 11, color: HeatColors.dim)),
            ],
          ),
        ),
        Padding(
          padding: const EdgeInsets.symmetric(horizontal: 12),
          child: Row(
            children: [
              Expanded(
                child: FilledButton.icon(
                  style: FilledButton.styleFrom(
                      backgroundColor: logging ? Colors.red.shade900 : HeatColors.accent),
                  onPressed:
                      svc.elm.state == SsmState.polling ? (logging ? _stopLog : _startLog) : null,
                  icon: Icon(logging ? Icons.stop : Icons.play_arrow),
                  label: Text(logging ? 'СТОП' : 'СТАРТ'),
                ),
              ),
            ],
          ),
        ),
        const SizedBox(height: 8),
        Expanded(
          child: FutureBuilder<List<FileSystemEntity>>(
            key: ValueKey(_filesVersion),
            future: LoggerService.listLogs(),
            builder: (ctx, snap) {
              final files = snap.data ?? [];
              if (files.isEmpty) {
                return const Center(
                    child: Text('Логов пока нет', style: TextStyle(color: HeatColors.dim)));
              }
              return ListView.builder(
                itemCount: files.length,
                itemBuilder: (ctx, i) {
                  final f = files[i];
                  final sizeKb = (f.statSync().size / 1024).toStringAsFixed(0);
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.description_outlined, color: HeatColors.accent),
                    title: Text(f.path.split('/').last, style: const TextStyle(fontSize: 13)),
                    subtitle: Text('$sizeKb KB',
                        style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
                    trailing: Row(
                      mainAxisSize: MainAxisSize.min,
                      children: [
                        IconButton(
                            icon: const Icon(Icons.share, size: 20),
                            onPressed: () => Share.shareXFiles([XFile(f.path)])),
                        IconButton(
                            icon: const Icon(Icons.delete_outline, size: 20),
                            onPressed: () async {
                              await f.delete();
                              setState(() => _filesVersion++);
                            }),
                      ],
                    ),
                  );
                },
              );
            },
          ),
        ),
      ],
    );
  }
}
''')
print('OK  logging_screen.dart')

# ============ lib/screens/events_screen.dart ============
with open('lib/screens/events_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/material.dart';
import 'package:intl/intl.dart';

import '../services/alert_service.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';

/// События: турбо-алерты со снимком всех параметров в момент срабатывания (как в V6)
class EventsScreen extends StatefulWidget {
  const EventsScreen({super.key});

  @override
  State<EventsScreen> createState() => _EventsScreenState();
}

class _EventsScreenState extends State<EventsScreen> {
  StreamSubscription<AlertEvent>? _sub;

  @override
  void initState() {
    super.initState();
    _sub = ConnectionService.I.alerts.stream.listen((_) {
      if (mounted) setState(() {});
    });
  }

  @override
  void dispose() {
    _sub?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final events = ConnectionService.I.alerts.events;
    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 8),
          child: Row(
            children: [
              Text('${events.length} событий',
                  style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
              const Spacer(),
              TextButton.icon(
                onPressed: () => setState(() => ConnectionService.I.alerts.clear()),
                icon: const Icon(Icons.delete_sweep_outlined, size: 18),
                label: const Text('очистить'),
              ),
            ],
          ),
        ),
        Expanded(
          child: events.isEmpty
              ? const Center(
                  child: Text('Событий нет — и это хорошо',
                      style: TextStyle(color: HeatColors.dim)))
              : ListView.builder(
                  itemCount: events.length,
                  itemBuilder: (ctx, i) {
                    final e = events[i];
                    final danger = e.level == 2;
                    return ListTile(
                      dense: true,
                      leading: Icon(
                          danger ? Icons.warning_amber_rounded : Icons.info_outline,
                          color: danger ? Colors.redAccent : Colors.orangeAccent),
                      title: Text(e.text, style: const TextStyle(fontSize: 12.5)),
                      subtitle: Text(
                          '${DateFormat('HH:mm:ss').format(e.ts)} · ${e.code}${e.snapshot.isEmpty ? '' : ' · snapshot: rpm ${e.snapshot['rpm']?.toStringAsFixed(0) ?? '--'}, boost ${e.snapshot['boost']?.toStringAsFixed(2) ?? '--'}'}',
                          style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                      onTap: () => _showSnapshot(e),
                    );
                  },
                ),
        ),
      ],
    );
  }

  void _showSnapshot(AlertEvent e) {
    showDialog(
      context: context,
      builder: (ctx) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: Text('${e.code} · ${DateFormat('HH:mm:ss').format(e.ts)}',
            style: const TextStyle(fontSize: 15)),
        content: SizedBox(
          width: 320,
          child: SingleChildScrollView(
            child: Column(
              crossAxisAlignment: CrossAxisAlignment.start,
              children: [
                Text(e.text, style: const TextStyle(fontSize: 13, color: HeatColors.gold)),
                const Divider(),
                ...e.snapshot.entries.map((kv) => Padding(
                      padding: const EdgeInsets.symmetric(vertical: 1.5),
                      child: Row(children: [
                        Expanded(
                            child: Text(kv.key,
                                style: const TextStyle(fontSize: 11, color: HeatColors.dim))),
                        Text(kv.value.toStringAsFixed(2), style: const TextStyle(fontSize: 12)),
                      ]),
                    )),
              ],
            ),
          ),
        ),
        actions: [
          TextButton(onPressed: () => Navigator.pop(ctx), child: const Text('ЗАКРЫТЬ')),
        ],
      ),
    );
  }
}
''')
print('OK  events_screen.dart')
print()
print('=' * 64)
print('  Ядро UI готово (6 экранов). Далее -> ячейка 8/10 (настройческие экраны)')
print('=' * 64)


OK  widgets/heat_colors.dart
OK  widgets/heat_map.dart (цветные карты как в V6)
OK  main.dart (17 вкладок)
OK  dashboard_screen.dart
OK  graphs_screen.dart
OK  log_graph_screen.dart
OK  logging_screen.dart
OK  events_screen.dart

  Ядро UI готово (6 экранов). Далее -> ячейка 8/10 (настройческие экраны)


In [79]:
# @title 🎛️ Ячейка 8/10: DTC · Анализатор(+сплиттер) · ЭБУ Карты · ROM · ROM Diff · Сервис · Замер · Терминал
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/screens/dtc_screen.dart ============
with open('lib/screens/dtc_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/dtc_service.dart';
import '../widgets/heat_colors.dart';

class DtcScreen extends StatefulWidget {
  const DtcScreen({super.key});

  @override
  State<DtcScreen> createState() => _DtcScreenState();
}

class _DtcScreenState extends State<DtcScreen> {
  List<DtcItem>? _codes;
  bool _busy = false;
  String _msg = '';

  Future<void> _read() async {
    setState(() { _busy = true; _msg = 'читаю DTC (опрос на паузе)...'; });
    try {
      final codes = await ConnectionService.I.exclusive((elm) => DtcService.read(elm));
      setState(() {
        _codes = codes;
        _msg = codes.isEmpty ? 'Ошибок нет (mode 03 пуст)' : 'Найдено: ${codes.length}';
      });
    } catch (e) {
      setState(() => _msg = 'ошибка: $e');
    }
    setState(() => _busy = false);
  }

  Future<void> _clear() async {
    setState(() { _busy = true; _msg = 'стираю (mode 04)...'; });
    try {
      final ok = await ConnectionService.I.exclusive((elm) => DtcService.clear(elm));
      setState(() {
        _msg = ok ? 'CEL стёрт. Дай мотору поработать и перечитай.' : 'ECU не подтвердил стирание';
        if (ok) _codes = [];
      });
    } catch (e) {
      setState(() => _msg = 'ошибка: $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    return Column(
      children: [
        Container(
          margin: const EdgeInsets.all(12),
          padding: const EdgeInsets.all(12),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          child: Column(children: [
            Row(children: [
              Expanded(
                child: FilledButton.icon(
                    onPressed: _busy ? null : _read,
                    icon: const Icon(Icons.search),
                    label: const Text('ЧИТАТЬ DTC')),
              ),
              const SizedBox(width: 8),
              Expanded(
                child: OutlinedButton.icon(
                    onPressed: _busy ? null : _clear,
                    icon: const Icon(Icons.cleaning_services_outlined),
                    label: const Text('СТЕРЕТЬ CEL')),
              ),
            ]),
            if (_msg.isNotEmpty)
              Padding(
                padding: const EdgeInsets.only(top: 8),
                child: Align(
                    alignment: Alignment.centerLeft,
                    child: Text(_msg, style: const TextStyle(fontSize: 12, color: HeatColors.gold))),
              ),
          ]),
        ),
        Expanded(
          child: _codes == null
              ? const Center(
                  child: Text('OBD mode 03/04 на шине 0x7E0 · описания Subaru на русском',
                      style: TextStyle(color: HeatColors.dim, fontSize: 12)))
              : _codes!.isEmpty
                  ? const Center(child: Text('Ошибок нет', style: TextStyle(color: Colors.greenAccent)))
                  : ListView.builder(
                      itemCount: _codes!.length,
                      itemBuilder: (ctx, i) {
                        final c = _codes![i];
                        return ListTile(
                          dense: true,
                          leading: const Icon(Icons.error_outline, color: Colors.redAccent),
                          title: Text(c.code,
                              style: const TextStyle(fontWeight: FontWeight.w800, fontSize: 15)),
                          subtitle: Text(c.text,
                              style: const TextStyle(fontSize: 12, color: HeatColors.text)),
                        );
                      },
                    ),
        ),
      ],
    );
  }
}
''')
print('OK  dtc_screen.dart')

# ============ lib/screens/analyzer_screen.dart (сплиттер слияния как в V6) ============
with open('lib/screens/analyzer_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:io';

import 'package:flutter/material.dart';

import '../services/analyzer_service.dart';
import '../services/connection_service.dart';
import '../services/logger_service.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// Анализатор V8: Subaru-метрики по сетке (по умолчанию Обороты × Буст).
/// Онлайн + из лога + СЛИЯНИЕ нескольких CSV (сплиттер V6: суммы+счётчики складываются).
class AnalyzerScreen extends StatefulWidget {
  const AnalyzerScreen({super.key});

  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen> {
  bool _onlineMode = true;
  Timer? _ticker;
  String _status = '';
  final List<String> _mergedLogs = [];

  AnalyzerService get a => ConnectionService.I.analyzer;

  @override
  void initState() {
    super.initState();
    _ticker = Timer.periodic(const Duration(milliseconds: 600), (_) {
      if (mounted && _onlineMode) setState(() {});
    });
  }

  @override
  void dispose() {
    _ticker?.cancel();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final metrics =
        AnalyzerService.metricNames.entries.where((e) => e.key != 'ect2').toList();
    final range = a.grid.range();
    return ListView(
      padding: const EdgeInsets.all(12),
      children: [
        SegmentedButton<bool>(
          segments: const [
            ButtonSegment(value: true, icon: Icon(Icons.wifi_tethering), label: Text('ОНЛАЙН')),
            ButtonSegment(value: false, icon: Icon(Icons.description_outlined), label: Text('ИЗ ЛОГА')),
          ],
          selected: {_onlineMode},
          onSelectionChanged: (s) => setState(() => _onlineMode = s.first),
        ),
        const SizedBox(height: 10),
        Wrap(spacing: 8, runSpacing: 8, children: [
          _drop<AxisOpt>('Ось X', AnalyzerService.axisX, a.xOpt, (o) => o.label,
              (o) => setState(() => a.setAxes(o, a.yOpt, a.metric))),
          _drop<AxisOpt>('Ось Y', AnalyzerService.axisY, a.yOpt, (o) => o.label,
              (o) => setState(() => a.setAxes(a.xOpt, o, a.metric))),
          _drop<MapEntry<String, String>>(
              'Метрика',
              metrics,
              metrics.firstWhere((e) => e.key == a.metric, orElse: () => metrics.first),
              (o) => o.value,
              (o) => setState(() => a.setAxes(a.xOpt, a.yOpt, o.key))),
        ]),
        const SizedBox(height: 8),
        Wrap(spacing: 8, runSpacing: 8, crossAxisAlignment: WrapCrossAlignment.center, children: [
          if (!_onlineMode)
            FilledButton.icon(
              onPressed: () => _pickLogAndMerge(reset: true),
              icon: const Icon(Icons.folder_open, size: 18),
              label: const Text('Открыть CSV'),
            )
          else
            FilledButton.icon(
              onPressed: _toggleOnline,
              icon: Icon(a.online ? Icons.pause : Icons.play_arrow, size: 18),
              label: Text(a.online ? 'Пауза' : 'Захват из опроса'),
            ),
          if (!_onlineMode && a.grid.totalHits > 0)
            FilledButton.tonalIcon(
              onPressed: () => _pickLogAndMerge(reset: false),
              icon: const Icon(Icons.merge, size: 18),
              label: const Text('СЛИТЬ ЕЩЁ ЛОГ'),
            ),
          OutlinedButton.icon(
            onPressed: () => setState(() { a.grid.reset(); _mergedLogs.clear(); }),
            icon: const Icon(Icons.cleaning_services_outlined, size: 18),
            label: const Text('Сброс'),
          ),
          Text('точек: ${a.grid.totalHits}',
              style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
        ]),
        if (_status.isNotEmpty)
          Padding(
            padding: const EdgeInsets.only(top: 6),
            child: Text(_status, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
          ),
        if (_mergedLogs.isNotEmpty)
          Padding(
            padding: const EdgeInsets.only(top: 4),
            child: Text('слитые логи: ${_mergedLogs.join(" + ")}',
                style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
          ),
        const SizedBox(height: 12),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 420,
          child: HeatMapView.fromGrid(a.grid,
              title: AnalyzerService.metricNames[a.metric] ?? a.metric),
        ),
        const SizedBox(height: 8),
        Text(
          'X: ${a.xOpt.label}  ·  Y: ${a.yOpt.label}  ·  '
          'диапазон ${range.lo.toStringAsFixed(2)} … ${range.hi.toStringAsFixed(2)}\n'
          'Ячейка = среднее. Синий→красный = min→max. Тап — значение и число замеров.',
          style: const TextStyle(fontSize: 11, color: HeatColors.dim),
        ),
      ],
    );
  }

  Widget _drop<T>(String hint, List<T> items, T value, String Function(T) label, void Function(T) onSel) {
    return Container(
      padding: const EdgeInsets.symmetric(horizontal: 10),
      decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(10)),
      child: DropdownButtonHideUnderline(
        child: DropdownButton<T>(
          hint: Text(hint, style: const TextStyle(fontSize: 12)),
          value: value,
          isDense: true,
          dropdownColor: HeatColors.panel,
          items: items
              .map((e) => DropdownMenuItem<T>(
                  value: e, child: Text('$hint: ${label(e)}', style: const TextStyle(fontSize: 12))))
              .toList(),
          onChanged: (v) { if (v != null) onSel(v); },
        ),
      ),
    );
  }

  void _toggleOnline() {
    final svc = ConnectionService.I;
    if (a.online) {
      a.stopOnline();
    } else {
      final p = svc.poller;
      if (p == null) {
        setState(() => _status = 'Нет опроса: подключись в настройках');
        return;
      }
      a.startOnline(p.snapshots);
      setState(() => _status = 'Захват из живого опроса...');
    }
    setState(() {});
  }

  Future<void> _pickLogAndMerge({required bool reset}) async {
    final files = await LoggerService.listLogs();
    if (!mounted) return;
    if (files.isEmpty) {
      setState(() => _status = 'Логов нет');
      return;
    }
    final chosen = await showModalBottomSheet<File>(
      context: context,
      backgroundColor: HeatColors.panel,
      builder: (ctx) => ListView(
        children: files
            .map((f) => ListTile(
                  leading: const Icon(Icons.description_outlined, color: HeatColors.accent),
                  title: Text(f.path.split('/').last, style: const TextStyle(fontSize: 13)),
                  onTap: () => Navigator.pop(ctx, File(f.path)),
                ))
            .toList(),
      ),
    );
    if (chosen == null) return;
    if (reset) { a.grid.reset(); _mergedLogs.clear(); }
    final rows = await LoggerService.parseLog(chosen.path);
    a.buildFromRows(rows); // суммы и счётчики накапливаются => это и есть слияние
    final name = chosen.path.split('/').last;
    setState(() {
      _mergedLogs.add(name);
      _status = '$name: +${rows.length} строк → всего ${a.grid.totalHits} точек';
    });
  }
}
''')
print('OK  analyzer_screen.dart (сплиттер слияния CSV)')

# ============ lib/screens/ecu_maps_screen.dart ============
with open('lib/screens/ecu_maps_screen.dart', 'w') as f:
    f.write(r'''import 'dart:typed_data';

import 'package:flutter/material.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';
import '../services/connection_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// ЭБУ Карты: чтение таблиц ЖИВЬЁМ из ECU по SSM2-блоками (как «ЭБУ Карты» V6,
/// но по A8-чтению диапазонов). Просмотр; правка — только через ROM-экран в .bin.
class EcuMapsScreen extends StatefulWidget {
  const EcuMapsScreen({super.key});

  @override
  State<EcuMapsScreen> createState() => _EcuMapsScreenState();
}

class _EcuMapsScreenState extends State<EcuMapsScreen> {
  String _filter = 'all';
  bool _reading = false;
  double _progress = 0;
  String _status = '';

  @override
  Widget build(BuildContext context) {
    final defs = SubaruRom.tables;
    final cats = <String>{for (final d in defs) d.category}.toList()..sort();
    final filtered = defs.where((d) => _filter == 'all' || d.category == _filter).toList();
    final connected = ConnectionService.I.elm.state == SsmState.ecuReady ||
        ConnectionService.I.elm.state == SsmState.polling;

    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.all(10),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Text(
              connected
                  ? 'Чтение карт напрямую из ECU (SSM2 A8-блоками, ${defs.length} карт в дефинишне)'
                  : 'Сначала подключись в Настройках',
              style: const TextStyle(fontSize: 12),
            ),
            if (_reading) ...[
              const SizedBox(height: 8),
              LinearProgressIndicator(value: _progress, backgroundColor: HeatColors.bg),
              const SizedBox(height: 4),
              Text(_status, style: const TextStyle(fontSize: 11, color: HeatColors.gold)),
            ],
          ]),
        ),
        SizedBox(
          height: 40,
          child: ListView(scrollDirection: Axis.horizontal, padding: const EdgeInsets.symmetric(horizontal: 8), children: [
            Padding(
              padding: const EdgeInsets.only(right: 6),
              child: ChoiceChip(
                label: Text('ВСЕ (${defs.length})', style: const TextStyle(fontSize: 11)),
                selected: _filter == 'all',
                onSelected: (_) => setState(() => _filter = 'all'),
              ),
            ),
            ...cats.map((c) => Padding(
                  padding: const EdgeInsets.only(right: 6),
                  child: ChoiceChip(
                    label: Text('$c (${defs.where((d) => d.category == c).length})',
                        style: const TextStyle(fontSize: 11)),
                    selected: _filter == c,
                    onSelected: (_) => setState(() => _filter = c),
                  ),
                )),
          ]),
        ),
        Expanded(
          child: ListView.builder(
            itemCount: filtered.length,
            itemBuilder: (ctx, i) {
              final d = filtered[i];
              final bytes = d.rows * d.cols * d.data.sizeOf;
              return ListTile(
                dense: true,
                enabled: connected && !_reading,
                leading: const Icon(Icons.memory, size: 20, color: HeatColors.accent),
                title: Text(d.name, style: const TextStyle(fontSize: 13)),
                subtitle: Text('${d.rows}×${d.cols} · ${d.units} · ${d.addrHex} · $bytes B',
                    style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                trailing: const Icon(Icons.download_for_offline_outlined, size: 18),
                onTap: () => _readFromEcu(d),
              );
            },
          ),
        ),
      ],
    );
  }

  Future<void> _readFromEcu(RomTableDef def) async {
    setState(() { _reading = true; _progress = 0; _status = 'пауза опроса...'; });
    try {
      final table = await ConnectionService.I.exclusive((elm) => _readTable(elm, def));
      if (!mounted || table == null) return;
      setState(() { _reading = false; _status = ''; });
      await Navigator.push(
          context, MaterialPageRoute(builder: (_) => EcuTablePage(table: table)));
    } catch (e) {
      setState(() { _reading = false; _status = 'ошибка: $e'; });
    }
  }

  /// Собираем «виртуальный ROM» из адресных кусков ECU и декодируем как файл
  Future<RomTable?> _readTable(SsmElm elm, RomTableDef def) async {
    final jobs = <_RangeJob>[];
    jobs.add(_RangeJob(def.address, def.data.count * def.data.sizeOf));
    if (def.xAxis != null && def.xAxis!.address >= 0) {
      jobs.add(_RangeJob(def.xAxis!.address, def.xAxis!.byteLen));
    }
    if (def.yAxis != null && def.yAxis!.address >= 0) {
      jobs.add(_RangeJob(def.yAxis!.address, def.yAxis!.byteLen));
    }
    var minAddr = 1 << 30, maxAddr = 0;
    for (final j in jobs) {
      minAddr = j.addr < minAddr ? j.addr : minAddr;
      maxAddr = (j.addr + j.len) > maxAddr ? j.addr + j.len : maxAddr;
    }
    final shift = minAddr;
    final rom = Uint8List(maxAddr - shift);
    final idx = SubaruRom.tables.indexOf(def);

    final totalChunks = jobs.fold<int>(0, (a, j) => a + ((j.len + 0x7F) >> 7));
    var done = 0;
    for (final j in jobs) {
      for (var off = 0; off < j.len; off += 0x80) {
        final want = (j.len - off) > 0x80 ? 0x80 : (j.len - off);
        final chunk = await elm.readBytes(j.addr + off, want);
        done++;
        if (mounted) {
          setState(() {
            _progress = done / totalChunks;
            _status = '${def.name}  ·  кадр $done/$totalChunks';
          });
        }
        if (chunk == null) continue; // дырки оставляем нулями — ячейки покажут 0/NaN
        final base = j.addr + off - shift;
        for (var i = 0; i < chunk.length && base + i < rom.length; i++) {
          rom[base + i] = chunk[i];
        }
      }
    }

    // decode (те же правила, что у ROM-файла)
    ByteData bd = ByteData.sublistView(rom);
    double colVal(RomCol? col, int i, List<double> fallback) {
      if (col == null) return fallback[i];
      if (col.storage == 'static') {
        final vals = SubaruRom.staticAxes['${col == def.xAxis ? 'x' : 'y'}$idx'];
        if (vals != null && i < vals.length) return vals[i];
        return fallback[i];
      }
      final shifted = RomCol(
        address: col.address - shift, count: col.count,
        storage: col.storage, endian: col.endian, to: col.to, fr: col.fr);
      try {
        return shifted.value(rom, bd, i);
      } catch (_) {
        return double.nan;
      }
    }

    final xs = List<double>.generate(def.cols, (i) => i.toDouble());
    final ys = List<double>.generate(def.rows, (i) => i.toDouble());
    final xVals = List<double>.generate(def.cols, (i) => colVal(def.xAxis, i, xs));
    final yVals = List<double>.generate(def.rows, (i) => colVal(def.yAxis, i, ys));
    final dataShifted = RomCol(
        address: def.address - shift, count: def.data.count,
        storage: def.data.storage, endian: def.data.endian, to: def.data.to);
    final z = List.generate(def.rows, (_) => List<double>.filled(def.cols, 0));
    for (var r = 0; r < def.rows; r++) {
      for (var c = 0; c < def.cols; c++) {
        final i = def.swapxy ? (c * def.rows + r) : (r * def.cols + c);
        try {
          z[r][c] = dataShifted.value(rom, bd, i);
        } catch (_) {
          z[r][c] = double.nan;
        }
      }
    }
    final tdef = RomTableDef(
        name: def.name, category: def.category, address: def.address,
        rows: def.rows, cols: def.cols, swapxy: def.swapxy, units: def.units,
        data: def.data, xAxis: def.xAxis, yAxis: def.yAxis,
        minHint: def.minHint, maxHint: def.maxHint);
    return RomTable(def: tdef, xValues: xVals, yValues: yVals, z: z);
  }
}

class _RangeJob {
  final int addr;
  final int len;
  _RangeJob(this.addr, this.len);
}

/// Просмотр карты, прочитанной из ECU (read-only)
class EcuTablePage extends StatelessWidget {
  final RomTable table;
  const EcuTablePage({super.key, required this.table});

  @override
  Widget build(BuildContext context) {
    final d = table.def;
    return Scaffold(
      appBar: AppBar(title: Text('${d.name} · ECU live', style: const TextStyle(fontSize: 14))),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        Wrap(spacing: 12, runSpacing: 4, children: [
          _kv('адрес', d.addrHex), _kv('размер', '${d.rows}×${d.cols}'),
          _kv('ед.', d.units), _kv('min', table.minV.toStringAsFixed(2)),
          _kv('max', table.maxV.toStringAsFixed(2)),
        ]),
        const SizedBox(height: 10),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 440,
          child: HeatMapView(
            cols: d.cols,
            rows: d.rows,
            unit: d.units,
            hintLo: d.minHint.isNaN ? null : d.minHint,
            hintHi: d.maxHint.isNaN ? null : d.maxHint,
            value: (r, c) => table.z[d.rows - 1 - r][c],
            xLabel: (c) => c < table.xValues.length ? _ax(table.xValues[c]) : '$c',
            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < table.yValues.length ? _ax(table.yValues[i]) : '$i';
            },
          ),
        ),
        const SizedBox(height: 8),
        const Text('Прочитано живьём из ECU. Read-only: правка карт — на экране ROM (в .bin-файл).',
            style: TextStyle(fontSize: 11, color: HeatColors.dim)),
      ]),
    );
  }

  String _ax(double v) => v.abs() >= 100 ? v.toStringAsFixed(0) : v.toStringAsFixed(1);

  Widget _kv(String k, String v) => RichText(
        text: TextSpan(children: [
          TextSpan(text: '$k: ', style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextSpan(text: v, style: const TextStyle(fontSize: 11, color: HeatColors.text)),
        ]),
      );
}
''')
print('OK  ecu_maps_screen.dart (живое чтение карт из ECU)')

# ============ lib/screens/rom_screen.dart ============
with open('lib/screens/rom_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:share_plus/share_plus.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// ROM-экран: загрузка .bin A2TB100B, все карты из дефинишна,
/// цветной просмотр, правка ячеек, сохранение V8MOD_*.bin.
class RomScreen extends StatefulWidget {
  const RomScreen({super.key});

  @override
  State<RomScreen> createState() => _RomScreenState();
}

class _RomScreenState extends State<RomScreen> {
  String _filter = 'all';
  String _query = '';

  @override
  Widget build(BuildContext context) {
    final rom = ConnectionService.I.rom;
    if (!rom.loaded) {
      return Center(
        child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
          const Icon(Icons.memory, size: 56, color: HeatColors.dim),
          const SizedBox(height: 12),
          const Text('Загрузи ROM (.bin) A2TB100B\n— карты поднимутся из встроенного дефинишна',
              textAlign: TextAlign.center, style: TextStyle(color: HeatColors.dim)),
          const SizedBox(height: 16),
          FilledButton.icon(
              onPressed: _load, icon: const Icon(Icons.file_open), label: const Text('ЗАГРУЗИТЬ ROM')),
        ]),
      );
    }

    final defs = rom.defs;
    final cats = <String>{for (final d in defs) d.category}.toList()..sort();
    final filtered = defs.where((d) {
      if (_filter != 'all' && d.category != _filter) return false;
      if (_query.isNotEmpty && !d.name.toLowerCase().contains(_query.toLowerCase())) return false;
      return true;
    }).toList();

    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.all(10),
          child: Row(children: [
            const Icon(Icons.check_circle, color: Colors.greenAccent, size: 16),
            const SizedBox(width: 8),
            Expanded(
                child: Text('${rom.fileName} · ${defs.length} карт · ${SubaruRom.calId}',
                    style: const TextStyle(fontSize: 12))),
            TextButton(onPressed: _load, child: const Text('другой ROM')),
          ]),
        ),
        SizedBox(
          height: 40,
          child: ListView(scrollDirection: Axis.horizontal, padding: const EdgeInsets.symmetric(horizontal: 8), children: [
            Padding(
              padding: const EdgeInsets.only(right: 6),
              child: ChoiceChip(
                label: Text('ВСЕ (${defs.length})', style: const TextStyle(fontSize: 11)),
                selected: _filter == 'all',
                onSelected: (_) => setState(() => _filter = 'all'),
              ),
            ),
            ...cats.map((c) => Padding(
                  padding: const EdgeInsets.only(right: 6),
                  child: ChoiceChip(
                    label: Text('$c (${defs.where((d) => d.category == c).length})',
                        style: const TextStyle(fontSize: 11)),
                    selected: _filter == c,
                    onSelected: (_) => setState(() => _filter = c),
                  ),
                )),
          ]),
        ),
        Padding(
          padding: const EdgeInsets.all(8),
          child: TextField(
            style: const TextStyle(fontSize: 13),
            decoration: InputDecoration(
              hintText: 'поиск карты...',
              isDense: true,
              prefixIcon: const Icon(Icons.search, size: 18),
              filled: true,
              fillColor: HeatColors.panel,
              border: OutlineInputBorder(
                  borderRadius: BorderRadius.circular(10), borderSide: BorderSide.none),
            ),
            onChanged: (v) => setState(() => _query = v),
          ),
        ),
        Expanded(
          child: ListView.builder(
            itemCount: filtered.length,
            itemBuilder: (ctx, i) {
              final d = filtered[i];
              return ListTile(
                dense: true,
                leading: Icon(d.editable ? Icons.grid_on : Icons.grid_off,
                    size: 20, color: d.editable ? HeatColors.gold : HeatColors.dim),
                title: Text(d.name, style: const TextStyle(fontSize: 13)),
                subtitle: Text('${d.rows}×${d.cols} · ${d.units} · ${d.addrHex} · ${d.category}',
                    style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                trailing: const Icon(Icons.chevron_right, size: 18),
                onTap: () => Navigator.push(
                    context, MaterialPageRoute(builder: (_) => RomTableScreen(def: d))),
              );
            },
          ),
        ),
      ],
    );
  }

  Future<void> _load() async {
    final rom = ConnectionService.I.rom;
    final err = await rom.pickAndLoad();
    if (!mounted) return;
    if (err != null) {
      ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(err)));
    }
    setState(() {});
  }
}

class RomTableScreen extends StatefulWidget {
  final RomTableDef def;
  const RomTableScreen({super.key, required this.def});

  @override
  State<RomTableScreen> createState() => _RomTableScreenState();
}

class _RomTableScreenState extends State<RomTableScreen> {
  RomTable? _table;
  bool _modified = false;

  @override
  void initState() {
    super.initState();
    _table = ConnectionService.I.rom.readTable(widget.def);
  }

  @override
  Widget build(BuildContext context) {
    final d = widget.def;
    final t = _table;
    if (t == null) {
      return Scaffold(appBar: AppBar(title: Text(d.name)), body: const Center(child: CircularProgressIndicator()));
    }
    return Scaffold(
      appBar: AppBar(
        title: Text(d.name, style: const TextStyle(fontSize: 15)),
        actions: [
          if (d.editable)
            TextButton.icon(
              onPressed: _modified ? _saveMod : null,
              icon: const Icon(Icons.save_outlined, size: 18),
              label: const Text('V8MOD'),
            ),
          IconButton(icon: const Icon(Icons.ios_share, size: 18), onPressed: _exportCsv),
        ],
      ),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        Wrap(spacing: 12, runSpacing: 4, children: [
          _info('адрес', d.addrHex), _info('размер', '${d.rows}×${d.cols}'),
          _info('ед.', d.units), _info('min', t.minV.toStringAsFixed(2)),
          _info('max', t.maxV.toStringAsFixed(2)), _info('storage', d.data.storage),
        ]),
        const SizedBox(height: 10),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 440,
          child: HeatMapView(
            cols: d.cols,
            rows: d.rows,
            unit: d.units,
            hintLo: d.minHint.isNaN ? null : d.minHint,
            hintHi: d.maxHint.isNaN ? null : d.maxHint,
            value: (r, c) => t.z[d.rows - 1 - r][c],
            xLabel: (c) => c < t.xValues.length ? _ax(t.xValues[c]) : '$c',
            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < t.yValues.length ? _ax(t.yValues[i]) : '$i';
            },
          ),
        ),
        const SizedBox(height: 10),
        if (d.editable)
          FilledButton.icon(
            onPressed: _editCell,
            icon: const Icon(Icons.edit, size: 18),
            label: const Text('Править ячейку'),
          )
        else
          const Text('Карта read-only (нет обратной формулы frexpr)',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
        if (_modified)
          const Padding(
            padding: EdgeInsets.only(top: 6),
            child: Text('Есть несохранённые правки — жми V8MOD',
                style: TextStyle(fontSize: 12, color: HeatColors.gold)),
          ),
      ]),
    );
  }

  String _ax(double v) => v.abs() >= 1000
      ? v.toStringAsFixed(0)
      : v.abs() >= 100
          ? v.toStringAsFixed(0)
          : v.toStringAsFixed(1);

  Widget _info(String k, String v) => RichText(
        text: TextSpan(children: [
          TextSpan(text: '$k: ', style: const TextStyle(fontSize: 11, color: HeatColors.dim)),
          TextSpan(text: v, style: const TextStyle(fontSize: 11, color: HeatColors.text)),
        ]),
      );

  Future<void> _editCell() async {
    final t = _table!;
    final d = widget.def;
    final rCtl = TextEditingController();
    final cCtl = TextEditingController();
    final vCtl = TextEditingController();
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => AlertDialog(
        backgroundColor: HeatColors.panel,
        title: const Text('Правка ячейки', style: TextStyle(fontSize: 15)),
        content: Column(mainAxisSize: MainAxisSize.min, children: [
          TextField(controller: rCtl, keyboardType: TextInputType.number,
              decoration: InputDecoration(labelText: 'Строка 0..${d.rows - 1} (Y)')),
          TextField(controller: cCtl, keyboardType: TextInputType.number,
              decoration: InputDecoration(labelText: 'Колонка 0..${d.cols - 1} (X)')),
          TextField(controller: vCtl,
              keyboardType: const TextInputType.numberWithOptions(decimal: true, signed: true),
              decoration: InputDecoration(labelText: 'Новое значение, ${d.units}')),
        ]),
        actions: [
          TextButton(onPressed: () => Navigator.pop(ctx, false), child: const Text('ОТМЕНА')),
          FilledButton(onPressed: () => Navigator.pop(ctx, true), child: const Text('ЗАПИСАТЬ')),
        ],
      ),
    );
    if (ok != true) return;
    final r = int.tryParse(rCtl.text) ?? -1;
    final c = int.tryParse(cCtl.text) ?? -1;
    final v = double.tryParse(vCtl.text.replaceAll(',', '.'));
    if (r < 0 || r >= d.rows || c < 0 || c >= d.cols || v == null) {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Некорректные данные')));
      return;
    }
    setState(() {
      t.setCell(r, c, v);
      _modified = true;
    });
  }

  Future<void> _saveMod() async {
    final path = await ConnectionService.I.rom.saveMod(_table!);
    if (!mounted) return;
    if (path != null) {
      setState(() => _modified = false);
      ScaffoldMessenger.of(context)
          .showSnackBar(SnackBar(content: Text('Сохранено: ${path.split('/').last}')));
      await Share.shareXFiles([XFile(path)]);
    } else {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Карта read-only')));
    }
  }

  Future<void> _exportCsv() async {
    final t = _table!;
    final buf = StringBuffer();
    for (final row in t.toCsvRows()) {
      buf.writeln(row.join(';'));
    }
    await Share.share(buf.toString(), subject: widget.def.name);
  }
}
''')
print('OK  rom_screen.dart')

# ============ lib/screens/rom_diff_screen.dart ============
with open('lib/screens/rom_diff_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../generated/subaru_rom.g.dart';
import '../models/rom_table.dart';
import '../services/rom_service.dart';
import '../widgets/heat_colors.dart';
import '../widgets/heat_map.dart';

/// ROM Diff: сравнение двух прошивок (как в V6) — дельта-карты, отчёт по всем картам.
class RomDiffScreen extends StatefulWidget {
  const RomDiffScreen({super.key});

  @override
  State<RomDiffScreen> createState() => _RomDiffScreenState();
}

class _Diff {
  final RomTableDef def;
  final int cells;
  final double maxDiff;
  final RomTable ta;
  final RomTable tb;
  _Diff(this.def, this.cells, this.maxDiff, this.ta, this.tb);
}

class _RomDiffScreenState extends State<RomDiffScreen> {
  final _romA = RomService();
  final _romB = RomService();
  List<_Diff>? _report;
  bool _busy = false;
  String _err = '';

  Future<void> _pick(bool isA) async {
    final rom = isA ? _romA : _romB;
    final err = await rom.pickAndLoad();
    setState(() {
      _err = err ?? '';
      _report = null;
    });
    if (err == null && _romA.loaded && _romB.loaded) {
      await _compare();
    }
  }

  Future<void> _compare() async {
    setState(() { _busy = true; _err = ''; });
    await Future.delayed(const Duration(milliseconds: 30));
    final out = <_Diff>[];
    for (final def in SubaruRom.tables) {
      try {
        final ta = _romA.readTable(def);
        final tb = _romB.readTable(def);
        var cells = 0;
        var maxDiff = 0.0;
        for (var r = 0; r < def.rows; r++) {
          for (var c = 0; c < def.cols; c++) {
            final dv = (ta.z[r][c] - tb.z[r][c]).abs();
            if (dv > 0.0001) {
              cells++;
              if (dv > maxDiff) maxDiff = dv;
            }
          }
        }
        if (cells > 0) out.add(_Diff(def, cells, maxDiff, ta, tb));
      } catch (_) {}
    }
    out.sort((a, b) => b.maxDiff.compareTo(a.maxDiff));
    setState(() { _report = out; _busy = false; });
  }

  @override
  Widget build(BuildContext context) {
    return Column(
      children: [
        Container(
          color: HeatColors.panel,
          padding: const EdgeInsets.all(10),
          child: Row(children: [
            Expanded(
              child: OutlinedButton.icon(
                onPressed: () => _pick(true),
                icon: Icon(_romA.loaded ? Icons.check_circle : Icons.file_upload_outlined,
                    size: 16, color: _romA.loaded ? Colors.greenAccent : null),
                label: Text(_romA.loaded ? _romA.fileName : 'ROM A (сток)',
                    overflow: TextOverflow.ellipsis, style: const TextStyle(fontSize: 11)),
              ),
            ),
            const Padding(
                padding: EdgeInsets.symmetric(horizontal: 6),
                child: Icon(Icons.compare_arrows, color: HeatColors.dim)),
            Expanded(
              child: OutlinedButton.icon(
                onPressed: () => _pick(false),
                icon: Icon(_romB.loaded ? Icons.check_circle : Icons.file_upload_outlined,
                    size: 16, color: _romB.loaded ? Colors.greenAccent : null),
                label: Text(_romB.loaded ? _romB.fileName : 'ROM B (мод)',
                    overflow: TextOverflow.ellipsis, style: const TextStyle(fontSize: 11)),
              ),
            ),
          ]),
        ),
        if (_busy) const LinearProgressIndicator(minHeight: 3),
        if (_err.isNotEmpty)
          Padding(padding: const EdgeInsets.all(8), child: Text(_err, style: const TextStyle(color: Colors.redAccent))),
        Expanded(
          child: _report == null
              ? const Center(
                  child: Text('Выбери два .bin — покажу карты с различиями\n(ячейки, max Δ, цветная дельта)',
                      textAlign: TextAlign.center, style: TextStyle(color: HeatColors.dim, fontSize: 12)))
              : _report!.isEmpty
                  ? const Center(child: Text('Различий нет', style: TextStyle(color: Colors.greenAccent)))
                  : ListView.builder(
                      itemCount: _report!.length,
                      itemBuilder: (ctx, i) {
                        final d = _report![i];
                        return ListTile(
                          dense: true,
                          leading: const Icon(Icons.difference, color: HeatColors.gold, size: 20),
                          title: Text(d.def.name, style: const TextStyle(fontSize: 13)),
                          subtitle: Text(
                              '${d.def.category} · ячеек отличается: ${d.cells}/${d.def.rows * d.def.cols} · max Δ ${d.maxDiff.toStringAsFixed(2)} ${d.def.units}',
                              style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                          trailing: const Icon(Icons.chevron_right, size: 18),
                          onTap: () => Navigator.push(
                              context, MaterialPageRoute(builder: (_) => _DiffPage(diff: d))),
                        );
                      },
                    ),
        ),
      ],
    );
  }
}

class _DiffPage extends StatelessWidget {
  final _Diff diff;
  const _DiffPage({required this.diff});

  @override
  Widget build(BuildContext context) {
    final d = diff.def;
    final hi = diff.maxDiff <= 0 ? 1.0 : diff.maxDiff;
    return Scaffold(
      appBar: AppBar(title: Text('Δ ${d.name}', style: const TextStyle(fontSize: 14))),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        Text('A − B · синий = ниже, красный = выше · шкала ±${hi.toStringAsFixed(2)} ${d.units}',
            style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
        const SizedBox(height: 10),
        Container(
          padding: const EdgeInsets.all(10),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          height: 420,
          child: HeatMapView(
            cols: d.cols,
            rows: d.rows,
            unit: 'Δ ${d.units}',
            hintLo: -hi,
            hintHi: hi,
            value: (r, c) => diff.ta.z[d.rows - 1 - r][c] - diff.tb.z[d.rows - 1 - r][c],
            xLabel: (c) => c < diff.ta.xValues.length ? diff.ta.xValues[c].toStringAsFixed(0) : '$c',
            yLabel: (r) {
              final i = d.rows - 1 - r;
              return i < diff.ta.yValues.length ? diff.ta.yValues[i].toStringAsFixed(1) : '$i';
            },
          ),
        ),
      ]),
    );
  }
}
''')
print('OK  rom_diff_screen.dart')

# ============ lib/screens/service_screen.dart ============
with open('lib/screens/service_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:math' as math;

import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/dtc_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

/// Сервис: диагностические операции с монопольным доступом к шине.
/// Запись в ECU из приложения НЕ выполняется (только .bin через ROM-экран).
class ServiceScreen extends StatefulWidget {
  const ServiceScreen({super.key});

  @override
  State<ServiceScreen> createState() => _ServiceScreenState();
}

class _ServiceScreenState extends State<ServiceScreen> {
  final List<String> _log = ['готов. операции ниже — опрос при этом на паузе.'];
  bool _busy = false;

  void _say(String s) => setState(() => _log.insert(0, s));

  Future<void> _run(String label, Future<String> Function(SsmElm elm) job) async {
    if (_busy) return;
    setState(() => _busy = true);
    _say('▶ $label ...');
    try {
      final r = await ConnectionService.I.exclusive(job);
      _say('✔ $label: $r');
    } catch (e) {
      _say('✖ $label: $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    final elm = ConnectionService.I.elm;
    final connected = elm.state == SsmState.ecuReady || elm.state == SsmState.polling;
    return Column(
      children: [
        Expanded(
          flex: 5,
          child: ListView(padding: const EdgeInsets.all(12), children: [
            _op('SSM PING · 20 кадров чтения ОЖ', Icons.speed, connected && !_busy,
                () => _run('PING', (elm) async {
                      final times = <double>[];
                      var err = 0;
                      for (var i = 0; i < 20; i++) {
                        final d = await elm.readBytes(0x000008, 1);
                        if (d == null) {
                          err++;
                        } else {
                          times.add(elm.stats.lastMs);
                        }
                        await Future<void>.delayed(const Duration(milliseconds: 5));
                      }
                      if (times.isEmpty) return 'все 20 кадров потеряны';
                      final mn = times.reduce(math.min), mx = times.reduce(math.max);
                      final avg = times.fold(0.0, (a, b) => a + b) / times.length;
                      return 'min ${mn.toStringAsFixed(0)} / avg ${avg.toStringAsFixed(0)} / max ${mx.toStringAsFixed(0)} мс · потерь $err';
                    })),
            _op('Повторный ECU INIT (BF)', Icons.restart_alt, connected && !_busy,
                () => _run('ECU INIT', (elm) async {
                      final id = await elm.ecuInit();
                      return id == null ? 'ECU не ответил' : 'OK, ECU ID = ${elm.ecuId.isEmpty ? id : elm.ecuId}';
                    })),
            _op('Стирание CEL (OBD mode 04)', Icons.cleaning_services_outlined, connected && !_busy,
                () => _run('CLEAR CEL', (elm) async {
                      final ok = await DtcService.clear(elm);
                      return ok ? 'подтверждено (44)' : 'ECU не подтвердил';
                    })),
            _op('Напряжение адаптера (ATRV)', Icons.bolt, connected && !_busy,
                () => _run('ATRV', (elm) async => (await elm.transact('ATRV')).trim())),
            _op('Чтение 6 ключевых параметров разом', Icons.punch_clock, connected && !_busy,
                () => _run('SNAPSHOT', (elm) async {
                      // блок 0x000008..0x000031 одним кадром — демонстрация блочного чтения
                      final d = await elm.readBytes(0x000008, 0x2A);
                      if (d == null) return 'нет ответа';
                      final ect = d[0] - 40;
                      final iat = d[1] - 40;
                      final tps = d[0x15 - 0x08] * 100 / 255;
                      final rpm = ((d[0x0E - 0x08] << 8) | d[0x0F - 0x08]) / 4;
                      final spd = d[0x10 - 0x08];
                      final kca = d[0x22 - 0x08] / 2 - 20;
                      return 'ECT $ect°C · IAT $iat°C · RPM ${rpm.toStringAsFixed(0)} · '
                          '$spd км/ч · TPS ${tps.toStringAsFixed(0)}% · KCA ${kca.toStringAsFixed(1)}°';
                    })),
            const Padding(
              padding: EdgeInsets.symmetric(vertical: 8),
              child: Text(
                'Правка прошивки в самом ECU из V8 не выполняется: только чтение карт (ЭБУ Карты) '
                'и правка .bin-файла (ROM → V8MOD). Это осознанно безопасно для первой версии.',
                style: TextStyle(fontSize: 11, color: HeatColors.dim),
              ),
            ),
          ]),
        ),
        Expanded(
          flex: 4,
          child: Container(
            color: HeatColors.panel,
            child: ListView.builder(
              reverse: true,
              padding: const EdgeInsets.all(10),
              itemCount: _log.length,
              itemBuilder: (ctx, i) => Text(_log[i],
                  style: const TextStyle(fontSize: 11, color: HeatColors.text, height: 1.5)),
            ),
          ),
        ),
      ],
    );
  }

  Widget _op(String label, IconData icon, bool enabled, VoidCallback onTap) {
    return Padding(
      padding: const EdgeInsets.only(bottom: 8),
      child: ListTile(
        tileColor: HeatColors.panel,
        shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(10)),
        leading: Icon(icon, color: HeatColors.accent),
        title: Text(label, style: const TextStyle(fontSize: 13)),
        trailing: const Icon(Icons.play_arrow, size: 18),
        enabled: enabled,
        onTap: onTap,
      ),
    );
  }
}
''')
print('OK  service_screen.dart')

# ============ lib/screens/perf_screen.dart ============
with open('lib/screens/perf_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';

import 'package:flutter/material.dart';

import '../models/live_snapshot.dart';
import '../services/connection_service.dart';
import '../widgets/heat_colors.dart';

/// Замер (как в V6): 0-60, 0-100, 100-200, 402 м — по скорости VSS.
class PerfScreen extends StatefulWidget {
  const PerfScreen({super.key});

  @override
  State<PerfScreen> createState() => _PerfScreenState();
}

class _Meas {
  final Map<int, double> times; // speed -> seconds
  final double q402m;
  final double q1km;
  _Meas(this.times, this.q402m, this.q1km);
}

class _PerfScreenState extends State<PerfScreen> {
  static const _targets = [60, 100, 150, 200];
  String _state = 'disarmed';
  final Map<int, double> _times = {};
  double _dist = 0;
  double _t0 = 0, _t100 = -1;
  DateTime? _lastTs;
  double _lastSpeed = 0;
  double? _q402, _q1km;
  final List<_Meas> _results = [];
  StreamSubscription? _sub;

  void _reset() {
    setState(() {
      _state = 'armed';
      _times.clear();
      _dist = 0;
      _q402 = null;
      _q1km = null;
      _t100 = -1;
      _lastTs = null;
    });
  }

  void _onSnap(LiveSnapshot s) {
    final v = s.speed;
    final now = s.ts;
    if (_state == 'armed') {
      if (v < 2 && _lastSpeed > 5) { _dist = 0; _times.clear(); }
      if (v > 3 && _lastSpeed <= 3) {
        _state = 'run';
        _t0 = now.millisecondsSinceEpoch / 1000.0;
        _dist = 0;
      }
    } else if (_state == 'run') {
      if (_lastTs != null) {
        final dt = now.difference(_lastTs!).inMilliseconds / 1000.0;
        _dist += (_lastSpeed / 3.6) * dt;
      }
      final t = now.millisecondsSinceEpoch / 1000.0 - _t0;
      for (final tg in _targets) {
        if (!_times.containsKey(tg) && v >= tg) {
          _times[tg] = t;
          if (tg == 100 && _t100 < 0) _t100 = t;
        }
      }
      if (_q402 == null && _dist >= 402) _q402 = t;
      if (_q1km == null && _dist >= 1000) _q1km = t;
      if (v >= 200 || (_times.length == _targets.length && _dist >= 1000) ||
          (v < 2 && _dist > 30)) {
        _finish();
      }
    }
    _lastSpeed = v;
    _lastTs = now;
    if (mounted) setState(() {});
  }

  void _finish() {
    _state = 'done';
    _results.insert(0, _Meas(Map.of(_times), _q402 ?? 0, _q1km ?? 0));
  }

  @override
  void initState() {
    super.initState();
    final p = ConnectionService.I.poller;
    if (p != null) _sub = p.snapshots.listen(_onSnap);
  }

  @override
  void dispose() {
    _sub?.cancel();
    super.dispose();
  }

  String _fmt(double? v) => v == null ? '--' : v.toStringAsFixed(2);

  @override
  Widget build(BuildContext context) {
    final connected = ConnectionService.I.poller != null;
    if (!connected) {
      return const Center(
          child: Text('Подключись в настройках', style: TextStyle(color: HeatColors.dim)));
    }
    return ListView(padding: const EdgeInsets.all(12), children: [
      Container(
        padding: const EdgeInsets.all(14),
        decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
        child: Row(children: [
          Expanded(
            child: Text(
              _state == 'armed'
                  ? 'ЖДУ СТАРТ: газ в пол с места'
                  : _state == 'run'
                      ? 'ИДЁТ ЗАМЕР · ${_dist.toStringAsFixed(0)} м'
                      : _state == 'done'
                          ? 'ЗАМЕР ЗАВЕРШЁН'
                          : 'Нажми АРМИРОВАТЬ',
              style: const TextStyle(fontWeight: FontWeight.w700),
            ),
          ),
          FilledButton(
              onPressed: _reset,
              child: Text(_state == 'armed' ? 'СБРОС' : 'АРМИРОВАТЬ')),
        ]),
      ),
      if (_state == 'run' || _state == 'done')
        Container(
          margin: const EdgeInsets.only(top: 12),
          padding: const EdgeInsets.all(14),
          decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
          child: Row(
            children: [
              for (final tg in _targets)
                Expanded(
                  child: Column(children: [
                    Text('0-$tg', style: const TextStyle(fontSize: 9, color: HeatColors.dim)),
                    Text(_fmt(_times[tg]),
                        style: const TextStyle(fontSize: 17, fontWeight: FontWeight.w800)),
                  ]),
                ),
              Expanded(
                child: Column(children: [
                  const Text('402м', style: TextStyle(fontSize: 9, color: HeatColors.dim)),
                  Text(_fmt(_q402),
                      style: const TextStyle(fontSize: 17, fontWeight: FontWeight.w800, color: HeatColors.gold)),
                ]),
              ),
            ],
          ),
        ),
      const SizedBox(height: 12),
      ..._results.map((r) => Container(
            margin: const EdgeInsets.only(bottom: 8),
            padding: const EdgeInsets.all(12),
            decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(10)),
            child: Text(
              '0-60: ${_fmt(r.times[60])}c · 0-100: ${_fmt(r.times[100])}c · '
              '0-150: ${_fmt(r.times[150])}c · 0-200: ${_fmt(r.times[200])}c · '
              '402м: ${r.q402m > 0 ? r.q402m.toStringAsFixed(2) : '--'}c',
              style: const TextStyle(fontSize: 12),
            ),
          )),
    ]);
  }
}
''')
print('OK  perf_screen.dart')

# ============ lib/screens/terminal_screen.dart ============
with open('lib/screens/terminal_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

/// Терминал: сырые AT/hex-команды ELM327 (как в V6), ответ как есть.
class TerminalScreen extends StatefulWidget {
  const TerminalScreen({super.key});

  @override
  State<TerminalScreen> createState() => _TerminalScreenState();
}

class _TerminalScreenState extends State<TerminalScreen> {
  final _ctl = TextEditingController();
  final List<String> _log = ['SSM2/ELM терминал. Примеры ниже. Отправка ставит опрос на паузу.'];
  bool _busy = false;

  static const _quick = [
    'ATRV',
    '8010F001BF40',
    'A8000000080E',
    'A8000000F900',
    'A8 00 00 01 99 00',
  ];

  Future<void> _send(String cmdRaw) async {
    final cmd = cmdRaw.replaceAll(' ', '').toUpperCase();
    if (cmd.isEmpty || _busy) return;
    setState(() { _busy = true; _log.add('> $cmd'); });
    try {
      final r = await ConnectionService.I.exclusive((elm) => elm.transact(cmd, timeoutMs: 1500));
      final text = r.trim().isEmpty ? '(пусто/timeout)' : r.trim();
      _log.add(text);
    } catch (e) {
      _log.add('ERR $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    final connected = ConnectionService.I.elm.state == SsmState.ecuReady ||
        ConnectionService.I.elm.state == SsmState.polling;
    return Column(
      children: [
        SizedBox(
          height: 42,
          child: ListView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 6),
            children: _quick
                .map((q) => Padding(
                      padding: const EdgeInsets.only(right: 6),
                      child: ActionChip(
                        label: Text(q, style: const TextStyle(fontSize: 10)),
                        onPressed: connected && !_busy ? () => _send(q) : null,
                      ),
                    ))
                .toList(),
          ),
        ),
        Expanded(
          child: Container(
            margin: const EdgeInsets.all(8),
            padding: const EdgeInsets.all(10),
            decoration: BoxDecoration(
                color: const Color(0xFF04070F), borderRadius: BorderRadius.circular(10)),
            child: ListView.builder(
              reverse: true,
              itemCount: _log.length,
              itemBuilder: (ctx, i) {
                final line = _log[_log.length - 1 - i];
                final isCmd = line.startsWith('>');
                return Text(line,
                    style: TextStyle(
                        fontSize: 11.5,
                        height: 1.45,
                        color: isCmd ? HeatColors.gold : HeatColors.text));
              },
            ),
          ),
        ),
        Padding(
          padding: const EdgeInsets.fromLTRB(8, 0, 8, 8),
          child: Row(children: [
            Expanded(
              child: TextField(
                controller: _ctl,
                style: const TextStyle(fontSize: 13),
                decoration: InputDecoration(
                  hintText: 'hex без пробелов: A8000000080E',
                  isDense: true,
                  filled: true,
                  fillColor: HeatColors.panel,
                  border: OutlineInputBorder(
                      borderRadius: BorderRadius.circular(8), borderSide: BorderSide.none),
                ),
                onSubmitted: connected ? _send : null,
              ),
            ),
            const SizedBox(width: 8),
            IconButton.filled(
              onPressed: connected && !_busy ? () { _send(_ctl.text); _ctl.clear(); } : null,
              icon: const Icon(Icons.send, size: 18),
            ),
          ]),
        ),
      ],
    );
  }
}
''')
print('OK  terminal_screen.dart')
print()
print('=' * 64)
print('  Настройческие экраны готовы (8 шт). Далее -> ячейка 9/10 (PID/Профили/Экспорт/Настройки)')
print('=' * 64)


OK  dtc_screen.dart
OK  analyzer_screen.dart (сплиттер слияния CSV)
OK  ecu_maps_screen.dart (живое чтение карт из ECU)
OK  rom_screen.dart
OK  rom_diff_screen.dart
OK  service_screen.dart
OK  perf_screen.dart
OK  terminal_screen.dart

  Настройческие экраны готовы (8 шт). Далее -> ячейка 9/10 (PID/Профили/Экспорт/Настройки)


In [80]:
# @title 🗃️ Ячейка 9/10: PID CRUD · Профили · Экспорт · Настройки (+ тест PID живьём)
# ============================================================================
import os
os.chdir('/content/suba_run_v8')

# ============ lib/screens/custom_pid_screen.dart ============
with open('lib/screens/custom_pid_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/custom_pid_service.dart';
import '../services/expr_eval.dart';
import '../widgets/heat_colors.dart';

/// PID CRUD (как в V6): добавить/изменить/удалить кастомные PID,
/// живой тест значения с ECU, автовключение в блочный план опроса.
class CustomPidScreen extends StatefulWidget {
  const CustomPidScreen({super.key});

  @override
  State<CustomPidScreen> createState() => _CustomPidScreenState();
}

class _CustomPidScreenState extends State<CustomPidScreen> {
  static const _storages = ['uint8', 'int8', 'uint16', 'int16', 'float'];
  static const _lens = [1, 2, 4];

  Future<void> _edit([CustomPid? existing]) async {
    final pid = existing ??
        CustomPid(id: DateTime.now().millisecondsSinceEpoch.toString(), name: '', address: 0x000000);
    final name = TextEditingController(text: pid.name);
    final addr = TextEditingController(
        text: pid.address.toRadixString(16).toUpperCase().padLeft(6, '0'));
    final expr = TextEditingController(text: pid.expr);
    final unit = TextEditingController(text: pid.unit);
    var len = pid.len;
    var storage = pid.storage;
    var prio = pid.priority;
    String testResult = '';

    final saved = await showDialog<bool>(
      context: context,
      builder: (ctx) => StatefulBuilder(
        builder: (ctx, setS) => AlertDialog(
          backgroundColor: HeatColors.panel,
          title: Text(existing == null ? 'Новый PID' : 'Правка PID', style: const TextStyle(fontSize: 15)),
          content: SingleChildScrollView(
            child: Column(mainAxisSize: MainAxisSize.min, children: [
              TextField(controller: name, decoration: const InputDecoration(labelText: 'Имя (например "EGT")')),
              TextField(controller: addr,
                  decoration: const InputDecoration(labelText: 'Адрес hex, 6 знаков (0xFF70C6)')),
              Row(children: [
                Expanded(
                  child: DropdownButtonFormField<int>(
                    initialValue: len,
                    decoration: const InputDecoration(labelText: 'Байт'),
                    items: _lens.map((l) => DropdownMenuItem(value: l, child: Text('$l'))).toList(),
                    onChanged: (v) => setS(() => len = v ?? 1),
                  ),
                ),
                const SizedBox(width: 8),
                Expanded(
                  child: DropdownButtonFormField<String>(
                    initialValue: storage,
                    decoration: const InputDecoration(labelText: 'Тип'),
                    items: _storages.map((s) => DropdownMenuItem(value: s, child: Text(s))).toList(),
                    onChanged: (v) => setS(() => storage = v ?? 'uint8'),
                  ),
                ),
              ]),
              TextField(controller: expr,
                  decoration: const InputDecoration(
                      labelText: 'Формула от x (напр. x*0.01953125  или  (x-40))')),
              Row(children: [
                Expanded(
                  child: TextField(controller: unit,
                      decoration: const InputDecoration(labelText: 'Единицы')),
                ),
                const SizedBox(width: 8),
                Expanded(
                  child: DropdownButtonFormField<int>(
                    initialValue: prio,
                    decoration: const InputDecoration(labelText: 'Ярус'),
                    items: const [
                      DropdownMenuItem(value: 1, child: Text('fast')),
                      DropdownMenuItem(value: 2, child: Text('mid')),
                      DropdownMenuItem(value: 3, child: Text('slow')),
                    ],
                    onChanged: (v) => setS(() => prio = v ?? 2),
                  ),
                ),
              ]),
              const SizedBox(height: 10),
              Row(children: [
                Expanded(
                  child: OutlinedButton.icon(
                    icon: const Icon(Icons.bolt, size: 16),
                    label: const Text('ТЕСТ С ECU'),
                    onPressed: () async {
                      try {
                        final a = int.parse(addr.text, radix: 16);
                        final v = await ConnectionService.I.exclusive((elm) async {
                          final d = await elm.readBytes(a, len);
                          if (d == null) return 'нет ответа';
                          final ev = ExprEval(expr.text);
                          final raw = ExprEval.rawOf(d, storage);
                          return 'raw=$raw → ${ev(raw).toStringAsFixed(3)}';
                        });
                        setS(() => testResult = v);
                      } catch (e) {
                        setS(() => testResult = 'ERR $e');
                      }
                    },
                  ),
                ),
              ]),
              if (testResult.isNotEmpty)
                Padding(
                  padding: const EdgeInsets.only(top: 6),
                  child: Text(testResult,
                      style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
                ),
            ]),
          ),
          actions: [
            TextButton(onPressed: () => Navigator.pop(ctx, false), child: const Text('ОТМЕНА')),
            FilledButton(onPressed: () => Navigator.pop(ctx, true), child: const Text('СОХРАНИТЬ')),
          ],
        ),
      ),
    );
    if (saved != true) return;
    try {
      ExprEval(expr.text); // валидация формулы
    } catch (_) {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Формула не парсится')));
      return;
    }
    final a = int.tryParse(addr.text, radix: 16);
    if (a == null || name.text.trim().isEmpty) {
      ScaffoldMessenger.of(context).showSnackBar(const SnackBar(content: Text('Проверь адрес и имя')));
      return;
    }
    await CustomPidService.I.upsert(CustomPid(
      id: pid.id, name: name.text.trim(), unit: unit.text.trim(),
      category: 'custom', address: a, len: len, storage: storage,
      expr: expr.text.trim(), priority: prio,
    ));
    ConnectionService.I.buildPoller();
    ConnectionService.I.startPolling();
    setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final items = CustomPidService.I.items;
    return Scaffold(
      appBar: AppBar(
          title: const Text('Кастомные PID', style: TextStyle(fontSize: 15)),
          automaticallyImplyLeading: false),
      floatingActionButton: FloatingActionButton.small(
          onPressed: () => _edit(), child: const Icon(Icons.add)),
      body: items.isEmpty
          ? const Center(
              child: Text(
                  'Свои PID: адрес + формула → попадут в блочный план опроса\n'
                  'и на приборную панель через канон. Тест с ECU — прямо в редакторе.',
                  textAlign: TextAlign.center,
                  style: TextStyle(color: HeatColors.dim, fontSize: 12)))
          : ListView.builder(
              itemCount: items.length,
              itemBuilder: (ctx, i) {
                final p = items[i];
                return ListTile(
                  dense: true,
                  leading: const Icon(Icons.tune, color: HeatColors.accent),
                  title: Text('${p.name}  [${p.unit}]', style: const TextStyle(fontSize: 13)),
                  subtitle: Text(
                      '0x${p.address.toRadixString(16).toUpperCase().padLeft(6, '0')} · ${p.len}B $p.storage · x → ${p.expr} · prio ${p.priority}'
                          .replaceAll('p.storage', p.storage),
                      style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                  trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                    IconButton(icon: const Icon(Icons.edit, size: 18), onPressed: () => _edit(p)),
                    IconButton(
                        icon: const Icon(Icons.delete_outline, size: 18),
                        onPressed: () async {
                          await CustomPidService.I.remove(p.id);
                          ConnectionService.I.buildPoller();
                          setState(() {});
                        }),
                  ]),
                );
              },
            ),
    );
  }
}
''')
print('OK  custom_pid_screen.dart')

# ============ lib/screens/profile_screen.dart ============
with open('lib/screens/profile_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';

import '../services/connection_service.dart';
import '../services/profile_service.dart';
import '../widgets/heat_colors.dart';

/// Профили: ВСЕ настройки опроса одним набором (как в V6).
class ProfileScreen extends StatefulWidget {
  const ProfileScreen({super.key});

  @override
  State<ProfileScreen> createState() => _ProfileScreenState();
}

class _ProfileScreenState extends State<ProfileScreen> {
  final _nameCtl = TextEditingController();

  @override
  Widget build(BuildContext context) {
    final names = ProfileService.I.profiles.keys.toList()..sort();
    return ListView(padding: const EdgeInsets.all(12), children: [
      Container(
        padding: const EdgeInsets.all(12),
        decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
        child: Row(children: [
          Expanded(
            child: TextField(
              controller: _nameCtl,
              style: const TextStyle(fontSize: 13),
              decoration: const InputDecoration(
                  hintText: 'имя профиля (напр. "стрит 98 бензин")', isDense: true),
            ),
          ),
          const SizedBox(width: 8),
          FilledButton(
            onPressed: () async {
              final n = _nameCtl.text.trim();
              if (n.isEmpty) return;
              await ProfileService.I.saveCurrent(n);
              _nameCtl.clear();
              setState(() {});
            },
            child: const Text('СОХР. ТЕКУЩИЕ'),
          ),
        ]),
      ),
      const SizedBox(height: 10),
      ...names.map((n) => Padding(
            padding: const EdgeInsets.only(bottom: 8),
            child: ListTile(
              tileColor: HeatColors.panel,
              shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(10)),
              leading: const Icon(Icons.bookmark_outline, color: HeatColors.gold),
              title: Text(n, style: const TextStyle(fontSize: 13.5)),
              subtitle: Text(
                  '${(ProfileService.I.profiles[n]?['enabledIds'] as List?)?.length ?? '?'} PID · '
                  'блок 0x${(ProfileService.I.profiles[n]?['maxBlock'] ?? 0x50).toString()}',
                  style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
              trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                IconButton(
                  icon: const Icon(Icons.play_circle_outline, color: Colors.greenAccent),
                  onPressed: () async {
                    await ProfileService.I.apply(n);
                    ConnectionService.I.buildPoller();
                    ConnectionService.I.startPolling();
                    if (mounted) {
                      ScaffoldMessenger.of(context)
                          .showSnackBar(SnackBar(content: Text('Профиль «$n» применён')));
                    }
                    setState(() {});
                  },
                ),
                IconButton(
                  icon: const Icon(Icons.delete_outline),
                  onPressed: () async {
                    await ProfileService.I.remove(n);
                    setState(() {});
                  },
                ),
              ]),
            ),
          )),
      const Padding(
        padding: EdgeInsets.all(8),
        child: Text('Профиль включает: набор PID, размер блока, ярусы, AT ST, автолог.',
            style: TextStyle(fontSize: 11, color: HeatColors.dim)),
      ),
    ]);
  }
}
''')
print('OK  profile_screen.dart')

# ============ lib/screens/export_screen.dart ============
with open('lib/screens/export_screen.dart', 'w') as f:
    f.write(r'''import 'dart:io';

import 'package:flutter/material.dart';
import 'package:share_plus/share_plus.dart';

import '../services/connection_service.dart';
import '../services/export_service.dart';
import '../services/logger_service.dart';
import '../widgets/heat_colors.dart';

/// Экспорт (как в V6): карты → CSV (ecuEdit/WinOLS), ROM → HEX, логи.
class ExportScreen extends StatefulWidget {
  const ExportScreen({super.key});

  @override
  State<ExportScreen> createState() => _ExportScreenState();
}

class _ExportScreenState extends State<ExportScreen> {
  bool _busy = false;
  String _msg = '';
  String? _table;

  Future<void> _run(String label, Future<File> Function() job) async {
    if (_busy) return;
    setState(() { _busy = true; _msg = '$label ...'; });
    try {
      final f = await job();
      setState(() => _msg = 'готово: ${f.path.split('/').last}');
      await Share.shareXFiles([XFile(f.path)]);
    } catch (e) {
      setState(() => _msg = 'ошибка: $e');
    }
    setState(() => _busy = false);
  }

  @override
  Widget build(BuildContext context) {
    final rom = ConnectionService.I.rom;
    final tables = rom.loaded ? rom.defs.map((d) => d.name).toList() : <String>[];
    return ListView(padding: const EdgeInsets.all(12), children: [
      _tile(
        'ВСЕ карты → CSV (ecuEdit-стиль, Excel)',
        Icons.table_view,
        rom.loaded && !_busy,
        () => _run('Экспорт всех карт', () => ExportService.exportAllTablesCsv(rom)),
      ),
      Container(
        margin: const EdgeInsets.only(bottom: 12),
        padding: const EdgeInsets.all(10),
        decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          const Text('ОДНА карта → WinOLS CSV', style: TextStyle(fontSize: 11, color: HeatColors.gold)),
          const SizedBox(height: 8),
          DropdownButtonHideUnderline(
            child: DropdownButton<String>(
              isExpanded: true,
              dropdownColor: HeatColors.panel,
              hint: const Text('выбери карту', style: TextStyle(fontSize: 12)),
              value: _table,
              items: tables.map((t) => DropdownMenuItem(value: t, child: Text(t, style: const TextStyle(fontSize: 12)))).toList(),
              onChanged: (v) => setState(() => _table = v),
            ),
          ),
          const SizedBox(height: 8),
          FilledButton.tonal(
            onPressed: rom.loaded && _table != null && !_busy
                ? () => _run('Экспорт карты', () => ExportService.exportTableWinols(rom, _table!))
                : null,
            child: const Text('ЭКСПОРТ КАРТЫ'),
          ),
        ]),
      ),
      _tile(
        'ROM → HEX-дамп (64 KB, текст)',
        Icons.code,
        rom.loaded && !_busy,
        () => _run('HEX-дамп', () => ExportService.hexDump(rom.rom!)),
      ),
      _tile(
        'Последний лог → поделиться',
        Icons.share,
        !_busy,
        () => _run('Лог', () async {
          final logs = await LoggerService.listLogs();
          if (logs.isEmpty) throw StateError('нет логов');
          return File(logs.first.path);
        }),
      ),
      if (_msg.isNotEmpty)
        Padding(
          padding: const EdgeInsets.all(8),
          child: Text(_msg, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
        ),
      if (!rom.loaded)
        const Padding(
          padding: EdgeInsets.all(8),
          child: Text('Для экспорта карт сначала загрузи ROM на экране ROM.',
              style: TextStyle(fontSize: 11, color: HeatColors.dim)),
        ),
    ]);
  }

  Widget _tile(String label, IconData icon, bool enabled, VoidCallback onTap) {
    return Padding(
      padding: const EdgeInsets.only(bottom: 12),
      child: ListTile(
        tileColor: HeatColors.panel,
        shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(12)),
        leading: Icon(icon, color: HeatColors.accent),
        title: Text(label, style: const TextStyle(fontSize: 13)),
        trailing: const Icon(Icons.chevron_right),
        enabled: enabled,
        onTap: onTap,
      ),
    );
  }
}
''')
print('OK  export_screen.dart')

# ============ lib/screens/settings_screen.dart ============
with open('lib/screens/settings_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';

import '../constants.dart';
import '../generated/subaru_pids.g.dart';
import '../services/connection_service.dart';
import '../services/settings_service.dart';
import '../ssm/ssm_elm.dart';
import '../widgets/heat_colors.dart';

class SettingsScreen extends StatefulWidget {
  const SettingsScreen({super.key});

  @override
  State<SettingsScreen> createState() => _SettingsScreenState();
}

class _SettingsScreenState extends State<SettingsScreen> {
  List<BluetoothDevice> _devices = [];
  bool _busy = false;
  String _msg = '';

  SettingsService get st => SettingsService.I;
  ConnectionService get svc => ConnectionService.I;

  @override
  void initState() {
    super.initState();
    _refreshDevices();
  }

  Future<void> _refreshDevices() async {
    try {
      await FlutterBluetoothSerial.instance.requestEnable();
      final list = await FlutterBluetoothSerial.instance.getBondedDevices();
      if (mounted) setState(() => _devices = list);
    } catch (_) {}
  }

  @override
  Widget build(BuildContext context) {
    final stats = svc.elm.stats;
    return ListView(
      padding: const EdgeInsets.all(12),
      children: [
        _card('ПОДКЛЮЧЕНИЕ ELM327', [
          DropdownButtonHideUnderline(
            child: DropdownButton<String>(
              isExpanded: true,
              dropdownColor: HeatColors.panel,
              hint: const Text('выбери сопряжённый ELM327'),
              value: st.btAddress.isEmpty ? null : st.btAddress,
              items: _devices
                  .map((d) => DropdownMenuItem(
                      value: d.address,
                      child: Text('${d.name ?? '?'} · ${d.address}', style: const TextStyle(fontSize: 13))))
                  .toList(),
              onChanged: (v) {
                if (v == null) return;
                final d = _devices.firstWhere((e) => e.address == v);
                st.btAddress = v;
                st.btName = d.name ?? '';
                st.save();
                setState(() {});
              },
            ),
          ),
          const SizedBox(height: 8),
          Row(children: [
            Expanded(
              child: FilledButton.icon(
                onPressed: _busy ? null : _connect,
                icon: const Icon(Icons.electric_bolt),
                label: Text(svc.elm.state == SsmState.polling ? 'ПЕРЕПОДКЛЮЧИТЬ' : 'CONNECT + INIT ECU'),
              ),
            ),
            const SizedBox(width: 8),
            IconButton(onPressed: _refreshDevices, icon: const Icon(Icons.refresh)),
            IconButton(
              onPressed: () async { await svc.disconnect(); setState(() {}); },
              icon: const Icon(Icons.link_off, color: Colors.redAccent),
            ),
          ]),
          if (_msg.isNotEmpty)
            Padding(
              padding: const EdgeInsets.only(top: 8),
              child: Text(_msg, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
            ),
          if (svc.elm.ecuId.isNotEmpty)
            _kv('ECU ID', '${svc.elm.ecuId} (ожидался ${AppConstants.ecuIdExpected})'),
          if (svc.elm.elmVersion.isNotEmpty) _kv('Адаптер', svc.elm.elmVersion),
        ]),
        _card('СТАТИСТИКА ПРОТОКОЛА (монитор скорости)', [
          Row(children: [
            _stat('${stats.hz.toStringAsFixed(1)}', 'снапш./с'),
            _stat('${stats.avgMs.toStringAsFixed(0)} мс', 'на CAN-кадр'),
            _stat('${svc.poller?.blockCount ?? 0}', 'блока'),
            _stat('${svc.poller?.pidCount ?? 0}', 'PID'),
          ]),
          const SizedBox(height: 6),
          _kv('Кадров OK / ошибок / NO DATA', '${stats.framesOk} / ${stats.framesErr} / ${stats.noData}'),
          Row(children: [
            Expanded(
              child: OutlinedButton(
                onPressed: () { svc.startPolling(); setState(() {}); },
                child: const Text('СТАРТ ОПРОС'),
              ),
            ),
            const SizedBox(width: 8),
            Expanded(
              child: OutlinedButton(
                onPressed: () { svc.stopPolling(); setState(() {}); },
                child: const Text('СТОП'),
              ),
            ),
          ]),
        ]),
        _card('ТЮНИНГ ОПРОСА (блоки SSM2)', [
          _slider('Макс. размер блока', st.maxBlock.toDouble(), 0x20, 0x80, (v) {
            st.maxBlock = v.round() & ~0xF;
          }, '0x${st.maxBlock.toRadixString(16).toUpperCase()}'),
          _slider('Средний ярус: каждый N цикл', st.midEveryN.toDouble(), 2, 10, (v) {
            st.midEveryN = v.round();
          }, '${st.midEveryN}'),
          _slider('Медленный ярус: каждый N цикл', st.slowEveryN.toDouble(), 10, 60, (v) {
            st.slowEveryN = v.round();
          }, '${st.slowEveryN}'),
          _slider('AT ST (×4 мс ожидание байта)', st.stCode.toDouble(), 2, 30, (v) {
            st.stCode = v.round();
            svc.elm.stTimeoutCode = st.stCode;
          }, '${st.stCode * 4} мс'),
          FilledButton.tonalIcon(
            onPressed: () {
              st.save();
              svc.buildPoller();
              svc.startPolling();
              setState(() {});
            },
            icon: const Icon(Icons.autorenew, size: 18),
            label: const Text('ПЕРЕСТРОИТЬ БЛОКИ И ОПРОС'),
          ),
        ]),
        _card('PID В ОПРОСЕ (${st.enabledIds.length})', [
          Row(children: [
            TextButton(
              onPressed: () {
                st.enabledIds = SubaruPids.defaults.map((e) => e.id).toSet();
                st.save();
                svc.buildPoller();
                setState(() {});
              },
              child: const Text('только канонические'),
            ),
            TextButton(
              onPressed: () {
                st.enabledIds = SubaruPids.all.map((e) => e.id).toSet();
                st.save();
                setState(() {});
              },
              child: const Text('ВСЕ 150+'),
            ),
            TextButton(
              onPressed: () {
                st.enabledIds = {};
                st.save();
                setState(() {});
              },
              child: const Text('ничего'),
            ),
          ]),
          ...SubaruPids.byCategory().entries.map((entry) => ExpansionTile(
                dense: true,
                title: Text('${entry.key} (${entry.value.length})', style: const TextStyle(fontSize: 13)),
                children: entry.value
                    .map((p) => CheckboxListTile(
                          dense: true,
                          value: st.enabledIds.contains(p.id),
                          onChanged: (v) {
                            if (v == true) {
                              st.enabledIds.add(p.id);
                            } else {
                              st.enabledIds.remove(p.id);
                            }
                            st.save();
                            setState(() {});
                          },
                          title: Text(p.name, style: const TextStyle(fontSize: 12)),
                          subtitle: Text(
                              '${p.addrHex} · ${p.unit} · prio ${p.priority}${p.canon.isNotEmpty ? ' · canon ${p.canon}' : ''}',
                              style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
                        ))
                    .toList(),
              )),
        ]),
        _card('О ПРОШИВКЕ', [
          _kv('CAL ID', AppConstants.calId),
          _kv('ECU ID', AppConstants.ecuIdExpected),
          _kv('Двигатель', AppConstants.engine),
          _kv('Протокол', 'SSM2 over CAN · ISO 15765-4 · 0x7E0/0x7E8 · 500 kbit · 11 bit'),
          _kv('Генератор', 'RomRaider logger.xml + ECUFlash A2TB100B(+K) → встроено'),
        ]),
      ],
    );
  }

  Widget _card(String title, List<Widget> children) {
    return Container(
      margin: const EdgeInsets.only(bottom: 12),
      padding: const EdgeInsets.all(12),
      decoration: BoxDecoration(color: HeatColors.panel, borderRadius: BorderRadius.circular(12)),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Text(title,
            style: const TextStyle(
                fontSize: 12, fontWeight: FontWeight.w800, letterSpacing: 1, color: HeatColors.gold)),
        const SizedBox(height: 8),
        ...children,
      ]),
    );
  }

  Widget _kv(String k, String v) => Padding(
        padding: const EdgeInsets.only(top: 4),
        child: RichText(
          text: TextSpan(children: [
            TextSpan(text: '$k: ', style: const TextStyle(fontSize: 12, color: HeatColors.dim)),
            TextSpan(text: v, style: const TextStyle(fontSize: 12, color: HeatColors.text)),
          ]),
        ),
      );

  Widget _stat(String v, String l) => Expanded(
        child: Column(children: [
          Text(v, style: const TextStyle(fontSize: 18, fontWeight: FontWeight.w800, color: HeatColors.text)),
          Text(l, style: const TextStyle(fontSize: 10, color: HeatColors.dim)),
        ]),
      );

  Widget _slider(String label, double v, double min, double max, void Function(double) on, String shown) {
    return Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
      Row(children: [
        Expanded(child: Text(label, style: const TextStyle(fontSize: 12))),
        Text(shown, style: const TextStyle(fontSize: 12, color: HeatColors.gold)),
      ]),
      Slider(
        value: v.clamp(min, max), min: min, max: max,
        divisions: (max - min).round(),
        onChanged: (nv) { on(nv); setState(() {}); },
      ),
    ]);
  }

  Future<void> _connect() async {
    setState(() { _busy = true; _msg = 'подключаюсь...'; });
    final r = await svc.connectAndInit();
    if (!mounted) return;
    setState(() { _busy = false; _msg = r; });
  }
}
''')
print('OK  settings_screen.dart')
print()
print('=' * 64)
print('  Все 17 экранов готовы. Далее -> ячейка 10/10 (сборка APK)')
print('=' * 64)


OK  custom_pid_screen.dart
OK  profile_screen.dart
OK  export_screen.dart
OK  settings_screen.dart

  Все 17 экранов готовы. Далее -> ячейка 10/10 (сборка APK)


In [81]:
# @title 🔨 Ячейка 10/10: Сборка SUBA RUN V8 APK (~10-15 мин)
# ============================================================================
import os, glob

os.chdir('/content/suba_run_v8')

# окружение (на случай перезапуска сессии)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH', '')
os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

print('=' * 64)
print('  ЭТАП 1: flutter clean + pub get')
print('=' * 64)
!flutter clean 2>&1 | tail -2
!rm -rf build
!rm -f pubspec.lock
pub_result = !flutter pub get 2>&1
for line in pub_result[-6:]:
    print(line)
if any('version solving failed' in l for l in pub_result):
    raise SystemExit('pub get провалился — проверь констрейнт SDK в pubspec (должно быть <4.0.0)')

print()
print('=' * 64)
print('  ЭТАП 2: Патч flutter_bluetooth_serial (namespace/SDK36 — фикс из V6)')
print('=' * 64)
plugin_dirs = glob.glob('/content/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*') or \
              glob.glob('/root/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')
for plugin_dir in plugin_dirs:
    with open(f'{plugin_dir}/android/build.gradle', 'w') as f:
        f.write('''group 'io.github.edufolly.flutterbluetoothserial'
version '1.0-SNAPSHOT'
buildscript {
    repositories { google(); mavenCentral() }
    dependencies { classpath 'com.android.tools.build:gradle:8.13.0' }
}
allprojects { repositories { google(); mavenCentral() } }
apply plugin: 'com.android.library'
android {
    namespace 'io.github.edufolly.flutterbluetoothserial'
    compileSdk 36
    compileOptions {
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }
    defaultConfig { minSdk 21 }
    lintOptions {
        disable 'InvalidPackage'
        checkReleaseBuilds false
        abortOnError false
    }
}
dependencies { implementation 'androidx.core:core:1.13.1' }
''')
    with open(f'{plugin_dir}/android/src/main/AndroidManifest.xml', 'w') as f:
        f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
</manifest>
''')
    print('  OK ' + plugin_dir.split('/')[-1])
if not plugin_dirs:
    print('  плагин не найден в кэше (проверь pub get)')

print()
print('=' * 64)
print('  ЭТАП 2.5: Версии Gradle/AGP/Kotlin под ТЕКУЩИЙ Flutter stable')
print('=' * 64)
# Flutter stable постоянно двигает минимумы (сейчас Gradle >= 8.14).
# Принудительно выставляем совместимую связку перед сборкой.
import re as _re
_wp = 'android/gradle/wrapper/gradle-wrapper.properties'
with open(_wp, 'w') as f:
    f.write('''distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\\://services.gradle.org/distributions/gradle-8.14-all.zip
''')
_sg = 'android/settings.gradle.kts'
_s = open(_sg).read()
_s = _re.sub(r'"com\.android\.application"\) version "[^"]+"',
             '"com.android.application") version "8.13.0"', _s)
_s = _re.sub(r'"org\.jetbrains\.kotlin\.android"\) version "[^"]+"',
             '"org.jetbrains.kotlin.android") version "2.1.20"', _s)
open(_sg, 'w').write(_s)
print('  Gradle 8.14 · AGP 8.13.0 · Kotlin 2.1.20')

print()
print('=' * 64)
print('  ЭТАП 3: Проверка структуры')
print('=' * 64)
required = [
    'lib/main.dart', 'lib/constants.dart',
    'lib/models/live_snapshot.dart', 'lib/models/rom_table.dart',
    'lib/ssm/ssm_elm.dart',
    'lib/generated/subaru_pids.g.dart', 'lib/generated/subaru_rom.g.dart',
    'lib/services/settings_service.dart', 'lib/services/logger_service.dart',
    'lib/services/alert_service.dart', 'lib/services/rom_service.dart',
    'lib/services/analyzer_service.dart', 'lib/services/expr_eval.dart',
    'lib/services/dtc_service.dart', 'lib/services/custom_pid_service.dart',
    'lib/services/profile_service.dart', 'lib/services/export_service.dart',
    'lib/services/connection_service.dart',
    'lib/widgets/heat_colors.dart', 'lib/widgets/heat_map.dart',
    'lib/screens/dashboard_screen.dart', 'lib/screens/graphs_screen.dart',
    'lib/screens/log_graph_screen.dart', 'lib/screens/logging_screen.dart',
    'lib/screens/events_screen.dart', 'lib/screens/dtc_screen.dart',
    'lib/screens/analyzer_screen.dart', 'lib/screens/ecu_maps_screen.dart',
    'lib/screens/rom_screen.dart', 'lib/screens/rom_diff_screen.dart',
    'lib/screens/service_screen.dart', 'lib/screens/perf_screen.dart',
    'lib/screens/export_screen.dart', 'lib/screens/custom_pid_screen.dart',
    'lib/screens/profile_screen.dart', 'lib/screens/terminal_screen.dart',
    'lib/screens/settings_screen.dart',
]
all_ok = True
for rf in required:
    ok = os.path.exists(rf) and os.path.getsize(rf) > 50
    all_ok &= ok
    if not ok:
        print('  НЕТ ' + rf)
import subprocess
total_lines = int(subprocess.check_output("find lib -name '*.dart' | xargs wc -l | tail -1 | awk '{print $1}'", shell=True))
print(f'  файлов: {sum(1 for rf in required if os.path.exists(rf))}/{len(required)} · строк Dart: {total_lines}')
if not all_ok:
    raise SystemExit('Не хватает файлов — прогони ячейки 0-9 по порядку!')

print()
print('=' * 64)
print('  ЭТАП 4: СБОРКА APK')
print('=' * 64)
result = !flutter build apk --release --no-tree-shake-icons --android-skip-build-dependency-validation 2>&1
tail = result[-70:]
keep = [l for l in result if any(k in l.lower() for k in
        ['error', 'failed', 'built', 'app-release.apk', 'exception'])]
for l in keep[-25:]:
    print(l)

apk = '/content/suba_run_v8/build/app/outputs/flutter-apk/app-release.apk'
print()
if os.path.exists(apk):
    size_mb = os.path.getsize(apk) / (1024 * 1024)
    print('=' * 64)
    print(f'  APK СОБРАН: {size_mb:.1f} MB · {total_lines} строк Dart')
    print('=' * 64)
    !cp {apk} /content/SubaRunV8.apk
    from google.colab import files
    files.download('/content/SubaRunV8.apk')
    print()
    print('УСТАНОВКА:')
    print(' 1. Установи SubaRunV8.apk, разреши Bluetooth')
    print(' 2. Настройки -> выбери ELM327 -> CONNECT + INIT ECU (жди ECU ID 5204584007)')
    print(' 3. Смотри статистику протокола: снапш/с и мс/кадр')
    print(' 4. Анализатор: Ось Y = Буст, Метрика = FBKC / KCA — цветная карта как в V6')
    print(' 5. ROM: загрузи свой .bin A2TB100B — все карты из дефинишна (через include A2TB100K)')
    print(' 6. ЭБУ Карты: чтение карты живьём из ECU блоками')
    print(' 7. 17 экранов как в V6 — но с параметрами Subaru V7')
else:
    print('APK НЕ СОБРАН. Хвост лога:')
    for l in tail:
        print(l)
    print('Скинь этот вывод — найдём причину.')


  ЭТАП 1: flutter clean + pub get
Deleting ephemeral...                                                0ms
Deleting .flutter-plugins-dependencies...                            0ms
+ xdg_directories 1.1.0
+ xml 6.6.1 (7.0.1 available)
+ yaml 3.1.4
Changed 87 dependencies!
17 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.

  ЭТАП 2: Патч flutter_bluetooth_serial (namespace/SDK36 — фикс из V6)
  OK flutter_bluetooth_serial-0.4.0

  ЭТАП 2.5: Версии Gradle/AGP/Kotlin под ТЕКУЩИЙ Flutter stable
  Gradle 8.14 · AGP 8.13.0 · Kotlin 2.1.20

  ЭТАП 3: Проверка структуры
  файлов: 37/37 · строк Dart: 7899

  ЭТАП 4: СБОРКА APK
✓ Built build/app/outputs/flutter-apk/app-release.apk (59.0MB)

  APK СОБРАН: 56.2 MB · 7899 строк Dart


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


УСТАНОВКА:
 1. Установи SubaRunV8.apk, разреши Bluetooth
 2. Настройки -> выбери ELM327 -> CONNECT + INIT ECU (жди ECU ID 5204584007)
 3. Смотри статистику протокола: снапш/с и мс/кадр
 4. Анализатор: Ось Y = Буст, Метрика = FBKC / KCA — цветная карта как в V6
 5. ROM: загрузи свой .bin A2TB100B — все карты из дефинишна (через include A2TB100K)
 6. ЭБУ Карты: чтение карты живьём из ECU блоками
 7. 17 экранов как в V6 — но с параметрами Subaru V7
